> **Research provenance.** This notebook records the original empirical workflow. Licensed inputs, saved forecasts and private research infrastructure are not distributed. Outputs and attachments are removed; see [notebook configuration](README.md). The maintained public HMM includes post-study correctness hardening and has not been rerun across the complete 2001-2024 historical sample. Saved paper-era HMM results are not replication targets for the maintained implementation.


# Fusion Ablation, Statistical Validation, and Implementation Analysis

This notebook records the original study's gross-signal, ablation, mechanism, spanning, stability, portfolio-overlap, and fusion diagnostics and uses the original stateful transaction-cost engine for implementation, capacity, and dynamic-NAV analysis.

### Research questions

1. Do the CNN and RF contain complementary cross-sectional information?
2. Does combining them improve gross portfolio performance?
3. Does the HMM add value beyond simple static fusion?
4. Does the HMM produce incremental return variation after controlling for simpler alternatives?
5. Which signals survive realistic transaction costs and liquidity constraints?
6. How does implementation performance change with AUM?
7. What is the economic and statistical capacity of each strategy?
8. Are the bounded missing-return accounting events economically immaterial?
9. What does a path-dependent dynamic-NAV implementation look like?

### Evaluation conventions

- Saved CNN, RF, and HMM predictions are reused; the predictive models are **not retrained**.
- CNN predicts a 5-day up probability.
- RF predicts a 20-day up probability.
- HMM combines the two experts and produces a 5-day probability.
- The 50/50 benchmark therefore blends **cross-sectional ranks**, not raw probabilities.
- All reported evaluation begins in **2001**. Partial 2000 observations are pre-sample history only.
- Gross signal tests and corrected implementation tests are kept conceptually separate.
- The corrected cost-aware path uses frozen OOS **daily ADV** and **daily volatility** forecasts, weekly portfolio decisions, and a **one-trading-day execution horizon**.
- Legacy holdings with a temporarily missing realized return are marked flat for that period. After four consecutive missing periods, a fifth missing observation closes only the carried residual state into cash at the last mark; no artificial trade or transaction cost is created.
- Missing-return exposure above **0.5% of a sleeve** is logged as a materiality warning and audited explicitly below.
- Old shadow-cost and pre-refactor transaction-cost results are not used as final evidence.


## 0. Environment preflight

This section verifies the active kernel and imports the final portfolio/statistical modules. It does not install or modify packages.


In [ ]:
from pathlib import Path
import sys
import json
import math
import re
import warnings
import importlib
import gc
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from IPython.display import display


def _save_show(fig, filename):
    """Save a research figure to the research figure directory and display it."""
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / filename,
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

required = [
    "numpy",
    "pandas",
    "matplotlib",
    "scipy",
    "sklearn",
    "statsmodels",
]

print("Python:", sys.executable)
print(sys.version)

if sys.version_info < (3, 10):
    raise RuntimeError("The public package requires Python 3.10 or newer.")


for package in required:
    module = importlib.import_module(package)
    print(f"{package:12s}: {getattr(module, '__version__', 'OK')}")


## 1. Configuration and run controls

The expensive implementation runs are checkpointed. `resume=True` is safe because the final runners fingerprint the signal, economic configuration, portfolio kwargs, forecast artifact, and relevant portfolio implementation.

For the first research implementation run, leave the expensive switches on. After the outputs are complete, they can be set to `False` and the notebook will load the saved accounting streams.

All new research outputs are written to a separate directory so they cannot be confused with pre-policy capacity results.


In [ ]:
import os

def find_project_root():
    configured = os.environ.get("FUSION_RESEARCH_ROOT")
    if not configured:
        raise RuntimeError(
            "Set FUSION_RESEARCH_ROOT to the original licensed research environment. "
            "The public package does not include Scripts.Data or Scripts.Portfolio."
        )
    root = Path(configured).expanduser().resolve()
    if not (root / "Scripts").is_dir():
        raise FileNotFoundError("FUSION_RESEARCH_ROOT must contain the private Scripts directory.")
    return root

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ANNUAL_ROOT = PROJECT_ROOT / "WORK_SPACE" / "expanding_window_hmm_from_saved_cnn"
ROOT_MANIFEST = ANNUAL_ROOT / "annual_expanding_window_run_manifest.json"

if not os.environ.get("FUSION_ANALYSIS_DIR"):
    raise RuntimeError("Set FUSION_ANALYSIS_DIR to the analysis output directory.")
OUTPUT_DIR = Path(os.environ["FUSION_ANALYSIS_DIR"]).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TC_FORECAST_PATH = (
    PROJECT_ROOT
    / "WORK_SPACE"
    / "portfolio"
    / "transaction_costs"
    / "tcost_inputs_oos.parquet"
)

assert ROOT_MANIFEST.exists(), f"Run manifest not found: {ROOT_MANIFEST}"

with ROOT_MANIFEST.open("r", encoding="utf-8") as f:
    MANIFEST = json.load(f)

SOURCE_FIRST_YEAR = int(MANIFEST.get("first_pred_year", 2000))
SOURCE_LAST_YEAR = int(MANIFEST.get("last_pred_year", 2024))

ANALYSIS_START_YEAR = 2001
ANALYSIS_END_YEAR = SOURCE_LAST_YEAR

FREQ = str(MANIFEST.get("freq", "week"))
COUNTRY = str(MANIFEST.get("country", "USA"))
CUT = int(MANIFEST.get("cut", 10))
DELAY = int((MANIFEST.get("delay_list") or [0])[0])

# Preserve the same gross tradability screen used in the prior analysis,
# but explicitly disable transaction-cost logic for gross-signal evaluation.
SCREENED_KWARGS = dict(
    MANIFEST.get("screened_portfolio_kwargs")
    or {
        "min_price": 1.0,
        "min_adv_dollar": 10_000_000,
        "tradability_screens": True,
        "include_price_adv": True,
    }
)

for key in (
    "signal_df", "freq", "portfolio_dir", "start_year", "end_year",
    "eval_start_year", "country", "delay_list", "load_signal",
    "tc_forecast_path", "tc_forecast_df",
):
    SCREENED_KWARGS.pop(key, None)

GROSS_SCREENED_KWARGS = dict(SCREENED_KWARGS)
GROSS_SCREENED_KWARGS.update(
    {
        "tc_enable": False,
        "tc_use_nonlinear": False,
        "tc_use_forecast_inputs": False,
        "tradability_screens": True,
        "include_price_adv": True,
    }
)

# Walk-forward fusion settings.
LOGIT_LABEL_LAG_PERIODS = 1
LOGIT_C = 1.0
RETURN_FUSION_RIDGE_ALPHA = 1.0
RETURN_FUSION_LABEL_LAG_PERIODS = 1

# Inference settings.
RANDOM_SEED = 1729
HAC_LAGS = 8
BOOTSTRAP_REPS = 5000
BOOTSTRAP_BLOCK_WEEKS = 8
BLOCK_LENGTH_SENSITIVITY = (4, 8, 13, 26)

# Shared statistical helpers used by the research-diagnostic sections.
import statsmodels.api as sm

from Scripts.Portfolio import statistical_validation as sv

PERIODS_PER_YEAR = 52

annualized_sharpe = sv.annualized_sharpe
paired_hac_test = sv.paired_hac_test

# Corrected implementation settings.
PRIMARY_AUM = 100_000_000.0
AUM_GRID = (
    10_000_000.0,
    25_000_000.0,
    50_000_000.0,
    100_000_000.0,
    250_000_000.0,
    500_000_000.0,
    1_000_000_000.0,
)
INFERENCE_AUMS = (
    100_000_000.0,
    250_000_000.0,
    500_000_000.0,
    1_000_000_000.0,
)
EXECUTION_DAYS = 1.0

# Bounded missing-return accounting for legacy holdings.
MISSING_RETURN_MAX_CARRY_PERIODS = 4
MISSING_RETURN_WARN_WEIGHT = 0.005

GROSS_MARKOWITZ_WEIGHT_TYPE = "mw_h_plus_lowneg"
COST_AWARE_MARKOWITZ_WEIGHT_TYPE = "mw_h_plus_lowneg_tc_nl"

# Run switches.
RUN_GROSS_EW = True
RUN_GROSS_MARKOWITZ = True
RUN_FIXED_AUM_CAPACITY = True
RUN_DYNAMIC_NAV = True

print("PROJECT_ROOT :", PROJECT_ROOT)
print("ANNUAL_ROOT  :", ANNUAL_ROOT)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("Evaluation   :", ANALYSIS_START_YEAR, "-", ANALYSIS_END_YEAR)
print("Frequency    :", FREQ)
print("Cut          :", CUT)
print("TC forecast  :", TC_FORECAST_PATH)
print("Missing-return carry periods:", MISSING_RETURN_MAX_CARRY_PERIODS)
print("Missing-return warning level:", f"{MISSING_RETURN_WARN_WEIGHT:.2%} of sleeve")


## 1A. Source/API compatibility preflight

Fail early if the notebook is paired with an older portfolio/statistics module rather than discovering an interface mismatch after a long run.


In [ ]:
from Scripts.Portfolio.portfolio import PortfolioManager
from Scripts.Portfolio.gross_regression import compare_gross_return_frames
from Scripts.Portfolio.capacity_runner import (
    CapacityRunResult,
    FixedAUMCapacityConfig,
    run_one_fixed_aum,
)
from Scripts.Portfolio.dynamic_nav_runner import (
    DynamicNAVConfig,
    run_one_dynamic_nav,
)

def require_parameters(callable_obj, required_names, label):
    params = inspect.signature(callable_obj).parameters
    missing = [name for name in required_names if name not in params]
    if missing:
        raise RuntimeError(
            f"{label} is missing required parameters {missing}. "
            "The notebook and project source files are out of sync."
        )

require_parameters(
    PortfolioManager.__init__,
    [
        "tc_execution_days",
        "tc_use_nonlinear",
        "tc_use_forecast_inputs",
        "tc_forecast_path",
        "tc_missing_return_max_carry_weeks",
        "tc_missing_return_warn_weight",
    ],
    "PortfolioManager.__init__",
)

require_parameters(
    sv.pairwise_inference_table,
    ["require_identical_index"],
    "statistical_validation.pairwise_inference_table",
)

require_parameters(
    sv.pairwise_inference_by_aum,
    ["require_identical_index"],
    "statistical_validation.pairwise_inference_by_aum",
)

require_parameters(
    sv.bootstrap_block_length_sensitivity,
    ["require_identical_index"],
    "statistical_validation.bootstrap_block_length_sensitivity",
)

require_parameters(
    compare_gross_return_frames,
    ["require_same_dates"],
    "gross_regression.compare_gross_return_frames",
)

require_parameters(
    FixedAUMCapacityConfig,
    ["execution_days", "require_forecast_inputs", "resume"],
    "FixedAUMCapacityConfig",
)

require_parameters(
    DynamicNAVConfig,
    ["execution_days", "require_forecast_inputs", "resume"],
    "DynamicNAVConfig",
)


_prepare_signal_source = inspect.getsource(
    PortfolioManager._prepare_signal
)

for token in (
    "_tc_selection_missing_forecast_rows",
    "require_complete=False",
):
    if token not in _prepare_signal_source:
        raise RuntimeError(
            "PortfolioManager._prepare_signal is missing the final frozen-TC "
            "forecast eligibility patch. Replace Scripts/Portfolio/portfolio.py "
            "with the tested version before running research."
        )


from Scripts.Portfolio.strategies import TCOptimizerMixin

_store_state_source = inspect.getsource(
    TCOptimizerMixin._tc_store_leg_state
)

for token in (
    "tc_missing_return_events",
    "terminal_mark_to_cash",
    "tc_missing_return_max_carry_weeks",
    "tc_missing_return_warn_weight",
):
    if token not in _store_state_source:
        raise RuntimeError(
            "TCOptimizerMixin._tc_store_leg_state is missing the tested "
            "bounded missing-return policy. Replace Scripts/Portfolio/strategies.py "
            "with the tested version before running research."
        )

if "missing_return_df" not in getattr(CapacityRunResult, "__annotations__", {}):
    raise RuntimeError(
        "CapacityRunResult does not expose missing_return_df. "
        "Replace Scripts/Portfolio/capacity_runner.py with the tested version."
    )

print("Source/API compatibility preflight: PASS")


## 2. Load saved CNN, RF, and HMM signals

The loader prefers stitched Parquet archives and otherwise reconstructs each signal from the saved annual prediction files.


In [ ]:
# 2. Load saved signal families

def standardize_signal(df: pd.DataFrame, prob_name: str) -> pd.DataFrame:
    out = df.copy()

    if isinstance(out.index, pd.MultiIndex):
        out = out.reset_index()

    if "Date" not in out.columns and "ending_date" in out.columns:
        out = out.rename(columns={"ending_date": "Date"})

    if "StockID" not in out.columns:
        for alt in ("permno", "PERMNO", "Permno", "sid", "SID", "code"):
            if alt in out.columns:
                out = out.rename(columns={alt: "StockID"})
                break

    if "up_prob" not in out.columns:
        raise KeyError(
            f"'up_prob' missing from signal file. Columns={list(out.columns)}"
        )

    out["Date"] = pd.to_datetime(out["Date"], errors="coerce").dt.normalize()
    out["StockID"] = pd.to_numeric(out["StockID"], errors="coerce")
    out["up_prob"] = pd.to_numeric(out["up_prob"], errors="coerce")

    out = out.dropna(subset=["Date", "StockID", "up_prob"]).copy()
    out["StockID"] = out["StockID"].astype(int).astype(str)

    return (
        out[["Date", "StockID", "up_prob"]]
        .drop_duplicates(["Date", "StockID"], keep="last")
        .rename(columns={"up_prob": prob_name})
        .sort_values(["Date", "StockID"])
        .reset_index(drop=True)
    )


def load_saved_signal(signal_name: str, prob_name: str) -> pd.DataFrame:
    stitched_dir = ANNUAL_ROOT / "stitched" / "signals" / signal_name

    expected = (
        stitched_dir
        / f"{signal_name}_stitched_{SOURCE_FIRST_YEAR}_{SOURCE_LAST_YEAR}.parquet"
    )

    if expected.exists():
        print(f"[load] {signal_name}: {expected}")
        return standardize_signal(pd.read_parquet(expected), prob_name)

    stitched_candidates = (
        sorted(stitched_dir.glob(f"{signal_name}_stitched_*.parquet"))
        if stitched_dir.exists()
        else []
    )

    if stitched_candidates:
        fp = stitched_candidates[-1]
        print(f"[load] {signal_name}: {fp}")
        return standardize_signal(pd.read_parquet(fp), prob_name)

    parts = []
    missing = []

    for year in range(SOURCE_FIRST_YEAR, SOURCE_LAST_YEAR + 1):
        fp = (
            ANNUAL_ROOT
            / f"year_{year}"
            / "signals"
            / signal_name
            / f"{signal_name}_up_prob_{year}.parquet"
        )

        if fp.exists():
            parts.append(pd.read_parquet(fp))
        else:
            missing.append(year)

    if missing:
        raise FileNotFoundError(
            f"Could not construct {signal_name}. "
            f"Missing annual files for years: {missing}"
        )

    print(f"[load] {signal_name}: stitching {len(parts)} annual files")

    return standardize_signal(
        pd.concat(parts, ignore_index=True),
        prob_name,
    )


rf = load_saved_signal("rf_full", "p_rf")
cnn = load_saved_signal("cnn_overlap", "p_cnn")
hmm = load_saved_signal("hmm_full", "p_hmm")

print("\nRows:")
print("RF :", f"{len(rf):,}")
print("CNN:", f"{len(cnn):,}")
print("HMM:", f"{len(hmm):,}")


## 3. Strict common-universe control

All signal comparisons use the exact intersection of `(Date, StockID)` observations. This prevents stock-availability differences from being mistaken for model improvement.


In [ ]:
# 3. Build the exact common stock-date archive, then define the 2001–2024 sample.

common_history = (
    rf.merge(cnn, on=["Date", "StockID"], how="inner")
      .merge(hmm, on=["Date", "StockID"], how="inner")
      .sort_values(["Date", "StockID"])
      .reset_index(drop=True)
)

common_history = common_history[
    (common_history["Date"].dt.year >= SOURCE_FIRST_YEAR)
    & (common_history["Date"].dt.year <= SOURCE_LAST_YEAR)
].copy()

assert not common_history.duplicated(["Date", "StockID"]).any()
assert common_history[["p_rf", "p_cnn", "p_hmm"]].notna().all().all()

common = common_history[
    (common_history["Date"].dt.year >= ANALYSIS_START_YEAR)
    & (common_history["Date"].dt.year <= ANALYSIS_END_YEAR)
].copy()

assert common["Date"].dt.year.min() == ANALYSIS_START_YEAR
assert common["Date"].dt.year.max() == ANALYSIS_END_YEAR

# Canonical strict-common key set used throughout the notebook.
common_keys = (
    common[["Date", "StockID"]]
    .drop_duplicates()
    .sort_values(["Date", "StockID"])
    .reset_index(drop=True)
)

if len(common_keys) != len(common):
    raise AssertionError(
        "The strict common evaluation panel contains duplicate Date/StockID keys."
    )

coverage = pd.DataFrame(
    {
        "signal": [
            "RF saved",
            "CNN saved",
            "HMM saved",
            "strict common archive",
            "evaluation common 2001-2024",
        ],
        "rows": [
            len(rf),
            len(cnn),
            len(hmm),
            len(common_history),
            len(common),
        ],
        "dates": [
            rf["Date"].nunique(),
            cnn["Date"].nunique(),
            hmm["Date"].nunique(),
            common_history["Date"].nunique(),
            common["Date"].nunique(),
        ],
        "stocks": [
            rf["StockID"].nunique(),
            cnn["StockID"].nunique(),
            hmm["StockID"].nunique(),
            common_history["StockID"].nunique(),
            common["StockID"].nunique(),
        ],
    }
)

display(coverage)

print(
    "Archive date range   :",
    common_history["Date"].min(),
    "to",
    common_history["Date"].max(),
)

print(
    "Evaluation date range:",
    common["Date"].min(),
    "to",
    common["Date"].max(),
)

print(
    "Strict common evaluation keys:",
    f"{len(common_keys):,}",
)

if common["Date"].min().year != 2001:
    raise AssertionError(
        "Evaluation sample does not begin in 2001."
    )


## 4. Untuned 50/50 rank blend

CNN and RF operate at different horizons. The simple untuned benchmark therefore averages their within-week percentile ranks rather than averaging raw probabilities.


In [ ]:
# 4. Construct the untuned 50/50 rank blend.

# Compute ranks on every available date in the source archive so the partial
# 2000 history can still be used if needed as pre-sample information.
common_history["rank_cnn"] = (
    common_history.groupby("Date")["p_cnn"]
    .rank(method="average", pct=True)
)

common_history["rank_rf"] = (
    common_history.groupby("Date")["p_rf"]
    .rank(method="average", pct=True)
)

common_history["rank_blend_5050"] = (
    0.5 * common_history["rank_cnn"]
    + 0.5 * common_history["rank_rf"]
)

# Refresh the actual evaluation panel after creating the rank columns.
common = common_history[
    (common_history["Date"].dt.year >= ANALYSIS_START_YEAR)
    & (common_history["Date"].dt.year <= ANALYSIS_END_YEAR)
].copy()

display(
    common[
        [
            "p_cnn",
            "p_rf",
            "rank_cnn",
            "rank_rf",
            "rank_blend_5050",
            "p_hmm",
        ]
    ].describe().T
)

print(
    "50/50 rank-blend evaluation range:",
    common["Date"].min(),
    "to",
    common["Date"].max(),
)


In [ ]:
# -------------------------------------------------------------------------
# Kernel-restart recovery
#
# Reconstruct the finalized optimizer signal dictionary from artifacts that
# were already saved during the completed research run. This avoids rerunning the
# forward-return load, walk-forward logistic estimation, expected-return
# fusion estimation, gross portfolios, or fixed-AUM capacity grid.
# -------------------------------------------------------------------------

from scipy.special import expit


# -------------------------------------------------------------------------
# 1. Recover the walk-forward logistic signal
# -------------------------------------------------------------------------

logistic_coef_path = (
    OUTPUT_DIR
    / "logistic_stack_annual_coefficients.csv"
)

logistic_contrib_path = (
    OUTPUT_DIR
    / "logistic_stack_row_contributions.parquet"
)

if not logistic_coef_path.exists():
    raise FileNotFoundError(
        f"Saved logistic coefficients not found: {logistic_coef_path}"
    )

if not logistic_contrib_path.exists():
    raise FileNotFoundError(
        f"Saved logistic row contributions not found: {logistic_contrib_path}"
    )

logistic_coefs = pd.read_csv(
    logistic_coef_path
)

logistic_coefs["target_year"] = pd.to_numeric(
    logistic_coefs["target_year"],
    errors="raise",
).astype(int)

logistic_row_contributions = pd.read_parquet(
    logistic_contrib_path
)

logistic_row_contributions["Date"] = pd.to_datetime(
    logistic_row_contributions["Date"],
    errors="raise",
).dt.normalize()

logistic_row_contributions["StockID"] = pd.to_numeric(
    logistic_row_contributions["StockID"],
    errors="raise",
).astype(int).astype(str)

logistic_row_contributions["target_year"] = (
    logistic_row_contributions["Date"]
    .dt.year
    .astype(int)
)

intercepts = (
    logistic_coefs
    .set_index("target_year")["intercept"]
)

logistic_row_contributions["intercept"] = (
    logistic_row_contributions["target_year"]
    .map(intercepts)
)

if logistic_row_contributions["intercept"].isna().any():
    bad_years = sorted(
        logistic_row_contributions.loc[
            logistic_row_contributions["intercept"].isna(),
            "target_year",
        ].unique()
    )

    raise RuntimeError(
        "Could not recover logistic intercepts for years: "
        f"{bad_years}"
    )

linear_score = (
    logistic_row_contributions["intercept"]
    + logistic_row_contributions["cnn_linear_contribution"]
    + logistic_row_contributions["rf_linear_contribution"]
)

logistic_estimated = logistic_row_contributions[
    ["Date", "StockID"]
].copy()

logistic_estimated["up_prob"] = expit(
    linear_score.to_numpy(dtype=float)
)


# The first walk-forward year can use the explicit 50/50 rank warm start.
warm_years = (
    logistic_coefs.loc[
        logistic_coefs["mode"].astype(str)
        == "warm_start_rank_5050",
        "target_year",
    ]
    .astype(int)
    .tolist()
)

if warm_years:
    logistic_warm = (
        common.loc[
            common["Date"].dt.year.isin(warm_years),
            ["Date", "StockID", "rank_blend_5050"],
        ]
        .rename(
            columns={
                "rank_blend_5050": "up_prob"
            }
        )
        .copy()
    )
else:
    logistic_warm = pd.DataFrame(
        columns=[
            "Date",
            "StockID",
            "up_prob",
        ]
    )

logistic_signal = (
    pd.concat(
        [
            logistic_warm,
            logistic_estimated,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        ["Date", "StockID"],
        keep="last",
    )
    .sort_values(
        ["Date", "StockID"]
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# 2. Recover the saved walk-forward expected-return fusion
# -------------------------------------------------------------------------

expected_return_path = (
    OUTPUT_DIR
    / "expected_return_fusion_5d_predictions.parquet"
)

if not expected_return_path.exists():
    raise FileNotFoundError(
        f"Saved expected-return predictions not found: "
        f"{expected_return_path}"
    )

expected_return_fusion = pd.read_parquet(
    expected_return_path
)

expected_return_fusion["Date"] = pd.to_datetime(
    expected_return_fusion["Date"],
    errors="raise",
).dt.normalize()

expected_return_fusion["StockID"] = pd.to_numeric(
    expected_return_fusion["StockID"],
    errors="raise",
).astype(int).astype(str)


# -------------------------------------------------------------------------
# 3. Reconstruct the optimizer signal dictionary
# -------------------------------------------------------------------------

def signal_from_common(col):
    return (
        common[
            [
                "Date",
                "StockID",
                col,
            ]
        ]
        .rename(
            columns={
                col: "up_prob"
            }
        )
        .copy()
    )


signals = {
    "CNN": signal_from_common("p_cnn"),
    "RF": signal_from_common("p_rf"),
    "RankBlend_50_50": signal_from_common(
        "rank_blend_5050"
    ),
    "LogisticStack_WF": logistic_signal[
        [
            "Date",
            "StockID",
            "up_prob",
        ]
    ].copy(),
    "HMM": signal_from_common("p_hmm"),
}

expected_return_signal = (
    expected_return_fusion[
        [
            "Date",
            "StockID",
            "mu_hat_5d",
        ]
    ]
    .rename(
        columns={
            "mu_hat_5d": "up_prob"
        }
    )
    .copy()
)

optimizer_signals = dict(signals)

optimizer_signals[
    "ExpectedReturnFusion_5d"
] = expected_return_signal


# -------------------------------------------------------------------------
# 4. Verify that every recovered signal has the exact common key set
# -------------------------------------------------------------------------

for name, sig in optimizer_signals.items():

    sig["Date"] = pd.to_datetime(
        sig["Date"],
        errors="raise",
    ).dt.normalize()

    sig["StockID"] = pd.to_numeric(
        sig["StockID"],
        errors="raise",
    ).astype(int).astype(str)

    keys = (
        sig[
            [
                "Date",
                "StockID",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "Date",
                "StockID",
            ]
        )
        .reset_index(drop=True)
    )

    if (
        len(keys) != len(common_keys)
        or not keys.equals(common_keys)
    ):
        raise AssertionError(
            f"{name} does not contain the strict common key set."
        )

print("Recovered optimizer signals:")
for name, sig in optimizer_signals.items():
    print(
        f"  {name:25s}: {len(sig):,}"
    )


# -------------------------------------------------------------------------
# 5. Recover transaction-cost configuration
# -------------------------------------------------------------------------

capacity_kwargs = dict(
    SCREENED_KWARGS
)

for key in (
    "signal_df",
    "freq",
    "portfolio_dir",
    "start_year",
    "end_year",
    "eval_start_year",
    "country",
    "delay_list",
    "load_signal",
    "tc_aum_dollars",
    "tc_execution_days",
):
    capacity_kwargs.pop(
        key,
        None,
    )

capacity_kwargs.update(
    {
        "tc_enable": True,
        "tc_use_nonlinear": True,
        "tc_use_forecast_inputs": True,
        "tc_forecast_path": str(
            TC_FORECAST_PATH
        ),
        "tradability_screens": True,
        "include_price_adv": True,
        "tc_missing_return_max_carry_weeks": (
            MISSING_RETURN_MAX_CARRY_PERIODS
        ),
        "tc_missing_return_warn_weight": (
            MISSING_RETURN_WARN_WEIGHT
        ),
    }
)

if not TC_FORECAST_PATH.exists():
    raise FileNotFoundError(
        f"TC forecast file not found: {TC_FORECAST_PATH}"
    )

print("\nKernel-restart recovery: PASS")
print("Ready to resume at Section 28: Dynamic-NAV simulation.")

## 5. Realized 5-day outcomes

The static logistic fusion and expected-return fusion are trained walk-forward using realized weekly/5-day outcomes. A one-period label lag prevents the most recent unresolved label from entering each annual fit.


In [ ]:
# 5. Attach realized outcome labels

from Scripts.Data import equity_data as eqd
import gc


def load_forward_returns():
    # Do NOT .copy() the full return panel.
    ret = eqd.get_period_ret(
        FREQ,
        country=COUNTRY,
        include_price_adv=False,
        require_price_adv=False,
    )

    candidates = [
        f"next_{FREQ}_ret_0delay",
        f"next_{FREQ}_ret",
        f"Ret_{FREQ}",
        f"{FREQ}_ret",
        "ret",
        "RET",
        "Return",
    ]

    rcol = next((c for c in candidates if c in ret.columns), None)

    if rcol is None:
        raise KeyError(
            "Could not identify the forward-return column. "
            f"Available columns include: {list(ret.columns)[:100]}"
        )

    # Extract only the three columns we actually need as early as possible.
    if isinstance(ret.index, pd.MultiIndex):
        index_names = list(ret.index.names)

        if "Date" not in index_names or "StockID" not in index_names:
            raise KeyError(
                f"Expected Date and StockID in return index, found: {index_names}"
            )

        out = pd.DataFrame(
            {
                "Date": ret.index.get_level_values("Date"),
                "StockID": ret.index.get_level_values("StockID"),
                "fwd_ret": ret[rcol].to_numpy(),
            }
        )
    else:
        if "Date" not in ret.columns or "StockID" not in ret.columns:
            raise KeyError(
                "Return panel must contain Date and StockID "
                "either as columns or index levels."
            )

        out = ret[["Date", "StockID", rcol]].rename(
            columns={rcol: "fwd_ret"}
        )

    # Release the large source panel immediately.
    del ret
    gc.collect()

    out["Date"] = pd.to_datetime(
        out["Date"], errors="coerce"
    ).dt.normalize()

    out["StockID"] = pd.to_numeric(
        out["StockID"], errors="coerce"
    )

    out["fwd_ret"] = pd.to_numeric(
        out["fwd_ret"], errors="coerce"
    )

    out = out.dropna(
        subset=["Date", "StockID", "fwd_ret"]
    )

    # We only evaluate 2001-2024, with 2000 retained for historical fitting.
    out = out[
        out["Date"].dt.year.between(
            SOURCE_FIRST_YEAR,
            ANALYSIS_END_YEAR,
        )
    ]

    out["StockID"] = out["StockID"].astype(int).astype(str)
    out["y"] = (out["fwd_ret"] > 0.0).astype("int8")

    out = (
        out.drop_duplicates(
            ["Date", "StockID"],
            keep="last",
        )
        .sort_values(["Date", "StockID"])
        .reset_index(drop=True)
    )

    return out, rcol


labels, label_return_col = load_forward_returns()

model_panel = common_history.merge(
    labels,
    on=["Date", "StockID"],
    how="left",
)

label_coverage = model_panel["y"].notna().mean()

print("Forward-return column:", label_return_col)
print("Label coverage:", f"{label_coverage:.2%}")

if label_coverage <= 0.90:
    raise ValueError(
        "More than 10% of common-universe rows are missing labels. "
        "Inspect the return-column mapping before continuing."
    )

print(
    "Logistic modeling archive:",
    model_panel["Date"].min(),
    "to",
    model_panel["Date"].max(),
)

print(
    "Reported logistic predictions will still be restricted to:",
    ANALYSIS_START_YEAR,
    "-",
    ANALYSIS_END_YEAR,
)
# Release the source archive and standalone label table; model_panel now
# contains all history needed by the two walk-forward fusion models.
del labels
del common_history
gc.collect()


## 6. Walk-forward logistic stack

The logistic benchmark maps standardized CNN and RF logits into a 5-day probability without a hidden state. It is refit annually using only completed historical labels.


In [ ]:
# 6. Walk-forward standardized logistic stack

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def logit_clip(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def build_walk_forward_logistic_stack(panel: pd.DataFrame):
    full = panel.dropna(
        subset=["Date", "StockID", "p_cnn", "p_rf"]
    ).copy()

    labeled = full.dropna(subset=["y"]).copy()

    full["year"] = full["Date"].dt.year.astype(int)
    labeled["year"] = labeled["Date"].dt.year.astype(int)

    pred_parts = []
    coef_rows = []
    row_contrib_parts = []

    for target_year in range(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR + 1):
        target = full[full["year"] == target_year].copy()

        if target.empty:
            continue

        first_target_date = target["Date"].min()

        prior_dates = np.array(
            sorted(
                labeled.loc[
                    labeled["Date"] < first_target_date,
                    "Date",
                ].dropna().unique()
            )
        )

        # Earliest prediction year has no usable historical label history.
        if len(prior_dates) <= LOGIT_LABEL_LAG_PERIODS:
            warm = target[
                ["Date", "StockID", "rank_blend_5050"]
            ].copy()

            warm = warm.rename(
                columns={"rank_blend_5050": "up_prob"}
            )

            pred_parts.append(warm)

            coef_rows.append(
                {
                    "target_year": target_year,
                    "mode": "warm_start_rank_5050",
                    "train_rows": 0,
                    "max_train_date": pd.NaT,
                    "intercept": np.nan,
                    "beta_std_logit_cnn": np.nan,
                    "beta_std_logit_rf": np.nan,
                }
            )

            continue

        # With lag=1, omit the most recent historical weekly label.
        max_train_date = pd.Timestamp(
            prior_dates[-(LOGIT_LABEL_LAG_PERIODS + 1)]
        )

        train = labeled[
            labeled["Date"] <= max_train_date
        ].copy()

        X_train_raw = np.column_stack(
            [
                logit_clip(train["p_cnn"]),
                logit_clip(train["p_rf"]),
            ]
        )

        X_target_raw = np.column_stack(
            [
                logit_clip(target["p_cnn"]),
                logit_clip(target["p_rf"]),
            ]
        )

        y_train = train["y"].astype(int).to_numpy()

        if len(np.unique(y_train)) < 2:
            raise ValueError(
                f"Training labels have one class only for year {target_year}"
            )

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_target = scaler.transform(X_target_raw)

        model = LogisticRegression(
            C=float(LOGIT_C),
            penalty="l2",
            solver="lbfgs",
            max_iter=500,
            random_state=RANDOM_SEED,
        )

        model.fit(X_train, y_train)

        beta_cnn = float(model.coef_[0, 0])
        beta_rf = float(model.coef_[0, 1])

        target["up_prob"] = model.predict_proba(X_target)[:, 1]

        pred_parts.append(
            target[["Date", "StockID", "up_prob"]].copy()
        )

        coef_rows.append(
            {
                "target_year": target_year,
                "mode": "walk_forward_logistic",
                "train_rows": len(train),
                "max_train_date": max_train_date,
                "intercept": float(model.intercept_[0]),
                "beta_std_logit_cnn": beta_cnn,
                "beta_std_logit_rf": beta_rf,
            }
        )

        diag = target[["Date", "StockID"]].copy()
        diag["cnn_linear_contribution"] = beta_cnn * X_target[:, 0]
        diag["rf_linear_contribution"] = beta_rf * X_target[:, 1]

        row_contrib_parts.append(diag)

        print(
            f"[logistic] year={target_year} "
            f"train_rows={len(train):,} "
            f"through={max_train_date.date()} "
            f"beta_cnn={beta_cnn:.3f} "
            f"beta_rf={beta_rf:.3f}"
        )

    signal = pd.concat(pred_parts, ignore_index=True)
    coefs = pd.DataFrame(coef_rows)

    if row_contrib_parts:
        row_contrib = pd.concat(
            row_contrib_parts,
            ignore_index=True,
        )

        weekly = (
            row_contrib.groupby("Date")
            .agg(
                mean_abs_cnn_contribution=(
                    "cnn_linear_contribution",
                    lambda x: np.mean(np.abs(x)),
                ),
                mean_abs_rf_contribution=(
                    "rf_linear_contribution",
                    lambda x: np.mean(np.abs(x)),
                ),
                n_stocks=("StockID", "nunique"),
            )
            .reset_index()
            .sort_values("Date")
        )

        denom = (
            weekly["mean_abs_cnn_contribution"]
            + weekly["mean_abs_rf_contribution"]
        )

        weekly["cnn_influence_share"] = (
            weekly["mean_abs_cnn_contribution"] / denom
        )

        weekly["rf_influence_share"] = (
            weekly["mean_abs_rf_contribution"] / denom
        )
    else:
        row_contrib = pd.DataFrame()
        weekly = pd.DataFrame()

    return signal, coefs, row_contrib, weekly


(
    logistic_signal,
    logistic_coefs,
    logistic_row_contributions,
    logistic_weekly_influence,
) = build_walk_forward_logistic_stack(model_panel)

logistic_coefs.to_csv(
    OUTPUT_DIR / "logistic_stack_annual_coefficients.csv",
    index=False,
)

if len(logistic_row_contributions):
    logistic_row_contributions.to_parquet(
        OUTPUT_DIR / "logistic_stack_row_contributions.parquet",
        index=False,
    )

if len(logistic_weekly_influence):
    logistic_weekly_influence.to_csv(
        OUTPUT_DIR / "logistic_stack_weekly_influence.csv",
        index=False,
    )

display(logistic_coefs)


### Logistic coefficient and influence diagnostics


In [ ]:
# 7. Plot annual coefficients and weekly logistic influence

import matplotlib.pyplot as plt

coef_plot = logistic_coefs.dropna(
    subset=["beta_std_logit_cnn", "beta_std_logit_rf"]
).copy()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    coef_plot["target_year"],
    coef_plot["beta_std_logit_cnn"],
    marker="o",
    label="CNN standardized coefficient",
)

ax.plot(
    coef_plot["target_year"],
    coef_plot["beta_std_logit_rf"],
    marker="o",
    label="RF standardized coefficient",
)

ax.axhline(0.0, linewidth=1)
ax.set_title("Logistic Stack: Annual Standardized Coefficients")
ax.set_xlabel("Prediction year")
ax.set_ylabel("Coefficient")
ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)


if len(logistic_weekly_influence):
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(
        logistic_weekly_influence["Date"],
        logistic_weekly_influence["cnn_influence_share"],
        label="CNN weekly influence",
    )

    ax.plot(
        logistic_weekly_influence["Date"],
        logistic_weekly_influence["rf_influence_share"],
        label="RF weekly influence",
    )

    ax.axhline(
        0.5,
        linestyle="--",
        linewidth=1,
        label="Equal influence",
    )

    ax.set_title(
        "Logistic Stack: Weekly Realized CNN vs RF Influence"
    )
    ax.set_xlabel("Week")
    ax.set_ylabel("Normalized mean absolute contribution")
    ax.set_ylim(-0.03, 1.03)
    ax.legend()

    fig.tight_layout()
    plt.show()
    plt.close(fig)


## 7. Walk-forward expected-return fusion

This benchmark maps the two expert logits directly into an expected 5-day return using expanding-window Ridge regression. It is an alternative to probability-space fusion and is evaluated as a score/alpha rather than as a calibrated probability.


In [ ]:
# 22. Walk-forward CNN + RF expected-return fusion.

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler


def build_walk_forward_expected_return_fusion(
    panel: pd.DataFrame,
):
    full = panel.dropna(
        subset=[
            "Date",
            "StockID",
            "p_cnn",
            "p_rf",
        ]
    ).copy()

    labeled = full.dropna(
        subset=["fwd_ret"]
    ).copy()

    full["year"] = (
        full["Date"]
        .dt.year
        .astype(int)
    )

    pred_parts = []
    coef_rows = []

    for target_year in range(
        ANALYSIS_START_YEAR,
        ANALYSIS_END_YEAR + 1,
    ):
        target = full[
            full["year"] == target_year
        ].copy()

        if target.empty:
            continue

        first_target_date = target[
            "Date"
        ].min()

        prior_dates = np.array(
            sorted(
                labeled.loc[
                    labeled["Date"]
                    < first_target_date,
                    "Date",
                ]
                .dropna()
                .unique()
            )
        )

        if len(prior_dates) <= RETURN_FUSION_LABEL_LAG_PERIODS:
            raise RuntimeError(
                f"Insufficient pre-{target_year} labeled history "
                "for the expected-return fusion. "
                "The archive should contain partial 2000 history."
            )

        max_train_date = pd.Timestamp(
            prior_dates[
                -(
                    RETURN_FUSION_LABEL_LAG_PERIODS
                    + 1
                )
            ]
        )

        train = labeled[
            labeled["Date"]
            <= max_train_date
        ].copy()

        X_train_raw = np.column_stack(
            [
                logit_clip(
                    train["p_cnn"]
                ),
                logit_clip(
                    train["p_rf"]
                ),
            ]
        )

        X_target_raw = np.column_stack(
            [
                logit_clip(
                    target["p_cnn"]
                ),
                logit_clip(
                    target["p_rf"]
                ),
            ]
        )

        y_train = pd.to_numeric(
            train["fwd_ret"],
            errors="coerce",
        ).to_numpy(float)

        finite = (
            np.isfinite(X_train_raw).all(axis=1)
            & np.isfinite(y_train)
        )

        X_train_raw = X_train_raw[
            finite
        ]

        y_train = y_train[
            finite
        ]

        if len(y_train) < 1000:
            raise RuntimeError(
                f"Too few training rows for year {target_year}: "
                f"{len(y_train):,}"
            )

        scaler = StandardScaler()

        X_train = scaler.fit_transform(
            X_train_raw
        )

        X_target = scaler.transform(
            X_target_raw
        )

        model = Ridge(
            alpha=float(
                RETURN_FUSION_RIDGE_ALPHA
            ),
            fit_intercept=True,
        )

        model.fit(
            X_train,
            y_train,
        )

        target[
            "mu_hat_5d"
        ] = model.predict(
            X_target
        )

        pred_parts.append(
            target[
                [
                    "Date",
                    "StockID",
                    "mu_hat_5d",
                ]
            ].copy()
        )

        train_pred = model.predict(
            X_train
        )

        ss_res = float(
            np.sum(
                (
                    y_train
                    - train_pred
                ) ** 2
            )
        )

        ss_tot = float(
            np.sum(
                (
                    y_train
                    - y_train.mean()
                ) ** 2
            )
        )

        train_r2 = (
            1.0
            - ss_res / ss_tot
            if ss_tot > 0
            else np.nan
        )

        coef_rows.append(
            {
                "target_year": target_year,
                "train_rows": len(
                    y_train
                ),
                "max_train_date": max_train_date,
                "intercept": float(
                    model.intercept_
                ),
                "beta_std_logit_cnn": float(
                    model.coef_[0]
                ),
                "beta_std_logit_rf": float(
                    model.coef_[1]
                ),
                "train_r2": train_r2,
                "pred_mean": float(
                    np.mean(
                        target[
                            "mu_hat_5d"
                        ]
                    )
                ),
                "pred_std": float(
                    np.std(
                        target[
                            "mu_hat_5d"
                        ],
                        ddof=1,
                    )
                ),
            }
        )

        print(
            f"[5d return fusion] "
            f"{target_year}: "
            f"train={len(y_train):,}, "
            f"max_train={max_train_date.date()}, "
            f"beta_CNN={model.coef_[0]:+.6f}, "
            f"beta_RF={model.coef_[1]:+.6f}, "
            f"train_R2={train_r2:.6f}"
        )

    out = pd.concat(
        pred_parts,
        ignore_index=True,
    )

    out["Date"] = pd.to_datetime(
        out["Date"],
        errors="coerce",
    ).dt.normalize()

    out["StockID"] = (
        out["StockID"]
        .astype(str)
    )

    out = (
        out.drop_duplicates(
            [
                "Date",
                "StockID",
            ],
            keep="last",
        )
        .sort_values(
            [
                "Date",
                "StockID",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    coefs = pd.DataFrame(
        coef_rows
    )

    return out, coefs


(
    expected_return_fusion,
    expected_return_fusion_coefs,
) = build_walk_forward_expected_return_fusion(
    model_panel
)

expected_return_fusion.to_parquet(
    OUTPUT_DIR
    / "expected_return_fusion_5d_predictions.parquet",
    index=False,
)

expected_return_fusion_coefs.to_csv(
    OUTPUT_DIR
    / "expected_return_fusion_5d_coefficients.csv",
    index=False,
)

display(
    expected_return_fusion_coefs.style.format(
        {
            "intercept": "{:+.6f}",
            "beta_std_logit_cnn": "{:+.6f}",
            "beta_std_logit_rf": "{:+.6f}",
            "train_r2": "{:.6f}",
            "pred_mean": "{:+.6f}",
            "pred_std": "{:.6f}",
        }
    )
)

# Hard key check against the exact 2001–2024 common evaluation panel.
expected_keys = (
    expected_return_fusion[
        [
            "Date",
            "StockID",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Date",
            "StockID",
        ]
    )
    .reset_index(
        drop=True
    )
)

if (
    len(expected_keys)
    != len(common_keys)
    or not expected_keys.equals(
        common_keys
    )
):
    raise AssertionError(
        "Expected-return fusion does not contain "
        "the exact strict-common 2001–2024 key set."
    )

print(
    "\nExpected-return fusion key check: PASS"
)

display(
    expected_return_fusion[
        "mu_hat_5d"
    ].describe().to_frame(
        "mu_hat_5d"
    )
)

# The walk-forward logistic and expected-return predictions are now frozen.
# Release the large labeled modeling panel before portfolio analysis.
del model_panel
gc.collect()


## 8. Assemble the comparison signals

The five primary probability/rank signals are retained for the original fusion ablation. The expected-return fusion is added to the broader portfolio comparison.


In [ ]:
def signal_from_common(col):
    return (
        common[["Date", "StockID", col]]
        .rename(columns={col: "up_prob"})
        .copy()
    )

signals = {
    "CNN": signal_from_common("p_cnn"),
    "RF": signal_from_common("p_rf"),
    "RankBlend_50_50": signal_from_common("rank_blend_5050"),
    "LogisticStack_WF": logistic_signal[["Date", "StockID", "up_prob"]].copy(),
    "HMM": signal_from_common("p_hmm"),
}

expected_return_signal = (
    expected_return_fusion[["Date", "StockID", "mu_hat_5d"]]
    .rename(columns={"mu_hat_5d": "up_prob"})
    .copy()
)

optimizer_signals = dict(signals)
optimizer_signals["ExpectedReturnFusion_5d"] = expected_return_signal

for name, sig in optimizer_signals.items():
    sig["Date"] = pd.to_datetime(sig["Date"], errors="coerce").dt.normalize()
    sig["StockID"] = sig["StockID"].astype(str)

    keys = (
        sig[["Date", "StockID"]]
        .drop_duplicates()
        .sort_values(["Date", "StockID"])
        .reset_index(drop=True)
    )

    if len(keys) != len(common_keys) or not keys.equals(common_keys):
        raise AssertionError(f"{name} does not contain the strict common key set.")

print("Signals:")
for name, sig in optimizer_signals.items():
    print(f"  {name:25s}: {len(sig):,}")


## 9. Signal complementarity

Complementarity is examined before portfolio optimization. The main quantities are weekly cross-sectional correlation and top/bottom-decile overlap. Low correlation or low tail overlap indicates that the experts rank stocks differently.


In [ ]:
def weekly_cross_sectional_corr(df, x, y, method):
    vals = []
    for date, g in df.groupby("Date", sort=True):
        pair = g[[x, y]].dropna()
        if len(pair) < 10:
            continue
        corr = pair[x].corr(pair[y], method=method)
        if np.isfinite(corr):
            vals.append(corr)
    return float(np.mean(vals)) if vals else np.nan

def tail_set_overlap(df, x, y, tail="high", q=0.10):
    vals = []
    for date, g in df.groupby("Date", sort=True):
        pair = g[["StockID", x, y]].dropna()
        if len(pair) < 20:
            continue

        if tail == "high":
            tx = pair[x].quantile(1.0 - q)
            ty = pair[y].quantile(1.0 - q)
            sx = set(pair.loc[pair[x] >= tx, "StockID"])
            sy = set(pair.loc[pair[y] >= ty, "StockID"])
        else:
            tx = pair[x].quantile(q)
            ty = pair[y].quantile(q)
            sx = set(pair.loc[pair[x] <= tx, "StockID"])
            sy = set(pair.loc[pair[y] <= ty, "StockID"])

        denom = min(len(sx), len(sy))
        if denom > 0:
            vals.append(len(sx & sy) / float(denom))

    return float(np.mean(vals)) if vals else np.nan

logit_panel = logistic_signal.rename(columns={"up_prob": "p_logistic"})
signal_panel = (
    common.merge(
        logit_panel[["Date", "StockID", "p_logistic"]],
        on=["Date", "StockID"],
        how="inner",
    )
)

complementarity_specs = [
    ("CNN vs RF", "p_cnn", "p_rf"),
    ("HMM vs RF", "p_hmm", "p_rf"),
    ("HMM vs CNN", "p_hmm", "p_cnn"),
    ("HMM vs Logistic", "p_hmm", "p_logistic"),
]

rows = []
for label, x, y in complementarity_specs:
    rows.append(
        {
            "comparison": label,
            "pearson": weekly_cross_sectional_corr(signal_panel, x, y, "pearson"),
            "spearman": weekly_cross_sectional_corr(signal_panel, x, y, "spearman"),
            "top_decile_overlap": tail_set_overlap(signal_panel, x, y, "high"),
            "bottom_decile_overlap": tail_set_overlap(signal_panel, x, y, "low"),
        }
    )

complementarity = pd.DataFrame(rows)
complementarity.to_csv(OUTPUT_DIR / "signal_complementarity.csv", index=False)

display(
    complementarity.style.format(
        {
            "pearson": "{:+.3f}",
            "spearman": "{:+.3f}",
            "top_decile_overlap": "{:.1%}",
            "bottom_decile_overlap": "{:.1%}",
        }
    )
)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(complementarity))
width = 0.35
ax.bar(x - width / 2, complementarity["top_decile_overlap"], width, label="Top decile")
ax.bar(x + width / 2, complementarity["bottom_decile_overlap"], width, label="Bottom decile")
ax.set_xticks(x)
ax.set_xticklabels(complementarity["comparison"], rotation=20, ha="right")
ax.set_ylabel("Mean membership overlap")
ax.set_title("Cross-Sectional Tail Overlap")
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)


# Part I — Gross signal and portfolio evidence

The first part asks whether the signals are predictive and distinct before implementation costs. It preserves the core research questions from the earlier notebooks and uses the strict common universe.


## 10. Screened equal-weight high-minus-low portfolios

Each signal is ranked within the same screened universe. The gross equal-weight portfolio is the cleanest first-stage comparison because it isolates signal quality from optimizer-specific choices.


In [ ]:

from Scripts.Portfolio.portfolio import PortfolioManager


def calculate_gross_hl(signal_df, name, weight_type="ew"):
    """Run one gross portfolio and release the manager immediately."""

    kwargs = dict(GROSS_SCREENED_KWARGS)

    out_dir = OUTPUT_DIR / "gross" / weight_type / name
    out_dir.mkdir(parents=True, exist_ok=True)

    pm = PortfolioManager(
        signal_df=signal_df.copy(),
        freq=FREQ,
        portfolio_dir=str(out_dir),
        start_year=ANALYSIS_START_YEAR,
        end_year=ANALYSIS_END_YEAR,
        eval_start_year=ANALYSIS_START_YEAR,
        country=COUNTRY,
        delay_list=[DELAY],
        load_signal=True,
        **kwargs,
    )

    pf_ret, avg_turnover = pm.calculate_portfolio_rets(
        weight_type=weight_type,
        cut=CUT,
        delay=DELAY,
    )

    if "H-L" not in pf_ret.columns:
        raise KeyError(f"{name}: PortfolioManager output has no H-L column.")

    idx = pd.to_datetime(
        pf_ret.index,
        errors="coerce",
    ).normalize()

    out = pd.Series(
        pd.to_numeric(
            pf_ret["H-L"],
            errors="coerce",
        ).to_numpy(),
        index=idx,
        name=name,
    )

    out = out.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    if out.isna().any():
        raise RuntimeError(
            f"{name}: gross portfolio contains missing/non-finite returns."
        )

    result = out.sort_index()
    turnover = float(avg_turnover)

    # PortfolioManager may hold large prepared/covariance panels.
    # Nothing downstream needs the manager object for gross runs.
    del pf_ret
    del pm
    gc.collect()

    return result, turnover


GROSS_EW_CACHE = OUTPUT_DIR / "gross_ew_hl_returns.csv"

if RUN_GROSS_EW:
    gross_ew_runs = {}
    gross_ew_turnover = {}

    for name, sig in optimizer_signals.items():
        print("Gross EW:", name)

        r, to = calculate_gross_hl(
            sig,
            name,
            weight_type="ew",
        )

        gross_ew_runs[name] = r
        gross_ew_turnover[name] = to

    hl = pd.concat(
        gross_ew_runs,
        axis=1,
    )

    hl.to_csv(GROSS_EW_CACHE)

    del gross_ew_runs
    gc.collect()

else:
    if not GROSS_EW_CACHE.exists():
        raise FileNotFoundError(
            "RUN_GROSS_EW=False but no saved gross EW return panel exists."
        )

    hl = pd.read_csv(
        GROSS_EW_CACHE,
        index_col=0,
        parse_dates=True,
    )

    gross_ew_turnover = {}

if hl.isna().any().any():
    raise RuntimeError(
        "Gross EW return panel is not a strict complete weekly panel."
    )

print(
    "Gross EW panel:",
    len(hl),
    "weeks |",
    hl.index.min(),
    "to",
    hl.index.max(),
)


## 11. Gross performance and cumulative return

Use both the table and the path. A high full-sample Sharpe is more credible when the cumulative return does not depend on one short interval and the rolling performance remains reasonably stable.


In [ ]:
from Scripts.Portfolio import statistical_validation as sv

gross_perf = pd.DataFrame(
    {
        name: sv.performance_stats(hl[name], periods_per_year=52)
        for name in hl.columns
    }
).T

if gross_ew_turnover:
    gross_perf["avg_turnover"] = pd.Series(gross_ew_turnover)

gross_perf.to_csv(OUTPUT_DIR / "gross_ew_performance.csv")

display(
    gross_perf.style.format(
        {
            "ann_mean": "{:.2%}",
            "ann_vol": "{:.2%}",
            "sharpe": "{:.3f}",
            "avg_turnover": "{:.3f}",
        }
    )
)

fig, ax = plt.subplots(figsize=(14, 6))
wealth = (1.0 + hl).cumprod()
for name in wealth.columns:
    ax.plot(wealth.index, wealth[name], label=name)
ax.set_title("Gross Equal-Weight H-L Cumulative Growth")
ax.set_xlabel("Date")
ax.set_ylabel("Growth of $1")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ordered = gross_perf["sharpe"].sort_values()
ax.barh(ordered.index, ordered.values)
ax.axvline(0.0, linewidth=1)
ax.set_title("Gross Equal-Weight Annualized Sharpe")
ax.set_xlabel("Sharpe ratio")
fig.tight_layout()
plt.show()
plt.close(fig)


## 12. Gross statistical inference

Two distinct questions are tested:

- **Mean-return difference:** paired HAC/Newey-West inference on `R_A - R_B`.
- **Risk-adjusted difference:** paired circular moving-block bootstrap inference on `Sharpe_A - Sharpe_B`.

The final statistical module requires identical dates by default; sample mismatch is treated as an error rather than silently intersected away.


In [ ]:
gross_comparisons = [
    ("HMM", "RF"),
    ("HMM", "RankBlend_50_50"),
    ("HMM", "LogisticStack_WF"),
    ("HMM", "ExpectedReturnFusion_5d"),
]

gross_hac, gross_sharpe_boot = sv.pairwise_inference_table(
    hl,
    comparisons=gross_comparisons,
    maxlags=HAC_LAGS,
    reps=BOOTSTRAP_REPS,
    block_len=BOOTSTRAP_BLOCK_WEEKS,
    seed=RANDOM_SEED,
    periods_per_year=52,
    require_identical_index=True,
)

gross_hac.to_csv(OUTPUT_DIR / "gross_paired_hac.csv", index=False)
gross_sharpe_boot.to_csv(OUTPUT_DIR / "gross_sharpe_difference_bootstrap.csv", index=False)

print("PAIRED HAC — MEAN RETURN DIFFERENCES")
display(
    gross_hac[
        [
            "model_a", "model_b", "n", "ann_mean",
            "ann_mean_ci_low", "ann_mean_ci_high",
            "hac_t", "p_value_two_sided",
        ]
    ].style.format(
        {
            "ann_mean": "{:+.2%}",
            "ann_mean_ci_low": "{:+.2%}",
            "ann_mean_ci_high": "{:+.2%}",
            "hac_t": "{:.3f}",
            "p_value_two_sided": "{:.4g}",
        }
    )
)

print("PAIRED BLOCK BOOTSTRAP — SHARPE DIFFERENCES")
display(
    gross_sharpe_boot[
        [
            "model_a", "model_b", "n_periods",
            "observed_delta_sharpe", "ci_low", "ci_high",
            "bootstrap_prob_delta_gt_0",
        ]
    ].style.format(
        {
            "observed_delta_sharpe": "{:+.3f}",
            "ci_low": "{:+.3f}",
            "ci_high": "{:+.3f}",
            "bootstrap_prob_delta_gt_0": "{:.3f}",
        }
    )
)

sensitivity_parts = []
for i, (a, b) in enumerate([
    ("HMM", "RF"),
    ("HMM", "RankBlend_50_50"),
    ("HMM", "LogisticStack_WF"),
    ("HMM", "ExpectedReturnFusion_5d"),
]):
    d = sv.bootstrap_block_length_sensitivity(
        hl[a],
        hl[b],
        a,
        b,
        block_lengths=BLOCK_LENGTH_SENSITIVITY,
        reps=BOOTSTRAP_REPS,
        seed=RANDOM_SEED + 100 * i,
        periods_per_year=52,
        require_identical_index=True,
    )
    sensitivity_parts.append(d)

gross_block_sensitivity = pd.concat(sensitivity_parts, ignore_index=True)
gross_block_sensitivity.to_csv(
    OUTPUT_DIR / "gross_sharpe_block_length_sensitivity.csv",
    index=False,
)

display(
    gross_block_sensitivity[
        ["model_a", "model_b", "block_len", "observed_delta_sharpe", "ci_low", "ci_high"]
    ].style.format(
        {
            "observed_delta_sharpe": "{:+.3f}",
            "ci_low": "{:+.3f}",
            "ci_high": "{:+.3f}",
        }
    )
)


## 13. Return spanning, annual stability, and rolling Sharpe

Spanning asks whether HMM returns contain an intercept after controlling for simpler return streams. Rolling and calendar-year results test whether the aggregate result is stable rather than concentrated in a small part of the sample.


In [ ]:
spanning_specs = [
    ("HMM ~ RF", "HMM", ["RF"]),
    ("HMM ~ CNN", "HMM", ["CNN"]),
    ("HMM ~ RF + CNN", "HMM", ["RF", "CNN"]),
    ("HMM ~ 50/50 rank blend", "HMM", ["RankBlend_50_50"]),
    ("HMM ~ logistic stack", "HMM", ["LogisticStack_WF"]),
    ("HMM ~ expected-return fusion", "HMM", ["ExpectedReturnFusion_5d"]),
    ("HMM ~ RF + logistic stack", "HMM", ["RF", "LogisticStack_WF"]),
]

spanning_results = sv.spanning_table(
    hl,
    specs=spanning_specs,
    maxlags=HAC_LAGS,
    periods_per_year=52,
)
spanning_results.to_csv(OUTPUT_DIR / "gross_spanning_regressions.csv", index=False)

display(
    spanning_results.style.format(
        {
            "ann_alpha": "{:+.2%}",
            "ann_alpha_ci_low": "{:+.2%}",
            "ann_alpha_ci_high": "{:+.2%}",
            "alpha_t_hac": "{:.3f}",
            "alpha_p_two_sided": "{:.4g}",
            "r_squared": "{:.4f}",
        }
    )
)

annual_perf = sv.annual_model_stats(hl, periods_per_year=52)
annual_perf.to_csv(OUTPUT_DIR / "gross_annual_model_stats.csv", index=False)

annual_sharpe = annual_perf.pivot(index="year", columns="model", values="sharpe")
display(annual_sharpe.round(3))

year_wins = pd.DataFrame(
    [
        sv.year_win_summary(annual_perf, benchmark="RF", focal_model="HMM"),
        sv.year_win_summary(annual_perf, benchmark="RankBlend_50_50", focal_model="HMM"),
        sv.year_win_summary(annual_perf, benchmark="LogisticStack_WF", focal_model="HMM"),
        sv.year_win_summary(annual_perf, benchmark="ExpectedReturnFusion_5d", focal_model="HMM"),
    ]
)
year_wins.to_csv(OUTPUT_DIR / "gross_hmm_calendar_year_wins.csv", index=False)
display(year_wins)

rolling52 = pd.DataFrame(index=hl.index)
rolling104 = pd.DataFrame(index=hl.index)

for model in ["RF", "RankBlend_50_50", "LogisticStack_WF", "ExpectedReturnFusion_5d", "HMM"]:
    rolling52[model] = sv.rolling_sharpe(hl[model], 52, periods_per_year=52)
    rolling104[model] = sv.rolling_sharpe(hl[model], 104, periods_per_year=52)

fig, ax = plt.subplots(figsize=(14, 5))
for model in rolling52.columns:
    ax.plot(rolling52.index, rolling52[model], label=model)
ax.axhline(0.0, linewidth=1)
ax.set_title("Rolling 52-Week Gross Sharpe")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized Sharpe")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 5))
for model in rolling104.columns:
    ax.plot(rolling104.index, rolling104[model], label=model)
ax.axhline(0.0, linewidth=1)
ax.set_title("Rolling 104-Week Gross Sharpe")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized Sharpe")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()
plt.close(fig)


# Part II — HMM mechanism and portfolio diagnostics

These sections test whether state sensitivity explains incremental performance, how the HMM maps the two experts through time, how much its tails differ from simpler benchmarks, and whether the signal is more persistent.


## 14. HMM state probabilities and CNN/RF attribution

State labels are fit-specific; interpretation focuses on probability concentration, entropy, and the implied expert mapping rather than assigning permanent economic names to arbitrary state numbers.


In [ ]:
# 14. HMM weekly state diagnostics and post-hoc CNN/RF attribution.

from sklearn.linear_model import LinearRegression


def _standardize_state_file(df: pd.DataFrame, target_year: int) -> pd.DataFrame:
    out = df.copy()

    if isinstance(out.index, pd.MultiIndex):
        out = out.reset_index()

    out["Date"] = pd.to_datetime(
        out["Date"],
        errors="coerce",
    ).dt.normalize()

    # The annual file may contain historical state probabilities.
    # Retain only the target year's dates from that annual fit.
    out = out[
        out["Date"].dt.year == int(target_year)
    ].copy()

    if {"state", "prob"}.issubset(out.columns):
        out["state"] = pd.to_numeric(out["state"], errors="coerce")
        out["prob"] = pd.to_numeric(out["prob"], errors="coerce")
        out = out.dropna(subset=["Date", "state", "prob"]).copy()
        out["state"] = out["state"].astype(int)
        return out[["Date", "state", "prob"]]

    # Wide fallback: state_0, state_1, ... or prob_state_0, ...
    state_cols = [
        c for c in out.columns
        if re.search(r"(?:state|prob).*?\d+$", str(c), flags=re.I)
    ]

    if not state_cols:
        raise KeyError(
            f"Could not identify state-probability columns for {target_year}. "
            f"Columns={list(out.columns)}"
        )

    melted = out[["Date"] + state_cols].melt(
        id_vars="Date",
        var_name="state_col",
        value_name="prob",
    )

    melted["state"] = (
        melted["state_col"]
        .str.extract(r"(\d+)$")[0]
        .astype(int)
    )

    melted["prob"] = pd.to_numeric(
        melted["prob"],
        errors="coerce",
    )

    return melted.dropna(
        subset=["Date", "state", "prob"]
    )[["Date", "state", "prob"]]


def load_all_target_year_state_probs(
    start_year: int,
    end_year: int,
):
    parts = []
    missing = []

    for year in range(int(start_year), int(end_year) + 1):
        fp = (
            ANNUAL_ROOT
            / f"year_{year}"
            / f"hmm_state_probs_{year}.parquet"
        )

        if not fp.exists():
            missing.append(year)
            continue

        raw = pd.read_parquet(fp)
        one = _standardize_state_file(raw, year)

        if len(one):
            one["fit_year"] = int(year)
            parts.append(one)

    if not parts:
        raise FileNotFoundError(
            "No annual HMM state-probability files were found."
        )

    out = pd.concat(parts, ignore_index=True)

    # Re-normalize tiny numerical deviations within each date.
    totals = out.groupby("Date")["prob"].transform("sum")
    out["prob"] = np.where(
        totals > 0,
        out["prob"] / totals,
        np.nan,
    )

    out = out.dropna(subset=["prob"]).copy()

    return out, missing


hmm_state_probs_all, missing_state_years = load_all_target_year_state_probs(
    ANALYSIS_START_YEAR,
    ANALYSIS_END_YEAR,
)

if missing_state_years:
    print("Missing state files:", missing_state_years)

print(
    "Exact state-probability date range:",
    hmm_state_probs_all["Date"].min(),
    "to",
    hmm_state_probs_all["Date"].max(),
)


def summarize_state_probabilities(state_long: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for dt, g in state_long.groupby("Date", sort=True):
        p = np.asarray(g["prob"], dtype=float)
        states = np.asarray(g["state"])

        valid = np.isfinite(p) & (p >= 0)
        p = p[valid]
        states = states[valid]

        if len(p) == 0 or p.sum() <= 0:
            continue

        p = p / p.sum()
        k = len(p)

        if k <= 1:
            entropy_norm = 0.0
        else:
            entropy = -float(np.sum(p * np.log(np.clip(p, 1e-12, 1.0))))
            entropy_norm = entropy / np.log(k)

        j = int(np.argmax(p))

        rows.append(
            {
                "Date": pd.Timestamp(dt),
                "fit_year": int(pd.Timestamp(dt).year),
                "n_states": int(k),
                "dominant_state": int(states[j]),
                "max_state_prob": float(p[j]),
                "state_entropy": float(entropy_norm),
            }
        )

    out = pd.DataFrame(rows).sort_values("Date").reset_index(drop=True)

    # State labels only have meaning inside one annual fit.
    out["state_transition"] = np.nan

    for year, idx in out.groupby("fit_year").groups.items():
        loc = list(idx)
        s = out.loc[loc, "dominant_state"]
        changed = s.ne(s.shift(1))
        changed.iloc[0] = np.nan
        out.loc[loc, "state_transition"] = changed.astype(float).to_numpy()

    return out


hmm_state_weekly = summarize_state_probabilities(
    hmm_state_probs_all
)

hmm_state_weekly.to_csv(
    OUTPUT_DIR / "hmm_weekly_state_summary.csv",
    index=False,
)

display(hmm_state_weekly.head())


# ------------------------------------------------------------
# A. Exact state probabilities for one target year
# ------------------------------------------------------------

STATE_YEAR = ANALYSIS_END_YEAR

state_plot = hmm_state_probs_all[
    hmm_state_probs_all["Date"].dt.year == int(STATE_YEAR)
].copy()

pivot = (
    state_plot.pivot_table(
        index="Date",
        columns="state",
        values="prob",
        aggfunc="last",
    )
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 5))

for state in pivot.columns:
    ax.plot(
        pivot.index,
        pivot[state],
        label=f"State {state}",
    )

ax.set_title(
    f"HMM Exact Weekly State Probabilities — {STATE_YEAR}"
)
ax.set_xlabel("Week")
ax.set_ylabel("State probability")
ax.set_ylim(-0.03, 1.03)
ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)


# ------------------------------------------------------------
# B. Full-sample weekly CNN/RF post-hoc attribution proxy
# ------------------------------------------------------------

def weekly_hmm_attribution_proxy(
    panel: pd.DataFrame,
    min_stocks: int = 50,
) -> pd.DataFrame:

    dfx = panel[
        [
            "Date",
            "StockID",
            "p_cnn",
            "p_rf",
            "p_hmm",
        ]
    ].dropna().copy()

    rows = []

    for dt, g in dfx.groupby("Date", sort=True):
        if len(g) < int(min_stocks):
            continue

        X_raw = np.column_stack(
            [
                logit_clip(g["p_cnn"]),
                logit_clip(g["p_rf"]),
            ]
        )

        y = logit_clip(g["p_hmm"])

        scaler = StandardScaler()
        X = scaler.fit_transform(X_raw)

        model = LinearRegression()
        model.fit(X, y)

        beta_cnn = float(model.coef_[0])
        beta_rf = float(model.coef_[1])

        cnn_contrib = beta_cnn * X[:, 0]
        rf_contrib = beta_rf * X[:, 1]

        mean_abs_cnn = float(np.mean(np.abs(cnn_contrib)))
        mean_abs_rf = float(np.mean(np.abs(rf_contrib)))
        denom = mean_abs_cnn + mean_abs_rf

        rows.append(
            {
                "Date": pd.Timestamp(dt),
                "n_stocks": int(len(g)),
                "beta_std_logit_cnn": beta_cnn,
                "beta_std_logit_rf": beta_rf,
                "cnn_influence_share": (
                    mean_abs_cnn / denom
                    if denom > 0
                    else np.nan
                ),
                "rf_influence_share": (
                    mean_abs_rf / denom
                    if denom > 0
                    else np.nan
                ),
                "cross_sectional_r2": float(
                    model.score(X, y)
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values("Date")
        .reset_index(drop=True)
    )


hmm_weekly_attribution = weekly_hmm_attribution_proxy(
    common
)

hmm_weekly_attribution.to_csv(
    OUTPUT_DIR / "hmm_weekly_cnn_rf_attribution_proxy.csv",
    index=False,
)


fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    hmm_weekly_attribution["Date"],
    hmm_weekly_attribution["cnn_influence_share"],
    label="CNN attribution share",
)

ax.plot(
    hmm_weekly_attribution["Date"],
    hmm_weekly_attribution["rf_influence_share"],
    label="RF attribution share",
)

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
    label="Equal attribution",
)

ax.set_title(
    "HMM: Weekly CNN vs RF Post-Hoc Attribution"
)
ax.set_xlabel("Week")
ax.set_ylabel("Normalized mean absolute contribution")
ax.set_ylim(-0.03, 1.03)
ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)


fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(
    hmm_weekly_attribution["Date"],
    hmm_weekly_attribution["cross_sectional_r2"],
)

ax.set_title(
    "HMM Weekly Attribution Diagnostic: Cross-Sectional R²"
)
ax.set_xlabel("Week")
ax.set_ylabel("R²")
ax.set_ylim(-0.03, 1.03)

fig.tight_layout()
plt.show()
plt.close(fig)

display(
    hmm_weekly_attribution.describe().T
)


## 15. State-conditional incremental performance


In [ ]:
# 17. State-conditional HMM-versus-logistic performance.

state_perf = (
    hmm_state_weekly
    .merge(
        hl.reset_index().rename(
            columns={
                hl.index.name or "index": "Date",
            }
        ),
        on="Date",
        how="inner",
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

state_perf["HMM_minus_Logistic"] = (
    state_perf["HMM"]
    - state_perf["LogisticStack_WF"]
)

state_perf["HMM_minus_RF"] = (
    state_perf["HMM"]
    - state_perf["RF"]
)

# Entropy quartiles are defined from percentile ranks over the full
# 2001–2024 evaluation sample. Rank-based bins avoid qcut failures when
# repeated entropy values create duplicate quantile edges.
_entropy_pct = state_perf["state_entropy"].rank(
    method="average",
    pct=True,
)

state_perf["entropy_quartile"] = pd.cut(
    _entropy_pct,
    bins=[0.0, 0.25, 0.50, 0.75, 1.0],
    labels=[
        "Q1: lowest entropy",
        "Q2",
        "Q3",
        "Q4: highest entropy",
    ],
    include_lowest=True,
)


def conditional_perf_table(
    df: pd.DataFrame,
    group_col: str,
    model_a: str = "HMM",
    model_b: str = "LogisticStack_WF",
):
    rows = []

    for key, g in df.groupby(group_col, observed=True):
        a = pd.to_numeric(g[model_a], errors="coerce").dropna()
        b = pd.to_numeric(g[model_b], errors="coerce").dropna()
        pair = g[[model_a, model_b]].dropna()

        if len(pair) < 5:
            continue

        sr_a = annualized_sharpe(pair[model_a].to_numpy(float))
        sr_b = annualized_sharpe(pair[model_b].to_numpy(float))

        rows.append(
            {
                group_col: key,
                "n_weeks": len(pair),
                "HMM_ann_mean": float(pair[model_a].mean() * PERIODS_PER_YEAR),
                "benchmark_ann_mean": float(pair[model_b].mean() * PERIODS_PER_YEAR),
                "HMM_minus_benchmark_ann_mean": float(
                    (pair[model_a] - pair[model_b]).mean()
                    * PERIODS_PER_YEAR
                ),
                "HMM_sharpe": sr_a,
                "benchmark_sharpe": sr_b,
                "delta_sharpe": sr_a - sr_b,
            }
        )

    return pd.DataFrame(rows)


entropy_summary = conditional_perf_table(
    state_perf,
    "entropy_quartile",
)

entropy_summary.to_csv(
    OUTPUT_DIR / "hmm_vs_logistic_by_state_entropy_quartile.csv",
    index=False,
)

display(
    entropy_summary.style.format(
        {
            "HMM_ann_mean": "{:.2%}",
            "benchmark_ann_mean": "{:.2%}",
            "HMM_minus_benchmark_ann_mean": "{:+.2%}",
            "HMM_sharpe": "{:.3f}",
            "benchmark_sharpe": "{:.3f}",
            "delta_sharpe": "{:+.3f}",
        }
    )
)


def hac_mechanism_regression(
    df: pd.DataFrame,
    y_col: str,
    x_cols,
    label: str,
):
    d = df[
        [y_col] + list(x_cols)
    ].dropna().copy()

    y = d[y_col].astype(float)

    X = sm.add_constant(
        d[list(x_cols)].astype(float),
        has_constant="add",
    )

    fit = sm.OLS(
        y,
        X,
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": HAC_LAGS},
    )

    rows = []

    for x in x_cols:
        rows.append(
            {
                "regression": label,
                "variable": x,
                "n": int(fit.nobs),
                "beta_weekly": float(fit.params[x]),
                "beta_annualized_scale": float(
                    fit.params[x] * PERIODS_PER_YEAR
                ),
                "hac_t": float(fit.tvalues[x]),
                "p_value_two_sided": float(fit.pvalues[x]),
                "r_squared": float(fit.rsquared),
            }
        )

    return rows, fit


mechanism_rows = []
mechanism_models = {}

for x_cols, label in [
    (
        ["state_entropy"],
        "HMM-Logistic ~ entropy",
    ),
    (
        ["max_state_prob"],
        "HMM-Logistic ~ max state probability",
    ),
    (
        ["state_transition"],
        "HMM-Logistic ~ state transition",
    ),
    (
        ["state_entropy", "state_transition"],
        "HMM-Logistic ~ entropy + transition",
    ),
]:
    rows, fit = hac_mechanism_regression(
        state_perf,
        "HMM_minus_Logistic",
        x_cols,
        label,
    )

    mechanism_rows.extend(rows)
    mechanism_models[label] = fit

mechanism_results = pd.DataFrame(
    mechanism_rows
)

mechanism_results.to_csv(
    OUTPUT_DIR / "hmm_state_mechanism_hac_regressions.csv",
    index=False,
)

display(
    mechanism_results.style.format(
        {
            "beta_weekly": "{:+.4%}",
            "beta_annualized_scale": "{:+.2%}",
            "hac_t": "{:.3f}",
            "p_value_two_sided": "{:.4g}",
            "r_squared": "{:.4f}",
        }
    )
)


fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    entropy_summary["entropy_quartile"].astype(str),
    entropy_summary["HMM_minus_benchmark_ann_mean"],
)

ax.axhline(0.0, linewidth=1)
ax.set_title("HMM Minus Logistic: Mean Return by State-Entropy Quartile")
ax.set_xlabel("State entropy")
ax.set_ylabel("Annualized mean return difference")
ax.tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()
plt.close(fig)


state_perf.to_csv(
    OUTPUT_DIR / "hmm_state_weekly_performance_panel.csv",
    index=False,
)


## 16. Exact screened tail overlap and HMM-only trades

The screen and decile definitions are delegated to the actual portfolio engine. This avoids approximating the implementable universe or tail membership.


In [ ]:
# 18. Exact screened deciles, overlap, and HMM-only trade returns.
#
# Screening is delegated to the project's DataAssembler for implementation consistency.
# It therefore uses the same:
#   - MarketCap hygiene
#   - Price floor
#   - DollarVol_20d floor
#   - missing-ADV rejection
# as PortfolioManager.

import re as _re
import inspect

from Scripts.Portfolio.data_assembler import DataAssembler
from Scripts.Portfolio.portfolio_calculations import PortfolioMath


def _safe_prefix(name):
    return _re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        str(name),
    ).strip("_")


# ------------------------------------------------------------
# A. Reconstruct the exact screened stock-date panel through DataAssembler
# ------------------------------------------------------------

# The screen is independent of the model score once all signals share
# identical strict common stock-date keys. HMM is used as a neutral
# seed signal so DataAssembler performs the exact merge and screening steps.
screen_seed_signal = (
    common[
        ["Date", "StockID", "p_hmm"]
    ]
    .rename(columns={"p_hmm": "up_prob"})
    .copy()
)

# Read PortfolioManager defaults directly to avoid duplicating configuration.
_pm_signature = inspect.signature(PortfolioManager.__init__)

PM_MW_WINDOW = int(
    _pm_signature.parameters["mw_window"].default
)
PM_MW_MIN_PERIODS = int(
    _pm_signature.parameters["mw_min_periods"].default
)

screen_assembler = DataAssembler(
    freq=FREQ,
    start_year=ANALYSIS_START_YEAR,
    end_year=ANALYSIS_END_YEAR,
    country=COUNTRY,
    delay_list=[DELAY],
    tradability_screens=True,
    min_price=float(
        SCREENED_KWARGS.get(
            "min_price",
            1.0,
        )
    ),
    min_adv_dollar=(
        None
        if SCREENED_KWARGS.get(
            "min_adv_dollar",
            None,
        ) is None
        else float(
            SCREENED_KWARGS["min_adv_dollar"]
        )
    ),
    mw_window=PM_MW_WINDOW,
    mw_min_periods=PM_MW_MIN_PERIODS,
    include_price_adv=True,
    verbose=False,
)

prepared_screen = screen_assembler.prepare(
    screen_seed_signal
).reset_index()

screen_ret_col = screen_assembler.no_delay_ret_name

required_screen_cols = [
    "Date",
    "StockID",
    screen_ret_col,
]

missing_required = [
    c
    for c in required_screen_cols
    if c not in prepared_screen.columns
]

if missing_required:
    raise KeyError(
        "DataAssembler output is missing required columns: "
        f"{missing_required}. "
        f"Available={list(prepared_screen.columns)}"
    )

prepared_screen["Date"] = pd.to_datetime(
    prepared_screen["Date"],
    errors="coerce",
).dt.normalize()

prepared_screen["StockID"] = (
    pd.to_numeric(
        prepared_screen["StockID"],
        errors="coerce",
    )
    .astype("Int64")
    .astype(str)
)

prepared_screen[screen_ret_col] = pd.to_numeric(
    prepared_screen[screen_ret_col],
    errors="coerce",
)

screen_cols_to_keep = [
    "Date",
    "StockID",
    screen_ret_col,
]

for optional_col in [
    "MarketCap",
    "Price",
    "DollarVol_20d",
]:
    if optional_col in prepared_screen.columns:
        screen_cols_to_keep.append(
            optional_col
        )

screen_keys = (
    prepared_screen[
        screen_cols_to_keep
    ]
    .dropna(
        subset=[
            "Date",
            "StockID",
            screen_ret_col,
        ]
    )
    .drop_duplicates(
        ["Date", "StockID"],
        keep="last",
    )
    .rename(
        columns={
            screen_ret_col:
            "fwd_ret_overlap"
        }
    )
)

# Add the walk-forward logistic score.
logistic_for_overlap = (
    logistic_signal[
        ["Date", "StockID", "up_prob"]
    ]
    .rename(
        columns={
            "up_prob": "p_logistic"
        }
    )
    .copy()
)

logistic_for_overlap["Date"] = pd.to_datetime(
    logistic_for_overlap["Date"],
    errors="coerce",
).dt.normalize()

logistic_for_overlap["StockID"] = (
    logistic_for_overlap["StockID"]
    .astype(str)
)

screened_stock_panel = (
    common[
        [
            "Date",
            "StockID",
            "p_cnn",
            "p_rf",
            "rank_blend_5050",
            "p_hmm",
        ]
    ]
    .merge(
        logistic_for_overlap,
        on=["Date", "StockID"],
        how="inner",
    )
    .merge(
        screen_keys,
        on=["Date", "StockID"],
        how="inner",
    )
    .dropna(
        subset=[
            "p_cnn",
            "p_rf",
            "rank_blend_5050",
            "p_hmm",
            "p_logistic",
            "fwd_ret_overlap",
        ]
    )
    .sort_values(
        ["Date", "StockID"]
    )
    .reset_index(drop=True)
)

print(
    "Exact DataAssembler-screened panel:",
    f"{len(screened_stock_panel):,}",
    "rows |",
    f"{screened_stock_panel['Date'].nunique():,}",
    "dates |",
    f"{screened_stock_panel['StockID'].nunique():,}",
    "stocks",
)

print(
    "Screen return column:",
    screen_ret_col,
)

print(
    "Screen thresholds:",
    "min_price=",
    SCREENED_KWARGS.get("min_price"),
    "| min_adv_dollar=",
    SCREENED_KWARGS.get("min_adv_dollar"),
)


# ------------------------------------------------------------
# B. Use the exact PortfolioMath percentile-tail definitions
# ------------------------------------------------------------

score_map = {
    "CNN": "p_cnn",
    "RF": "p_rf",
    "RankBlend_50_50": "rank_blend_5050",
    "LogisticStack_WF": "p_logistic",
    "HMM": "p_hmm",
}


def add_exact_portfolio_tail_masks(
    df: pd.DataFrame,
    score_col: str,
    prefix: str,
    cut: int = CUT,
):
    low_mask = pd.Series(
        False,
        index=df.index,
        dtype=bool,
    )

    high_mask = pd.Series(
        False,
        index=df.index,
        dtype=bool,
    )

    rank_pct = pd.Series(
        np.nan,
        index=df.index,
        dtype=float,
    )

    for _, idx in df.groupby(
        "Date",
        sort=True,
    ).groups.items():

        idx = pd.Index(idx)

        up = pd.to_numeric(
            df.loc[idx, score_col],
            errors="coerce",
        )

        finite = (
            up.notna()
            & np.isfinite(
                up.to_numpy(float)
            )
        )

        valid_idx = idx[
            finite.to_numpy()
        ]

        if len(valid_idx) == 0:
            continue

        up_valid = up.loc[
            valid_idx
        ].astype(float)

        # Diagnostic percentile rank only.
        rank_pct.loc[valid_idx] = (
            up_valid.rank(
                method="average",
                pct=True,
            )
        )

        low_lo, low_hi = (
            PortfolioMath._decile_bounds(
                up_valid,
                int(cut),
                0,
            )
        )

        high_lo, high_hi = (
            PortfolioMath._decile_bounds(
                up_valid,
                int(cut),
                int(cut) - 1,
            )
        )

        low_mask.loc[valid_idx] = (
            PortfolioMath._decile_mask(
                up_valid,
                low_lo,
                low_hi,
                is_lowest=True,
            )
            .astype(bool)
        )

        high_mask.loc[valid_idx] = (
            PortfolioMath._decile_mask(
                up_valid,
                high_lo,
                high_hi,
                is_lowest=False,
            )
            .astype(bool)
        )

    df[f"{prefix}_rank_pct"] = rank_pct
    df[f"{prefix}_low"] = low_mask
    df[f"{prefix}_high"] = high_mask


for model, score_col in score_map.items():
    add_exact_portfolio_tail_masks(
        screened_stock_panel,
        score_col,
        _safe_prefix(model),
    )


# ------------------------------------------------------------
# C. Reproduction check against PortfolioManager H-L
# ------------------------------------------------------------

reconstruction_rows = []
reconstructed_hl = {}

for model in score_map:
    prefix = _safe_prefix(model)

    high_ret = (
        screened_stock_panel.loc[
            screened_stock_panel[
                f"{prefix}_high"
            ]
        ]
        .groupby("Date")[
            "fwd_ret_overlap"
        ]
        .mean()
    )

    low_ret = (
        screened_stock_panel.loc[
            screened_stock_panel[
                f"{prefix}_low"
            ]
        ]
        .groupby("Date")[
            "fwd_ret_overlap"
        ]
        .mean()
    )

    rec = (
        high_ret - low_ret
    ).rename("reconstructed")

    reconstructed_hl[model] = rec

    pair = pd.concat(
        [
            hl[model].rename(
                "portfolio_manager"
            ),
            rec,
        ],
        axis=1,
    ).dropna()

    reconstruction_rows.append(
        {
            "model": model,
            "n_dates": len(pair),
            "correlation": float(
                pair[
                    "portfolio_manager"
                ].corr(
                    pair[
                        "reconstructed"
                    ]
                )
            ),
            "mean_abs_difference": float(
                (
                    pair[
                        "portfolio_manager"
                    ]
                    - pair[
                        "reconstructed"
                    ]
                )
                .abs()
                .mean()
            ),
            "portfolio_manager_sharpe":
            annualized_sharpe(
                pair[
                    "portfolio_manager"
                ].to_numpy(float)
            ),
            "reconstructed_sharpe":
            annualized_sharpe(
                pair[
                    "reconstructed"
                ].to_numpy(float)
            ),
        }
    )


reconstruction_quality = pd.DataFrame(
    reconstruction_rows
)

reconstruction_quality.to_csv(
    OUTPUT_DIR
    / "overlap_screen_reconstruction_quality.csv",
    index=False,
)

display(
    reconstruction_quality.style.format(
        {
            "correlation": "{:.6f}",
            "mean_abs_difference": "{:.6%}",
            "portfolio_manager_sharpe": "{:.3f}",
            "reconstructed_sharpe": "{:.3f}",
        }
    )
)

min_reconstruction_corr = (
    reconstruction_quality[
        "correlation"
    ]
    .dropna()
    .min()
)

if min_reconstruction_corr < 0.995:
    print(
        "\nWARNING: at least one reconstructed "
        "H-L series has correlation < 0.995 "
        "with PortfolioManager. Do not interpret "
        "the HMM-only trade decomposition until "
        "the remaining discrepancy is reconciled."
    )
else:
    print(
        "\nReconstruction check passed: "
        f"minimum correlation={min_reconstruction_corr:.6f}"
    )


# ------------------------------------------------------------
# D. Overlap and unique-trade return analysis
# ------------------------------------------------------------

def overlap_and_unique_trade_analysis(
    df: pd.DataFrame,
    benchmark: str,
):
    hmm_prefix = _safe_prefix("HMM")
    bm_prefix = _safe_prefix(
        benchmark
    )

    out = df[
        [
            "Date",
            "StockID",
            "fwd_ret_overlap",
            f"{hmm_prefix}_high",
            f"{hmm_prefix}_low",
            f"{bm_prefix}_high",
            f"{bm_prefix}_low",
        ]
    ].copy()

    # Long-side membership.
    out["long_shared"] = (
        out[f"{hmm_prefix}_high"]
        & out[f"{bm_prefix}_high"]
    )

    out["long_hmm_only"] = (
        out[f"{hmm_prefix}_high"]
        & ~out[f"{bm_prefix}_high"]
    )

    out["long_benchmark_only"] = (
        out[f"{bm_prefix}_high"]
        & ~out[f"{hmm_prefix}_high"]
    )

    # Short-side membership.
    out["short_shared"] = (
        out[f"{hmm_prefix}_low"]
        & out[f"{bm_prefix}_low"]
    )

    out["short_hmm_only"] = (
        out[f"{hmm_prefix}_low"]
        & ~out[f"{bm_prefix}_low"]
    )

    out["short_benchmark_only"] = (
        out[f"{bm_prefix}_low"]
        & ~out[f"{hmm_prefix}_low"]
    )

    def _weekly_count(mask_col):
        return (
            out.loc[
                out[mask_col]
            ]
            .groupby("Date")[
                "StockID"
            ]
            .nunique()
        )

    hmm_high_n = _weekly_count(
        f"{hmm_prefix}_high"
    )

    bm_high_n = _weekly_count(
        f"{bm_prefix}_high"
    )

    shared_high_n = _weekly_count(
        "long_shared"
    )

    hmm_low_n = _weekly_count(
        f"{hmm_prefix}_low"
    )

    bm_low_n = _weekly_count(
        f"{bm_prefix}_low"
    )

    shared_low_n = _weekly_count(
        "short_shared"
    )

    dates = sorted(
        set(hmm_high_n.index)
        | set(bm_high_n.index)
        | set(hmm_low_n.index)
        | set(bm_low_n.index)
    )

    overlap = pd.DataFrame(
        index=pd.DatetimeIndex(
            dates
        )
    )

    overlap["hmm_high_n"] = hmm_high_n
    overlap["benchmark_high_n"] = bm_high_n
    overlap["shared_high_n"] = shared_high_n

    overlap["hmm_low_n"] = hmm_low_n
    overlap["benchmark_low_n"] = bm_low_n
    overlap["shared_low_n"] = shared_low_n

    overlap[
        "high_hmm_share_overlapping"
    ] = (
        overlap[
            "shared_high_n"
        ]
        / overlap[
            "hmm_high_n"
        ]
    )

    overlap[
        "low_hmm_share_overlapping"
    ] = (
        overlap[
            "shared_low_n"
        ]
        / overlap[
            "hmm_low_n"
        ]
    )

    overlap[
        "high_jaccard"
    ] = (
        overlap[
            "shared_high_n"
        ]
        / (
            overlap[
                "hmm_high_n"
            ]
            + overlap[
                "benchmark_high_n"
            ]
            - overlap[
                "shared_high_n"
            ]
        )
    )

    overlap[
        "low_jaccard"
    ] = (
        overlap[
            "shared_low_n"
        ]
        / (
            overlap[
                "hmm_low_n"
            ]
            + overlap[
                "benchmark_low_n"
            ]
            - overlap[
                "shared_low_n"
            ]
        )
    )

    # Weekly stock-group returns.
    weekly_returns = pd.DataFrame(
        index=overlap.index
    )

    for group in [
        "long_shared",
        "long_hmm_only",
        "long_benchmark_only",
        "short_shared",
        "short_hmm_only",
        "short_benchmark_only",
    ]:

        raw = (
            out.loc[
                out[group]
            ]
            .groupby("Date")[
                "fwd_ret_overlap"
            ]
            .mean()
        )

        # Report short-side returns from the actual short position perspective.
        if group.startswith(
            "short_"
        ):
            raw = -raw

        weekly_returns[
            group
        ] = raw

    overlap[
        "benchmark"
    ] = benchmark

    weekly_returns[
        "benchmark"
    ] = benchmark

    return (
        overlap,
        weekly_returns,
    )


overlap_results = {}
unique_weekly_results = {}
unique_summary_rows = []

for benchmark in [
    "RF",
    "LogisticStack_WF",
]:

    (
        overlap_df,
        weekly_unique,
    ) = overlap_and_unique_trade_analysis(
        screened_stock_panel,
        benchmark,
    )

    overlap_results[
        benchmark
    ] = overlap_df

    unique_weekly_results[
        benchmark
    ] = weekly_unique

    overlap_df.to_csv(
        OUTPUT_DIR
        / (
            f"hmm_vs_"
            f"{_safe_prefix(benchmark)}"
            f"_tail_overlap.csv"
        )
    )

    weekly_unique.to_csv(
        OUTPUT_DIR
        / (
            f"hmm_vs_"
            f"{_safe_prefix(benchmark)}"
            f"_unique_trade_returns.csv"
        )
    )

    # Compare HMM-only vs benchmark-only returns on each side.
    for side in [
        "long",
        "short",
    ]:

        a_col = (
            f"{side}_hmm_only"
        )

        b_col = (
            f"{side}_benchmark_only"
        )

        pair = weekly_unique[
            [a_col, b_col]
        ].dropna()

        if len(pair) < 5:
            continue

        test = paired_hac_test(
            pair[a_col],
            pair[b_col],
            f"HMM-only {side}",
            f"{benchmark}-only {side}",
        )

        unique_summary_rows.append(
            {
                "benchmark":
                benchmark,
                "side":
                side,
                "n_weeks":
                len(pair),
                "HMM_only_ann_mean":
                float(
                    pair[a_col].mean()
                    * PERIODS_PER_YEAR
                ),
                "benchmark_only_ann_mean":
                float(
                    pair[b_col].mean()
                    * PERIODS_PER_YEAR
                ),
                "HMM_minus_benchmark_unique_ann_mean":
                float(
                    (
                        pair[a_col]
                        - pair[b_col]
                    ).mean()
                    * PERIODS_PER_YEAR
                ),
                "hac_t":
                test["hac_t"],
                "p_value_two_sided":
                test[
                    "p_value_two_sided"
                ],
            }
        )


unique_trade_summary = pd.DataFrame(
    unique_summary_rows
)

unique_trade_summary.to_csv(
    OUTPUT_DIR
    / "hmm_unique_trade_return_summary.csv",
    index=False,
)

display(
    unique_trade_summary.style.format(
        {
            "HMM_only_ann_mean":
            "{:+.2%}",
            "benchmark_only_ann_mean":
            "{:+.2%}",
            "HMM_minus_benchmark_unique_ann_mean":
            "{:+.2%}",
            "hac_t":
            "{:.3f}",
            "p_value_two_sided":
            "{:.4g}",
        }
    )
)


overlap_summary = []

for benchmark, d in overlap_results.items():

    overlap_summary.append(
        {
            "benchmark":
            benchmark,
            "mean_high_HMM_share_overlapping":
            float(
                d[
                    "high_hmm_share_overlapping"
                ].mean()
            ),
            "mean_low_HMM_share_overlapping":
            float(
                d[
                    "low_hmm_share_overlapping"
                ].mean()
            ),
            "mean_high_jaccard":
            float(
                d[
                    "high_jaccard"
                ].mean()
            ),
            "mean_low_jaccard":
            float(
                d[
                    "low_jaccard"
                ].mean()
            ),
        }
    )

overlap_summary = pd.DataFrame(
    overlap_summary
)

overlap_summary.to_csv(
    OUTPUT_DIR
    / "hmm_tail_overlap_summary.csv",
    index=False,
)

display(
    overlap_summary.style.format(
        {
            "mean_high_HMM_share_overlapping":
            "{:.1%}",
            "mean_low_HMM_share_overlapping":
            "{:.1%}",
            "mean_high_jaccard":
            "{:.1%}",
            "mean_low_jaccard":
            "{:.1%}",
        }
    )
)


fig, ax = plt.subplots(
    figsize=(14, 5)
)

for benchmark, d in overlap_results.items():

    ax.plot(
        d.index,
        d[
            "high_hmm_share_overlapping"
        ].rolling(
            12,
            min_periods=6,
        ).mean(),
        label=(
            f"HMM high vs "
            f"{benchmark}"
        ),
    )

ax.set_title(
    "HMM Long-Tail Overlap — "
    "12-Week Rolling Mean"
)

ax.set_xlabel("Date")

ax.set_ylabel(
    "Share of HMM long tail "
    "also selected"
)

ax.set_ylim(
    0.0,
    1.0,
)

ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)


fig, ax = plt.subplots(
    figsize=(14, 5)
)

for benchmark, d in overlap_results.items():

    ax.plot(
        d.index,
        d[
            "low_hmm_share_overlapping"
        ].rolling(
            12,
            min_periods=6,
        ).mean(),
        label=(
            f"HMM low vs "
            f"{benchmark}"
        ),
    )

ax.set_title(
    "HMM Short-Tail Overlap — "
    "12-Week Rolling Mean"
)

ax.set_xlabel("Date")

ax.set_ylabel(
    "Share of HMM short tail "
    "also selected"
)

ax.set_ylim(
    0.0,
    1.0,
)

ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)


## 17. Signal persistence, tail retention, and target turnover


In [ ]:
# 19. Signal persistence, tail retention, turnover, and holding spells.

from scipy.stats import spearmanr


def rank_persistence_series(
    df: pd.DataFrame,
    rank_col: str,
) -> pd.Series:
    x = df[
        ["Date", "StockID", rank_col]
    ].copy()

    x = x.sort_values(
        ["StockID", "Date"]
    )

    x["prev_date_for_stock"] = (
        x.groupby("StockID")["Date"]
        .shift(1)
    )

    x["prev_rank"] = (
        x.groupby("StockID")[rank_col]
        .shift(1)
    )

    unique_dates = sorted(
        x["Date"].dropna().unique()
    )

    prev_date_map = {
        pd.Timestamp(unique_dates[i]):
        pd.Timestamp(unique_dates[i - 1])
        for i in range(1, len(unique_dates))
    }

    expected_prev = x["Date"].map(
        prev_date_map
    )

    valid = (
        x["prev_date_for_stock"]
        == expected_prev
    )

    x = x[
        valid
        & x["prev_rank"].notna()
        & x[rank_col].notna()
    ].copy()

    def _corr(g):
        if len(g) < 10:
            return np.nan

        return float(
            g[[rank_col, "prev_rank"]]
            .corr(
                method="spearman"
            )
            .iloc[0, 1]
        )

    return (
        x.groupby("Date")
        .apply(_corr)
        .rename("rank_persistence")
    )


def set_turnover_and_retention(
    df: pd.DataFrame,
    mask_col: str,
):
    selected = (
        df.loc[
            df[mask_col],
            ["Date", "StockID"],
        ]
        .groupby("Date")["StockID"]
        .agg(set)
    )

    dates = sorted(
        df["Date"].dropna().unique()
    )

    rows = []

    for i in range(1, len(dates)):
        prev_dt = pd.Timestamp(dates[i - 1])
        cur_dt = pd.Timestamp(dates[i])

        prev_set = selected.get(
            prev_dt,
            set(),
        )

        cur_set = selected.get(
            cur_dt,
            set(),
        )

        n_prev = len(prev_set)
        n_cur = len(cur_set)
        inter = len(
            prev_set.intersection(
                cur_set
            )
        )

        retention = (
            inter / n_prev
            if n_prev > 0
            else np.nan
        )

        if n_prev > 0 and n_cur > 0:
            # 0.5 * L1 distance between two equal-weight sleeve targets.
            turnover = 0.5 * (
                inter * abs(
                    1.0 / n_cur
                    - 1.0 / n_prev
                )
                + (n_cur - inter) * (
                    1.0 / n_cur
                )
                + (n_prev - inter) * (
                    1.0 / n_prev
                )
            )
        else:
            turnover = np.nan

        rows.append(
            {
                "Date": cur_dt,
                "retention": retention,
                "turnover": turnover,
                "n_prev": n_prev,
                "n_cur": n_cur,
                "intersection": inter,
            }
        )

    return pd.DataFrame(rows).set_index(
        "Date"
    )


def holding_spell_lengths(
    df: pd.DataFrame,
    mask_col: str,
):
    selected = df.loc[
        df[mask_col],
        ["Date", "StockID"],
    ].copy()

    if selected.empty:
        return np.array([], dtype=float)

    unique_dates = sorted(
        df["Date"].dropna().unique()
    )

    date_pos = {
        pd.Timestamp(dt): i
        for i, dt in enumerate(unique_dates)
    }

    selected["date_pos"] = (
        selected["Date"]
        .map(date_pos)
        .astype(int)
    )

    selected = selected.sort_values(
        ["StockID", "date_pos"]
    )

    selected["new_spell"] = (
        selected.groupby("StockID")["date_pos"]
        .diff()
        .ne(1)
        .fillna(True)
    )

    selected["spell_id"] = (
        selected.groupby("StockID")["new_spell"]
        .cumsum()
    )

    lengths = (
        selected.groupby(
            ["StockID", "spell_id"]
        )
        .size()
        .to_numpy(float)
    )

    return lengths


persistence_rows = []
weekly_turnover_frames = []

for model, score_col in score_map.items():
    prefix = _safe_prefix(model)

    # Cross-sectional rank on the reconstructed screened universe.
    rank_col = f"{prefix}_analysis_rank"

    screened_stock_panel[rank_col] = (
        screened_stock_panel.groupby("Date")[score_col]
        .rank(
            method="average",
            pct=True,
        )
    )

    rank_persist = rank_persistence_series(
        screened_stock_panel,
        rank_col,
    )

    high_stats = set_turnover_and_retention(
        screened_stock_panel,
        f"{prefix}_high",
    )

    low_stats = set_turnover_and_retention(
        screened_stock_panel,
        f"{prefix}_low",
    )

    weekly = pd.concat(
        [
            rank_persist,
            high_stats[
                ["retention", "turnover"]
            ].rename(
                columns={
                    "retention": "high_retention",
                    "turnover": "high_turnover",
                }
            ),
            low_stats[
                ["retention", "turnover"]
            ].rename(
                columns={
                    "retention": "low_retention",
                    "turnover": "low_turnover",
                }
            ),
        ],
        axis=1,
    )

    weekly["total_target_turnover"] = (
        weekly["high_turnover"]
        + weekly["low_turnover"]
    )

    weekly["model"] = model
    weekly_turnover_frames.append(
        weekly.reset_index()
    )

    high_spells = holding_spell_lengths(
        screened_stock_panel,
        f"{prefix}_high",
    )

    low_spells = holding_spell_lengths(
        screened_stock_panel,
        f"{prefix}_low",
    )

    persistence_rows.append(
        {
            "model": model,
            "mean_rank_persistence": float(
                weekly["rank_persistence"].mean()
            ),
            "mean_high_retention": float(
                weekly["high_retention"].mean()
            ),
            "mean_low_retention": float(
                weekly["low_retention"].mean()
            ),
            "mean_high_turnover": float(
                weekly["high_turnover"].mean()
            ),
            "mean_low_turnover": float(
                weekly["low_turnover"].mean()
            ),
            "mean_total_target_turnover": float(
                weekly["total_target_turnover"].mean()
            ),
            "median_high_holding_weeks": float(
                np.median(high_spells)
                if len(high_spells)
                else np.nan
            ),
            "mean_high_holding_weeks": float(
                np.mean(high_spells)
                if len(high_spells)
                else np.nan
            ),
            "median_low_holding_weeks": float(
                np.median(low_spells)
                if len(low_spells)
                else np.nan
            ),
            "mean_low_holding_weeks": float(
                np.mean(low_spells)
                if len(low_spells)
                else np.nan
            ),
        }
    )


persistence_summary = pd.DataFrame(
    persistence_rows
)

weekly_persistence = pd.concat(
    weekly_turnover_frames,
    ignore_index=True,
)

persistence_summary.to_csv(
    OUTPUT_DIR / "signal_persistence_turnover_summary.csv",
    index=False,
)

weekly_persistence.to_csv(
    OUTPUT_DIR / "weekly_signal_persistence_turnover.csv",
    index=False,
)

display(
    persistence_summary.style.format(
        {
            "mean_rank_persistence": "{:.3f}",
            "mean_high_retention": "{:.1%}",
            "mean_low_retention": "{:.1%}",
            "mean_high_turnover": "{:.3f}",
            "mean_low_turnover": "{:.3f}",
            "mean_total_target_turnover": "{:.3f}",
            "median_high_holding_weeks": "{:.1f}",
            "mean_high_holding_weeks": "{:.2f}",
            "median_low_holding_weeks": "{:.1f}",
            "mean_low_holding_weeks": "{:.2f}",
        }
    )
)


turnover_pivot = weekly_persistence.pivot(
    index="Date",
    columns="model",
    values="total_target_turnover",
)

fig, ax = plt.subplots(figsize=(14, 5))

for model in [
    "RF",
    "RankBlend_50_50",
    "LogisticStack_WF",
    "HMM",
]:
    if model in turnover_pivot.columns:
        ax.plot(
            turnover_pivot.index,
            turnover_pivot[model].rolling(
                12,
                min_periods=6,
            ).mean(),
            label=model,
        )

ax.set_title(
    "Equal-Weight Target Turnover — 12-Week Rolling Mean"
)
ax.set_xlabel("Date")
ax.set_ylabel("Long + short target turnover")
ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)


## 18. Dynamic expert-mapping test

This asks whether larger HMM departures from the static CNN/RF mapping are associated with larger HMM-minus-logistic returns. It is a mechanism test, not a separate alpha model.


In [ ]:
# 20. Direct dynamic-mapping mechanism test.

# ------------------------------------------------------------
# A. Build one weekly panel
# ------------------------------------------------------------

mapping_panel = (
    hmm_weekly_attribution[
        [
            "Date",
            "cnn_influence_share",
            "rf_influence_share",
            "cross_sectional_r2",
        ]
    ]
    .rename(
        columns={
            "cnn_influence_share": "hmm_cnn_influence",
            "rf_influence_share": "hmm_rf_influence",
            "cross_sectional_r2": "hmm_attribution_r2",
        }
    )
    .merge(
        logistic_weekly_influence[
            [
                "Date",
                "cnn_influence_share",
                "rf_influence_share",
            ]
        ].rename(
            columns={
                "cnn_influence_share": "logistic_cnn_influence",
                "rf_influence_share": "logistic_rf_influence",
            }
        ),
        on="Date",
        how="inner",
    )
    .merge(
        hl[
            [
                "HMM",
                "LogisticStack_WF",
                "RF",
            ]
        ]
        .reset_index()
        .rename(
            columns={
                hl.index.name or "index": "Date",
            }
        ),
        on="Date",
        how="inner",
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

mapping_panel["delta_rf_influence"] = (
    mapping_panel["hmm_rf_influence"]
    - mapping_panel["logistic_rf_influence"]
)

mapping_panel["delta_cnn_influence"] = (
    mapping_panel["hmm_cnn_influence"]
    - mapping_panel["logistic_cnn_influence"]
)

mapping_panel["abs_mapping_deviation"] = (
    mapping_panel["delta_rf_influence"].abs()
)

mapping_panel["rf_more_than_logistic"] = (
    mapping_panel["delta_rf_influence"]
    .clip(lower=0.0)
)

mapping_panel["cnn_more_than_logistic"] = (
    (-mapping_panel["delta_rf_influence"])
    .clip(lower=0.0)
)

mapping_panel["HMM_minus_Logistic"] = (
    mapping_panel["HMM"]
    - mapping_panel["LogisticStack_WF"]
)

mapping_panel["HMM_minus_RF"] = (
    mapping_panel["HMM"]
    - mapping_panel["RF"]
)

# Numerical consistency check: influence shares should be mirror images.
mirror_error = (
    mapping_panel["delta_rf_influence"]
    + mapping_panel["delta_cnn_influence"]
).abs().max()

print(
    "Weekly mapping panel:",
    f"{len(mapping_panel):,}",
    "weeks |",
    mapping_panel["Date"].min(),
    "to",
    mapping_panel["Date"].max(),
)

print(
    "Maximum CNN/RF mirror-consistency error:",
    f"{mirror_error:.3e}",
)

display(
    mapping_panel[
        [
            "hmm_rf_influence",
            "logistic_rf_influence",
            "delta_rf_influence",
            "abs_mapping_deviation",
            "hmm_attribution_r2",
            "HMM_minus_Logistic",
        ]
    ].describe().T
)


# ------------------------------------------------------------
# B. Quartile analysis by magnitude of remapping
# ------------------------------------------------------------

_mapping_pct = mapping_panel["abs_mapping_deviation"].rank(
    method="average",
    pct=True,
)

mapping_panel["mapping_deviation_quartile"] = pd.cut(
    _mapping_pct,
    bins=[0.0, 0.25, 0.50, 0.75, 1.0],
    labels=[
        "Q1: smallest remapping",
        "Q2",
        "Q3",
        "Q4: largest remapping",
    ],
    include_lowest=True,
)


def mapping_quartile_summary(
    df: pd.DataFrame,
):
    rows = []

    for label, g in df.groupby(
        "mapping_deviation_quartile",
        observed=True,
    ):
        pair = g[
            [
                "HMM",
                "LogisticStack_WF",
                "HMM_minus_Logistic",
                "abs_mapping_deviation",
                "delta_rf_influence",
            ]
        ].dropna()

        if len(pair) < 5:
            continue

        rows.append(
            {
                "mapping_deviation_quartile": str(label),
                "n_weeks": len(pair),
                "mean_abs_mapping_deviation": float(
                    pair["abs_mapping_deviation"].mean()
                ),
                "mean_signed_rf_deviation": float(
                    pair["delta_rf_influence"].mean()
                ),
                "HMM_ann_mean": float(
                    pair["HMM"].mean()
                    * PERIODS_PER_YEAR
                ),
                "Logistic_ann_mean": float(
                    pair["LogisticStack_WF"].mean()
                    * PERIODS_PER_YEAR
                ),
                "HMM_minus_Logistic_ann_mean": float(
                    pair["HMM_minus_Logistic"].mean()
                    * PERIODS_PER_YEAR
                ),
                "HMM_sharpe": annualized_sharpe(
                    pair["HMM"].to_numpy(float)
                ),
                "Logistic_sharpe": annualized_sharpe(
                    pair["LogisticStack_WF"].to_numpy(float)
                ),
            }
        )

    out = pd.DataFrame(rows)

    out["delta_sharpe"] = (
        out["HMM_sharpe"]
        - out["Logistic_sharpe"]
    )

    return out


mapping_quartiles = mapping_quartile_summary(
    mapping_panel
)

mapping_quartiles.to_csv(
    OUTPUT_DIR / "dynamic_mapping_deviation_quartiles.csv",
    index=False,
)

print("\n=== MAPPING-DEVIATION QUARTILES ===")
display(
    mapping_quartiles.style.format(
        {
            "mean_abs_mapping_deviation": "{:.3f}",
            "mean_signed_rf_deviation": "{:+.3f}",
            "HMM_ann_mean": "{:.2%}",
            "Logistic_ann_mean": "{:.2%}",
            "HMM_minus_Logistic_ann_mean": "{:+.2%}",
            "HMM_sharpe": "{:.3f}",
            "Logistic_sharpe": "{:.3f}",
            "delta_sharpe": "{:+.3f}",
        }
    )
)


# Direct high-versus-low remapping comparison:
# Q4 HMM-minus-logistic weekly return minus Q1 HMM-minus-logistic weekly return.
q1_label = "Q1: smallest remapping"
q4_label = "Q4: largest remapping"

q1 = (
    mapping_panel.loc[
        mapping_panel["mapping_deviation_quartile"].astype(str)
        == q1_label,
        "HMM_minus_Logistic",
    ]
    .dropna()
)

q4 = (
    mapping_panel.loc[
        mapping_panel["mapping_deviation_quartile"].astype(str)
        == q4_label,
        "HMM_minus_Logistic",
    ]
    .dropna()
)

high_low_mapping_diff = {
    "Q1_ann_mean_HMM_minus_Logistic": (
        float(q1.mean() * PERIODS_PER_YEAR)
        if len(q1)
        else np.nan
    ),
    "Q4_ann_mean_HMM_minus_Logistic": (
        float(q4.mean() * PERIODS_PER_YEAR)
        if len(q4)
        else np.nan
    ),
    "Q4_minus_Q1_ann_mean": (
        float(
            (q4.mean() - q1.mean())
            * PERIODS_PER_YEAR
        )
        if len(q1) and len(q4)
        else np.nan
    ),
}

display(
    pd.DataFrame(
        [high_low_mapping_diff]
    ).style.format(
        "{:+.2%}"
    )
)


# ------------------------------------------------------------
# C. HAC regressions
# ------------------------------------------------------------

def hac_mapping_regression(
    df: pd.DataFrame,
    x_cols,
    label: str,
):
    d = df[
        ["HMM_minus_Logistic"]
        + list(x_cols)
    ].dropna().copy()

    y = d["HMM_minus_Logistic"].astype(float)

    X = sm.add_constant(
        d[list(x_cols)].astype(float),
        has_constant="add",
    )

    fit = sm.OLS(
        y,
        X,
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": HAC_LAGS,
        },
    )

    rows = []

    for x in x_cols:
        rows.append(
            {
                "regression": label,
                "variable": x,
                "n": int(fit.nobs),
                "beta_weekly": float(
                    fit.params[x]
                ),
                # This scale says: if x increased by a full 1.0,
                # what annualized change would the linear fit imply?
                # Effects are also reported per 10-percentage-point change for interpretability.
                "beta_annualized_full_scale": float(
                    fit.params[x]
                    * PERIODS_PER_YEAR
                ),
                "effect_per_10pct_mapping_change_ann": float(
                    fit.params[x]
                    * 0.10
                    * PERIODS_PER_YEAR
                ),
                "hac_t": float(
                    fit.tvalues[x]
                ),
                "p_value_two_sided": float(
                    fit.pvalues[x]
                ),
                "r_squared": float(
                    fit.rsquared
                ),
            }
        )

    return rows, fit


mapping_regression_rows = []
mapping_regression_models = {}

mapping_specs = [
    (
        ["abs_mapping_deviation"],
        "HMM-Logistic ~ |HMM RF influence - Logistic RF influence|",
    ),
    (
        ["delta_rf_influence"],
        "HMM-Logistic ~ signed RF influence deviation",
    ),
    (
        [
            "rf_more_than_logistic",
            "cnn_more_than_logistic",
        ],
        "HMM-Logistic ~ RF-more + CNN-more",
    ),
]

for x_cols, label in mapping_specs:
    rows, fit = hac_mapping_regression(
        mapping_panel,
        x_cols,
        label,
    )

    mapping_regression_rows.extend(
        rows
    )

    mapping_regression_models[
        label
    ] = fit


mapping_regression_results = pd.DataFrame(
    mapping_regression_rows
)

mapping_regression_results.to_csv(
    OUTPUT_DIR / "dynamic_mapping_hac_regressions.csv",
    index=False,
)

print("\n=== DYNAMIC-MAPPING HAC REGRESSIONS ===")
display(
    mapping_regression_results.style.format(
        {
            "beta_weekly": "{:+.5f}",
            "beta_annualized_full_scale": "{:+.2%}",
            "effect_per_10pct_mapping_change_ann": "{:+.2%}",
            "hac_t": "{:.3f}",
            "p_value_two_sided": "{:.4g}",
            "r_squared": "{:.4f}",
        }
    )
)


# ------------------------------------------------------------
# D. Does remapping explain the HMM's turnover/stability advantage?
# ------------------------------------------------------------

# weekly_persistence is produced by the immediately preceding section.
# Treat a missing object as an ordering error rather than silently skipping
# this mechanism diagnostic.
if "weekly_persistence" not in globals():
    raise RuntimeError(
        "weekly_persistence is unavailable. Run the signal-persistence section "
        "before the dynamic-mapping mechanism test."
    )

if "weekly_persistence" in globals():
    turnover_wide = (
        weekly_persistence[
            [
                "Date",
                "model",
                "total_target_turnover",
                "rank_persistence",
            ]
        ]
        .pivot(
            index="Date",
            columns="model",
            values=[
                "total_target_turnover",
                "rank_persistence",
            ],
        )
    )

    turnover_wide.columns = [
        f"{metric}_{model}"
        for metric, model
        in turnover_wide.columns
    ]

    turnover_wide = (
        turnover_wide
        .reset_index()
    )

    mapping_panel = mapping_panel.merge(
        turnover_wide,
        on="Date",
        how="left",
    )

    if {
        "total_target_turnover_HMM",
        "total_target_turnover_LogisticStack_WF",
    }.issubset(mapping_panel.columns):

        mapping_panel[
            "HMM_minus_Logistic_turnover"
        ] = (
            mapping_panel[
                "total_target_turnover_HMM"
            ]
            - mapping_panel[
                "total_target_turnover_LogisticStack_WF"
            ]
        )

        turnover_corr = mapping_panel[
            [
                "abs_mapping_deviation",
                "HMM_minus_Logistic_turnover",
            ]
        ].corr().iloc[0, 1]

        print(
            "\nCorrelation between remapping magnitude "
            "and HMM-minus-logistic target turnover:",
            f"{turnover_corr:.3f}",
        )


# ------------------------------------------------------------
# E. Visual diagnostics
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(14, 5)
)

ax.plot(
    mapping_panel["Date"],
    mapping_panel[
        "hmm_rf_influence"
    ],
    label="HMM RF influence",
)

ax.plot(
    mapping_panel["Date"],
    mapping_panel[
        "logistic_rf_influence"
    ],
    label="Logistic RF influence",
)

ax.set_title(
    "Weekly Effective RF Influence: HMM vs Logistic Stack"
)
ax.set_xlabel("Date")
ax.set_ylabel("RF influence share")
ax.set_ylim(0.0, 1.0)
ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)


fig, ax = plt.subplots(
    figsize=(14, 4)
)

ax.plot(
    mapping_panel["Date"],
    mapping_panel[
        "delta_rf_influence"
    ],
)

ax.axhline(
    0.0,
    linewidth=1,
)

ax.set_title(
    "HMM Minus Logistic: Weekly RF-Influence Deviation"
)
ax.set_xlabel("Date")
ax.set_ylabel(
    "HMM RF influence − Logistic RF influence"
)

fig.tight_layout()
plt.show()
plt.close(fig)


fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.scatter(
    mapping_panel[
        "abs_mapping_deviation"
    ],
    mapping_panel[
        "HMM_minus_Logistic"
    ],
    alpha=0.25,
)

# Add fitted HAC-regression line using OLS point estimates.
fit_mag = mapping_regression_models[
    "HMM-Logistic ~ |HMM RF influence - Logistic RF influence|"
]

x_grid = np.linspace(
    mapping_panel[
        "abs_mapping_deviation"
    ].min(),
    mapping_panel[
        "abs_mapping_deviation"
    ].max(),
    100,
)

y_grid = (
    fit_mag.params["const"]
    + fit_mag.params[
        "abs_mapping_deviation"
    ]
    * x_grid
)

ax.plot(
    x_grid,
    y_grid,
    linewidth=2,
)

ax.axhline(
    0.0,
    linewidth=1,
)

ax.set_title(
    "Does Larger HMM Remapping Coincide With Higher Incremental Return?"
)
ax.set_xlabel(
    "Absolute HMM-vs-logistic RF influence deviation"
)
ax.set_ylabel(
    "Weekly HMM − Logistic H-L return"
)

fig.tight_layout()
plt.show()
plt.close(fig)


fig, ax = plt.subplots(
    figsize=(10, 5)
)

ax.bar(
    mapping_quartiles[
        "mapping_deviation_quartile"
    ],
    mapping_quartiles[
        "HMM_minus_Logistic_ann_mean"
    ],
)

ax.axhline(
    0.0,
    linewidth=1,
)

ax.set_title(
    "HMM Minus Logistic Return by Mapping-Deviation Quartile"
)
ax.set_xlabel(
    "Magnitude of HMM remapping"
)
ax.set_ylabel(
    "Annualized mean return difference"
)
ax.tick_params(
    axis="x",
    rotation=20,
)

fig.tight_layout()
plt.show()
plt.close(fig)


mapping_panel.to_csv(
    OUTPUT_DIR / "dynamic_mapping_weekly_panel.csv",
    index=False,
)

print(
    "\nSaved dynamic-mapping outputs to:",
    OUTPUT_DIR,
)


## 18A. Memory cleanup before portfolio implementation runs

Large research-only panels are released before the repeated optimizer runs. Saved result tables remain available for the final dashboard.


In [ ]:
_large_objects_to_release = [
    "rf",
    "cnn",
    "hmm",
    "common",
    "signals",
    "logistic_signal",
    "expected_return_fusion",
    "screened_stock_panel",
    "screen_assembler",
    "prepared_screen",
    "screen_keys",
    "screen_seed_signal",
    "logistic_for_overlap",
    "state_perf",
    "mapping_panel",
    "weekly_persistence",
    "overlap_results",
    "unique_weekly_results",
    "hmm_weekly_attribution",
    "hmm_state_weekly",
    "logistic_weekly_influence",
]

released = []

for name in _large_objects_to_release:
    if name in globals():
        globals().pop(name, None)
        released.append(name)

gc.collect()

print(
    "Released research-only objects before implementation runs:",
    ", ".join(released) if released else "none",
)


# Part III — Common-optimizer gross portfolio robustness

The same Markowitz construction is applied to every alpha score. This separates the signal comparison from differences in portfolio methodology.


## 19. Gross H+LowNeg Markowitz comparison


In [ ]:
GROSS_MW_CACHE = OUTPUT_DIR / "same_optimizer_gross_markowitz_returns.csv"

if RUN_GROSS_MARKOWITZ:
    gross_mw_runs = {}
    gross_mw_turnover = {}

    for name, sig in optimizer_signals.items():
        print("Gross Markowitz:", name)
        r, to = calculate_gross_hl(
            sig,
            name=name,
            weight_type=GROSS_MARKOWITZ_WEIGHT_TYPE,
        )
        gross_mw_runs[name] = r
        gross_mw_turnover[name] = to

    gross_markowitz_panel = pd.concat(gross_mw_runs, axis=1)
    if gross_markowitz_panel.isna().any().any():
        raise RuntimeError("Gross Markowitz return streams are not exactly aligned.")
    gross_markowitz_panel.to_csv(GROSS_MW_CACHE)
else:
    if not GROSS_MW_CACHE.exists():
        raise FileNotFoundError(
            "RUN_GROSS_MARKOWITZ=False but no saved gross Markowitz panel exists."
        )
    gross_markowitz_panel = pd.read_csv(GROSS_MW_CACHE, index_col=0, parse_dates=True)
    gross_mw_turnover = {}

gross_mw_perf = pd.DataFrame(
    {
        name: sv.performance_stats(gross_markowitz_panel[name], periods_per_year=52)
        for name in gross_markowitz_panel.columns
    }
).T

if gross_mw_turnover:
    gross_mw_perf["avg_turnover"] = pd.Series(gross_mw_turnover)

gross_mw_perf.to_csv(OUTPUT_DIR / "same_optimizer_gross_markowitz_performance.csv")

display(
    gross_mw_perf.style.format(
        {
            "ann_mean": "{:.2%}",
            "ann_vol": "{:.2%}",
            "sharpe": "{:.3f}",
            "avg_turnover": "{:.3f}",
        }
    )
)

fig, ax = plt.subplots(figsize=(10, 5))
ordered = gross_mw_perf["sharpe"].sort_values()
ax.barh(ordered.index, ordered.values)
ax.axvline(0.0, linewidth=1)
ax.set_title("Common-Optimizer Gross Markowitz Sharpe")
ax.set_xlabel("Annualized Sharpe")
fig.tight_layout()
plt.show()
plt.close(fig)


## 20. Gross-regression preservation check

This is a refactor guard, not a performance test. If a compatible pre-refactor gross return file is available, the final utility compares the saved stream with the research stream and fails if the supposedly unchanged gross calculation moved beyond tolerance.

Cost-aware executed-gross results are **not** expected to be identical to the old implementation because corrected pre-trade state and liquidity constraints can legitimately change the executed portfolio.


In [ ]:
from Scripts.Portfolio.gross_regression import compare_gross_return_frames


reference_candidates = [
    (
        ANNUAL_ROOT
        / "analysis_outputs"
        / "fusion_ablation_validation_v6_quant_research"
        / "screened_ew_hl_returns.csv"
    ),
    (
        ANNUAL_ROOT
        / "analysis_outputs"
        / "fusion_ablation_validation_v6_quant_research"
        / "same_optimizer_gross_markowitz_returns.csv"
    ),
]

reference_existing = [
    p for p in reference_candidates
    if p.exists()
]

if reference_existing:
    print("Available pre-refactor gross reference files:")

    for p in reference_existing:
        print(" ", p)

    # Prefer the same-optimizer reference when available.
    reference = next(
        (
            p for p in reference_existing
            if "same_optimizer" in p.name
        ),
        reference_existing[0],
    )

    old_gross = pd.read_csv(
        reference,
        index_col=0,
        parse_dates=True,
    )

    if "same_optimizer" in reference.name:
        current = gross_markowitz_panel.copy()
    else:
        current = hl[
            old_gross.columns.intersection(hl.columns)
        ].copy()

    common_cols = [
        c for c in old_gross.columns
        if c in current.columns
    ]

    if not common_cols:
        raise RuntimeError(
            "Gross reference exists but shares no "
            "model columns with research."
        )

    gross_regression = compare_gross_return_frames(
        old_gross[common_cols],
        current[common_cols],
        freq=FREQ,
        require_same_dates=True,   # correct argument name
        atol=1e-12,
        rtol=1e-10,
    )

    display(gross_regression)

    gross_regression.to_csv(
        OUTPUT_DIR / "gross_regression_check.csv",
        index=False,
    )

else:
    print(
        "No compatible pre-refactor gross reference file was found. "
        "The research gross streams are still saved for future "
        "regression checks."
    )


# Part IV — Corrected transaction costs and capacity

The remaining results supersede the old v9/v10 transaction-cost and capacity numbers.

### Corrected implementation convention

For each weekly rebalance:

1. previous post-trade holdings drift through realized returns;
2. the current pre-trade portfolio is built on the union of legacy and desired names;
3. the optimizer trades from that actual pre-trade state;
4. the authoritative trade vector is `Δw = w_post - w_pre`;
5. daily ADV and daily volatility forecasts govern impact and ADV constraints;
6. execution occurs over one trading day;
7. spread, impact, turnover, dollar trades, and diagnostics all use the same `Δw`;
8. partial exits remain in the state and are carried to the next weekly rebalance;
9. a screened stock-date without frozen daily ADV/volatility forecasts is ineligible for a **new** cost-aware position, while the broader execution-support panel still retains legacy holdings; no historical ADV/volatility substitute is introduced.


## 21. Transaction-cost input preflight


In [ ]:
from Scripts.Portfolio.transaction_cost_inputs import load_frozen_cost_forecasts, standardize_frozen_cost_forecasts

if not TC_FORECAST_PATH.exists():
    raise FileNotFoundError(
        "Frozen transaction-cost forecast artifact not found:\n"
        f"{TC_FORECAST_PATH}"
    )

tc_raw = load_frozen_cost_forecasts(str(TC_FORECAST_PATH))
tc_std = standardize_frozen_cost_forecasts(tc_raw)

print("Frozen TC artifact:", TC_FORECAST_PATH)
print("Rows:", f"{len(tc_std):,}")
print("Dates:", tc_std.index.get_level_values("Date").min(), "to", tc_std.index.get_level_values("Date").max())
display(tc_std.describe().T)

# The standardized frame is retained only for the coverage audit below.
del tc_raw
gc.collect()

capacity_kwargs = dict(SCREENED_KWARGS)
for key in (
    "signal_df", "freq", "portfolio_dir", "start_year", "end_year",
    "eval_start_year", "country", "delay_list", "load_signal",
    "tc_aum_dollars", "tc_execution_days",
):
    capacity_kwargs.pop(key, None)

capacity_kwargs.update(
    {
        "tc_enable": True,
        "tc_use_nonlinear": True,
        "tc_use_forecast_inputs": True,
        "tc_forecast_path": str(TC_FORECAST_PATH),
        "tradability_screens": True,
        "include_price_adv": True,
        "tc_missing_return_max_carry_weeks": MISSING_RETURN_MAX_CARRY_PERIODS,
        "tc_missing_return_warn_weight": MISSING_RETURN_WARN_WEIGHT,
    }
)


## 22. Fixed-AUM capacity grid

Each AUM level is a separate re-optimization. The strategy is **not** extrapolated from a single cost series.

The fixed-AUM experiment answers: *How does the same strategy behave if fund size is permanently set to a given AUM?*

The notebook executes model-AUM runs sequentially and releases each `PortfolioManager` after its checkpoint is written. This preserves the runner's fingerprint/resume behavior without retaining dozens of large managers in memory.


In [ ]:

# Audit frozen TC forecast coverage on the actual screened candidate universe.
#
# Missing forecast rows are not assigned historical ADV/volatility substitutes.
# The patched PortfolioManager excludes them from NEW selection while retaining
# the broader execution-support panel for legacy holdings.

diagnostic_signal = optimizer_signals["CNN"]

diag_pm = PortfolioManager(
    signal_df=diagnostic_signal,
    freq=FREQ,
    portfolio_dir=str(OUTPUT_DIR / "_tc_coverage_audit"),
    start_year=ANALYSIS_START_YEAR,
    end_year=ANALYSIS_END_YEAR,
    eval_start_year=ANALYSIS_START_YEAR,
    country=COUNTRY,
    delay_list=[DELAY],
    load_signal=True,
    min_price=float(capacity_kwargs.get("min_price", 1.0)),
    min_adv_dollar=capacity_kwargs.get("min_adv_dollar", 10_000_000),
    tradability_screens=True,
    include_price_adv=True,
    tc_enable=False,
    tc_use_nonlinear=False,
    tc_use_forecast_inputs=False,
)

screened = diag_pm.signal_df

screened_keys = pd.DataFrame(
    {
        "Date": screened.index.get_level_values("Date"),
        "StockID": screened.index.get_level_values("StockID").astype(str),
    }
).drop_duplicates()

forecast_keys = pd.DataFrame(
    {
        "Date": tc_std.index.get_level_values("Date"),
        "StockID": tc_std.index.get_level_values("StockID").astype(str),
    }
).drop_duplicates()

coverage_check = screened_keys.merge(
    forecast_keys.assign(has_tc_forecast=True),
    on=["Date", "StockID"],
    how="left",
    validate="one_to_one",
)

missing_tc = (
    coverage_check.loc[
        coverage_check["has_tc_forecast"].isna(),
        ["Date", "StockID"],
    ]
    .sort_values(["Date", "StockID"])
    .reset_index(drop=True)
)

tc_coverage_summary = pd.DataFrame(
    [
        {
            "screened_stock_date_rows": int(len(screened_keys)),
            "missing_frozen_tc_rows": int(len(missing_tc)),
            "missing_share": (
                float(len(missing_tc) / len(screened_keys))
                if len(screened_keys)
                else np.nan
            ),
        }
    ]
)

display(
    tc_coverage_summary.style.format(
        {"missing_share": "{:.6%}"}
    )
)

if len(missing_tc):
    print("Missing frozen TC rows by year:")
    display(
        missing_tc.assign(
            year=pd.to_datetime(missing_tc["Date"]).dt.year
        )
        .groupby("year")
        .size()
        .rename("missing_rows")
        .to_frame()
    )

missing_tc.to_csv(
    OUTPUT_DIR / "screened_rows_missing_frozen_tc_forecasts.csv",
    index=False,
)

tc_coverage_summary.to_csv(
    OUTPUT_DIR / "frozen_tc_forecast_coverage_summary.csv",
    index=False,
)

# Do not carry the diagnostic PortfolioManager or standardized forecast frame
# into the repeated capacity runs.
del diag_pm
del screened
del screened_keys
del forecast_keys
del coverage_check
del missing_tc
del tc_std
gc.collect()


### Missing-return accounting policy

The final implementation does not halt on a small, unobservable realized return for a legacy holding.

For a nonzero legacy position:

1. A missing realized return is marked at **0% for that holding period** and the residual position is carried.
2. If an observed return reappears, normal state accounting resumes and the missing streak resets.
3. If the return is still missing after four consecutive carry periods, the **fifth** missing period closes only the carried residual state into cash at its last marked value.
4. The terminal accounting close does **not** create an artificial trade, turnover, or transaction cost.
5. Aggregate missing-return exposure above **0.5% of a sleeve** is recorded as a materiality warning rather than stopping the run.

This is an explicit data-resolution convention, not an estimate of the true economic return. Its materiality is therefore audited across every model and AUM below.


In [ ]:

from Scripts.Portfolio.capacity_runner import (
    FixedAUMCapacityConfig,
    run_one_fixed_aum,
)

CAPACITY_DIR = OUTPUT_DIR / "fixed_aum_capacity"
CAPACITY_DIR.mkdir(parents=True, exist_ok=True)

capacity_config = FixedAUMCapacityConfig(
    output_dir=str(CAPACITY_DIR),
    aum_grid=AUM_GRID,
    weight_type=COST_AWARE_MARKOWITZ_WEIGHT_TYPE,
    freq=FREQ,
    cut=CUT,
    delay=DELAY,
    start_year=ANALYSIS_START_YEAR,
    end_year=ANALYSIS_END_YEAR,
    eval_start_year=ANALYSIS_START_YEAR,
    country=COUNTRY,
    execution_days=EXECUTION_DAYS,
    require_tradability_screens=True,
    require_forecast_inputs=True,
    save_run_details=True,
    resume=True,
    verbose=True,
)

capacity_summary_path = (
    CAPACITY_DIR / "fixed_aum_capacity_summary.csv"
)

if RUN_FIXED_AUM_CAPACITY:
    capacity_rows = []
    partial_path = (
        CAPACITY_DIR
        / "fixed_aum_capacity_summary_partial.csv"
    )

    for aum in AUM_GRID:
        print("\n" + "=" * 80)
        print(
            f"FIXED-AUM CAPACITY: "
            f"${float(aum) / 1e6:,.0f}m"
        )

        for model, signal_df in optimizer_signals.items():
            run = run_one_fixed_aum(
                signal_df,
                model=model,
                aum=float(aum),
                config=capacity_config,
                portfolio_kwargs=capacity_kwargs,
            )

            capacity_rows.append(
                dict(run.summary)
            )

            # run_one_fixed_aum saves the canonical weekly accounting and
            # diagnostics before returning. Drop large in-memory objects
            # immediately so 42 PortfolioManagers do not accumulate.
            run.manager = None
            run.portfolio_returns = None
            run.tc_debug_df = None
            run.portfolio_debug_df = None
            run.missing_return_df = None

            del run
            gc.collect()

            pd.DataFrame(
                capacity_rows
            ).to_csv(
                partial_path,
                index=False,
            )

    capacity_summary = (
        pd.DataFrame(capacity_rows)
        .sort_values(
            ["aum_dollars", "net_sharpe", "model"],
            ascending=[True, False, True],
        )
        .reset_index(drop=True)
    )

    capacity_summary.to_csv(
        capacity_summary_path,
        index=False,
    )

    # Save the principal pivots produced by the runner's full-grid wrapper.
    for metric in (
        "executed_gross_sharpe",
        "net_sharpe",
        "avg_turnover_oneway",
        "total_cost_bps_per_rebalance",
        "annualized_cost_drag",
        "position_cap_bind_date_pct",
        "trade_cap_bind_date_pct",
        "mean_unfilled_target_share",
    ):
        if metric in capacity_summary.columns:
            (
                capacity_summary.pivot(
                    index="model",
                    columns="aum_millions",
                    values=metric,
                )
                .to_csv(
                    CAPACITY_DIR
                    / f"fixed_aum_{metric}.csv"
                )
            )

    capacity_manifest = {
        "analysis": "fixed_aum_reoptimized_capacity",
        "weight_type": capacity_config.weight_type,
        "freq": capacity_config.freq,
        "cut": int(capacity_config.cut),
        "delay": int(capacity_config.delay),
        "start_year": int(capacity_config.start_year),
        "end_year": int(capacity_config.end_year),
        "eval_start_year": int(capacity_config.eval_start_year),
        "country": capacity_config.country,
        "execution_days": float(capacity_config.execution_days),
        "aum_grid": [float(x) for x in AUM_GRID],
        "models": list(optimizer_signals.keys()),
        "n_model_aum_runs": int(len(capacity_summary)),
        "missing_return_max_carry_periods": int(MISSING_RETURN_MAX_CARRY_PERIODS),
        "missing_return_warn_weight": float(MISSING_RETURN_WARN_WEIGHT),
        "missing_return_terminal_action": "last_mark_to_cash_without_artificial_trade",
        "memory_policy": (
            "Runs are executed sequentially and PortfolioManager/debug "
            "objects are released after each saved model-AUM run."
        ),
    }

    with (
        CAPACITY_DIR / "fixed_aum_capacity_manifest.json"
    ).open("w", encoding="utf-8") as f:
        json.dump(
            capacity_manifest,
            f,
            indent=2,
        )

    if partial_path.exists():
        partial_path.unlink()

    del capacity_rows
    gc.collect()

else:
    if not capacity_summary_path.exists():
        raise FileNotFoundError(
            "RUN_FIXED_AUM_CAPACITY=False but saved capacity summary is missing."
        )

    capacity_summary = pd.read_csv(
        capacity_summary_path
    )

display(capacity_summary)


### Capacity and implementation curves

The core implementation figures track four distinct channels as AUM rises:

- net Sharpe;
- realized trading cost;
- execution shortfall;
- frequency of trade-cap binding.

The HMM cost decomposition is shown separately so the relative contribution of linear spread cost and nonlinear impact remains visible.


In [ ]:
def plot_capacity_metric(
    summary,
    metric,
    ylabel,
    title,
    filename,
    percent_axis=False,
):
    if metric not in summary.columns:
        print(f"Skipping {metric}: column not available.")
        return

    fig, ax = plt.subplots(figsize=(11, 6))

    for model, g in summary.groupby("model", sort=False):
        d = g.sort_values("aum_millions")
        ax.plot(
            d["aum_millions"],
            d[metric],
            marker="o",
            label=model,
        )

    ax.set_xscale("log")
    ax.set_xlabel("AUM ($ millions, log scale)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.axhline(0.0, linewidth=0.8)
    if percent_axis:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(ncol=2)

    _save_show(fig, filename)


plot_capacity_metric(
    capacity_summary,
    "net_sharpe",
    "Annualized net Sharpe",
    "Corrected Net Sharpe vs AUM",
    "capacity_net_sharpe.png",
)

plot_capacity_metric(
    capacity_summary,
    "total_cost_bps_per_rebalance",
    "Basis points per rebalance",
    "Realized Trading Cost vs AUM",
    "capacity_total_cost_bps.png",
)

plot_capacity_metric(
    capacity_summary,
    "mean_unfilled_target_share",
    "Mean unfilled target share",
    "Execution Shortfall vs AUM",
    "capacity_unfilled_target_share.png",
    percent_axis=True,
)

plot_capacity_metric(
    capacity_summary,
    "trade_cap_bind_date_pct",
    "Share of rebalance dates",
    "Trade-Cap Binding Frequency vs AUM",
    "capacity_trade_cap_binding.png",
    percent_axis=True,
)


# Cost decomposition for the focal HMM implementation.
_hmm_cost_cols = [
    "linear_cost_bps_per_rebalance",
    "impact_cost_bps_per_rebalance",
    "total_cost_bps_per_rebalance",
]

if all(c in capacity_summary.columns for c in _hmm_cost_cols):
    hmm_cost = (
        capacity_summary.loc[
            capacity_summary["model"].eq("HMM"),
            ["aum_millions"] + _hmm_cost_cols,
        ]
        .sort_values("aum_millions")
        .copy()
    )

    if len(hmm_cost):
        fig, ax = plt.subplots(figsize=(11, 6))
        ax.plot(
            hmm_cost["aum_millions"],
            hmm_cost["linear_cost_bps_per_rebalance"],
            marker="o",
            label="Linear spread cost",
        )
        ax.plot(
            hmm_cost["aum_millions"],
            hmm_cost["impact_cost_bps_per_rebalance"],
            marker="o",
            label="Nonlinear impact cost",
        )
        ax.plot(
            hmm_cost["aum_millions"],
            hmm_cost["total_cost_bps_per_rebalance"],
            marker="o",
            linewidth=2,
            label="Total realized cost",
        )
        ax.set_xscale("log")
        ax.set_xlabel("AUM ($ millions, log scale)")
        ax.set_ylabel("Basis points per rebalance")
        ax.set_title("HMM Realized Cost Decomposition vs AUM")
        ax.legend()
        _save_show(fig, "hmm_cost_decomposition.png")


print("NET SHARPE BY AUM")
display(
    capacity_summary.pivot(
        index="model",
        columns="aum_millions",
        values="net_sharpe",
    ).round(3)
)

print("COST BPS / REBALANCE BY AUM")
display(
    capacity_summary.pivot(
        index="model",
        columns="aum_millions",
        values="total_cost_bps_per_rebalance",
    ).round(2)
)

print("MEAN UNFILLED TARGET SHARE")
display(
    capacity_summary.pivot(
        index="model",
        columns="aum_millions",
        values="mean_unfilled_target_share",
    ).round(4)
)

print("TRADE-CAP BINDING DATE SHARE")
display(
    capacity_summary.pivot(
        index="model",
        columns="aum_millions",
        values="trade_cap_bind_date_pct",
    ).round(4)
)


## 23. Missing-return materiality audit

Because the missing-return rule is an approximation, the notebook audits its frequency, persistence, and affected sleeve weight across every fixed-AUM run.

The key diagnostic is **maximum aggregate missing-return sleeve exposure**. The policy is economically innocuous only if affected weights remain small and materiality warnings are rare or absent.


In [ ]:
def _capacity_safe_name(value):
    text = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    ).strip("_.")
    return text or "model"


required_missing_summary_cols = {
    "missing_return_event_count",
    "missing_return_unique_stock_count",
    "missing_return_terminal_event_count",
    "missing_return_materiality_warning_count",
    "missing_return_max_abs_name_weight",
    "missing_return_mean_abs_name_weight",
    "missing_return_max_abs_sleeve_weight",
    "missing_return_mean_abs_sleeve_weight",
    "missing_return_max_consecutive_periods",
}

missing_cols = required_missing_summary_cols - set(capacity_summary.columns)
if missing_cols:
    raise RuntimeError(
        "Capacity summary is missing the research missing-return diagnostics: "
        f"{sorted(missing_cols)}"
    )

missing_return_grid = (
    capacity_summary[
        [
            "model",
            "aum_dollars",
            "aum_millions",
        ]
        + sorted(required_missing_summary_cols)
    ]
    .sort_values(["aum_dollars", "model"])
    .reset_index(drop=True)
)

missing_return_grid.to_csv(
    OUTPUT_DIR / "missing_return_materiality_grid.csv",
    index=False,
)

# Collect event-level records written by each completed capacity run.
missing_event_parts = []

for aum in AUM_GRID:
    for model in optimizer_signals:
        fp = (
            CAPACITY_DIR
            / "runs"
            / _capacity_safe_name(model)
            / f"aum_{int(float(aum))}"
            / "missing_return_events.csv"
        )

        if not fp.exists():
            continue

        d = pd.read_csv(fp)
        if d.empty:
            continue

        d["date"] = pd.to_datetime(
            d["date"],
            errors="coerce",
        ).dt.normalize()

        d.insert(0, "model", model)
        d.insert(1, "aum_dollars", float(aum))
        d.insert(2, "aum_millions", float(aum) / 1e6)

        missing_event_parts.append(d)

if missing_event_parts:
    missing_return_events = pd.concat(
        missing_event_parts,
        ignore_index=True,
    )
else:
    missing_return_events = pd.DataFrame(
        columns=[
            "model",
            "aum_dollars",
            "aum_millions",
            "date",
            "state_key",
            "stock_id",
            "abs_sleeve_weight",
            "missing_abs_sleeve_weight_total",
            "consecutive_missing_periods",
            "action",
            "materiality_warning",
        ]
    )

missing_return_events.to_csv(
    OUTPUT_DIR / "missing_return_events_all_capacity_runs.csv",
    index=False,
)

print("MISSING-RETURN EVENT COUNT BY AUM")
display(
    missing_return_grid.pivot(
        index="model",
        columns="aum_millions",
        values="missing_return_event_count",
    )
)

print("TERMINAL MARK-TO-CASH COUNT BY AUM")
display(
    missing_return_grid.pivot(
        index="model",
        columns="aum_millions",
        values="missing_return_terminal_event_count",
    )
)

print("MAXIMUM AGGREGATE MISSING-RETURN SLEEVE WEIGHT")
display(
    missing_return_grid.pivot(
        index="model",
        columns="aum_millions",
        values="missing_return_max_abs_sleeve_weight",
    ).style.format("{:.3%}")
)

warning_rows = missing_return_grid.loc[
    missing_return_grid["missing_return_materiality_warning_count"] > 0
].copy()

if len(warning_rows):
    print(
        "Materiality warnings were recorded. Review these model/AUM runs "
        "before treating the approximation as immaterial."
    )
    display(
        warning_rows[
            [
                "model",
                "aum_millions",
                "missing_return_event_count",
                "missing_return_materiality_warning_count",
                "missing_return_max_abs_sleeve_weight",
                "missing_return_max_consecutive_periods",
            ]
        ].style.format(
            {
                "aum_millions": "${:,.0f}m",
                "missing_return_max_abs_sleeve_weight": "{:.3%}",
            }
        )
    )
else:
    print(
        "No fixed-AUM run exceeded the configured "
        f"{MISSING_RETURN_WARN_WEIGHT:.2%} sleeve-level materiality threshold."
    )

if len(missing_return_events):
    print("LARGEST INDIVIDUAL MISSING-RETURN EVENTS")
    largest_missing_events = (
        missing_return_events.sort_values(
            "abs_sleeve_weight",
            ascending=False,
        )
        .head(20)
    )
    display(
        largest_missing_events[
            [
                c for c in [
                    "model",
                    "aum_millions",
                    "date",
                    "state_key",
                    "stock_id",
                    "abs_sleeve_weight",
                    "missing_abs_sleeve_weight_total",
                    "consecutive_missing_periods",
                    "action",
                    "materiality_warning",
                ]
                if c in largest_missing_events.columns
            ]
        ].style.format(
            {
                "aum_millions": "${:,.0f}m",
                "abs_sleeve_weight": "{:.4%}",
                "missing_abs_sleeve_weight_total": "{:.4%}",
            }
        )
    )


# Figure 1: event counts by AUM.
fig, ax = plt.subplots(figsize=(11, 6))
for model, g in missing_return_grid.groupby("model", sort=False):
    d = g.sort_values("aum_millions")
    ax.plot(
        d["aum_millions"],
        d["missing_return_event_count"],
        marker="o",
        label=model,
    )
ax.set_xscale("log")
ax.set_xlabel("AUM ($ millions, log scale)")
ax.set_ylabel("Logged stock-period events")
ax.set_title("Missing-Return Events vs AUM")
ax.legend(ncol=2)
_save_show(fig, "missing_return_event_count.png")


# Figure 2: maximum aggregate affected sleeve exposure.
fig, ax = plt.subplots(figsize=(11, 6))
for model, g in missing_return_grid.groupby("model", sort=False):
    d = g.sort_values("aum_millions")
    ax.plot(
        d["aum_millions"],
        d["missing_return_max_abs_sleeve_weight"],
        marker="o",
        label=model,
    )
ax.axhline(
    MISSING_RETURN_WARN_WEIGHT,
    linestyle="--",
    linewidth=1,
    label=f"Warning threshold ({MISSING_RETURN_WARN_WEIGHT:.2%})",
)
ax.set_xscale("log")
ax.set_xlabel("AUM ($ millions, log scale)")
ax.set_ylabel("Maximum aggregate affected sleeve weight")
ax.set_title("Maximum Missing-Return Exposure vs AUM")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(ncol=2)
_save_show(fig, "missing_return_max_sleeve_weight.png")


## 24. Load corrected weekly net-return panels

Inference uses the completed fixed-AUM weekly accounting streams. It bootstraps **returns**, not the optimizer itself.


In [ ]:
def _safe_name(value):
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_.")
    return text or "model"

def load_capacity_net_panels(capacity_dir, models, aums):
    panels = {}

    for aum in aums:
        series = {}
        for model in models:
            fp = (
                Path(capacity_dir)
                / "runs"
                / _safe_name(model)
                / f"aum_{int(float(aum))}"
                / "weekly_accounting.csv"
            )

            if not fp.exists():
                raise FileNotFoundError(f"Missing corrected capacity accounting: {fp}")

            d = pd.read_csv(fp)
            required = {"date", "net_return", "executed_gross_return", "total_cost"}
            missing = required - set(d.columns)
            if missing:
                raise KeyError(f"{fp} missing columns: {sorted(missing)}")

            idx = pd.to_datetime(d["date"], errors="coerce").dt.normalize()
            s = pd.Series(
                pd.to_numeric(d["net_return"], errors="coerce").to_numpy(),
                index=idx,
                name=model,
            )

            if s.index.isna().any() or s.isna().any() or s.index.has_duplicates:
                raise RuntimeError(f"Invalid corrected return stream: {model} @ {aum}")

            series[model] = s.sort_index()

        panel = pd.concat(series, axis=1)

        if panel.isna().any().any():
            raise RuntimeError(
                f"Model streams do not share an identical complete sample at AUM={aum:,.0f}."
            )

        panels[float(aum)] = panel

    return panels

net_panels_by_aum = load_capacity_net_panels(
    CAPACITY_DIR,
    list(optimizer_signals.keys()),
    AUM_GRID,
)

print("Loaded corrected net panels:")
for aum, panel in net_panels_by_aum.items():
    print(f"  ${aum/1e6:,.0f}m: {len(panel):,} weeks")


### Primary-AUM after-cost growth

The fixed-AUM capacity table summarizes average performance, while the cumulative path below shows when the after-cost differences arose at the primary $100 million implementation size.


In [ ]:
primary_net_panel = net_panels_by_aum[float(PRIMARY_AUM)].copy()

primary_net_growth = (
    1.0 + primary_net_panel
).cumprod()

primary_net_growth.to_csv(
    OUTPUT_DIR / "primary_aum_cumulative_net_growth.csv"
)

fig, ax = plt.subplots(figsize=(14, 6))
for model in primary_net_growth.columns:
    ax.plot(
        primary_net_growth.index,
        primary_net_growth[model],
        label=model,
    )
ax.axhline(1.0, linewidth=0.8)
ax.set_title(
    f"After-Cost Cumulative Growth at Fixed ${PRIMARY_AUM/1e6:,.0f}m AUM"
)
ax.set_xlabel("Date")
ax.set_ylabel("Growth of $1")
ax.legend(ncol=2)
_save_show(fig, "primary_aum_cumulative_net_growth.png")


## 25. Corrected after-cost paired inference

The principal HMM comparisons are against RF, the untuned rank blend, the static logistic stack, and expected-return fusion.


In [ ]:
net_comparisons = [
    ("HMM", "RF"),
    ("HMM", "RankBlend_50_50"),
    ("HMM", "LogisticStack_WF"),
    ("HMM", "ExpectedReturnFusion_5d"),
]

selected_net_panels = {
    float(aum): net_panels_by_aum[float(aum)]
    for aum in INFERENCE_AUMS
}

net_hac, net_sharpe_boot = sv.pairwise_inference_by_aum(
    selected_net_panels,
    comparisons=net_comparisons,
    maxlags=HAC_LAGS,
    reps=BOOTSTRAP_REPS,
    block_len=BOOTSTRAP_BLOCK_WEEKS,
    seed=RANDOM_SEED,
    periods_per_year=52,
    require_identical_index=True,
)

net_hac.to_csv(OUTPUT_DIR / "corrected_net_paired_hac_by_aum.csv", index=False)
net_sharpe_boot.to_csv(OUTPUT_DIR / "corrected_net_sharpe_bootstrap_by_aum.csv", index=False)

print("CORRECTED NET PAIRED HAC")
display(
    net_hac[
        [
            "AUM_millions", "model_a", "model_b",
            "ann_mean", "ann_mean_ci_low", "ann_mean_ci_high",
            "hac_t", "p_value_two_sided",
        ]
    ].style.format(
        {
            "AUM_millions": "${:,.0f}m",
            "ann_mean": "{:+.2%}",
            "ann_mean_ci_low": "{:+.2%}",
            "ann_mean_ci_high": "{:+.2%}",
            "hac_t": "{:.3f}",
            "p_value_two_sided": "{:.4g}",
        }
    )
)

print("CORRECTED NET SHARPE DIFFERENCE BOOTSTRAP")
display(
    net_sharpe_boot[
        [
            "AUM_millions", "model_a", "model_b",
            "observed_delta_sharpe", "ci_low", "ci_high",
            "bootstrap_prob_delta_gt_0",
        ]
    ].style.format(
        {
            "AUM_millions": "${:,.0f}m",
            "observed_delta_sharpe": "{:+.3f}",
            "ci_low": "{:+.3f}",
            "ci_high": "{:+.3f}",
            "bootstrap_prob_delta_gt_0": "{:.3f}",
        }
    )
)


In [ ]:
# Visualize the HMM Sharpe difference and its paired block-bootstrap interval.
fig, ax = plt.subplots(figsize=(11, 6))

for benchmark, g in net_sharpe_boot.groupby("model_b", sort=False):
    d = g.sort_values("AUM_millions")
    observed = d["observed_delta_sharpe"].to_numpy(dtype=float)
    lower = d["ci_low"].to_numpy(dtype=float)
    upper = d["ci_high"].to_numpy(dtype=float)

    yerr = np.vstack(
        [
            observed - lower,
            upper - observed,
        ]
    )

    ax.errorbar(
        d["AUM_millions"],
        observed,
        yerr=yerr,
        marker="o",
        capsize=3,
        label=f"HMM − {benchmark}",
    )

ax.axhline(0.0, linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("AUM ($ millions, log scale)")
ax.set_ylabel("Difference in annualized Sharpe")
ax.set_title("HMM Net Sharpe Advantage with 95% Block-Bootstrap Intervals")
ax.legend()

_save_show(fig, "hmm_net_delta_sharpe_ci.png")


## 26. Net Sharpe confidence intervals, net spanning, and statistical capacity

Economic capacity is the largest tested AUM with positive point-estimate net Sharpe.

Statistical capacity is the largest tested AUM whose lower 95% block-bootstrap Sharpe confidence bound remains above zero.


In [ ]:
# Own-strategy net Sharpe confidence intervals at selected AUM.
own_ci_parts = []
for i, aum in enumerate(INFERENCE_AUMS):
    d = sv.bootstrap_sharpe_panel(
        net_panels_by_aum[float(aum)],
        reps=BOOTSTRAP_REPS,
        block_len=BOOTSTRAP_BLOCK_WEEKS,
        seed=RANDOM_SEED + 5000 + i,
        periods_per_year=52,
    )
    d.insert(0, "AUM_dollars", float(aum))
    d.insert(1, "AUM_millions", float(aum) / 1e6)
    own_ci_parts.append(d)

net_own_sharpe_ci = pd.concat(own_ci_parts, ignore_index=True)
net_own_sharpe_ci.to_csv(OUTPUT_DIR / "corrected_net_own_sharpe_ci.csv", index=False)

display(
    net_own_sharpe_ci[
        ["AUM_millions", "model", "observed_sharpe", "ci_low", "ci_high"]
    ].style.format(
        {
            "AUM_millions": "${:,.0f}m",
            "observed_sharpe": "{:.3f}",
            "ci_low": "{:.3f}",
            "ci_high": "{:.3f}",
        }
    )
)

# Net spanning at selected AUM.
net_spanning_parts = []
for aum in INFERENCE_AUMS:
    panel = net_panels_by_aum[float(aum)]
    table = sv.spanning_table(
        panel,
        specs=[
            ("HMM ~ RF", "HMM", ["RF"]),
            ("HMM ~ 50/50", "HMM", ["RankBlend_50_50"]),
            ("HMM ~ logistic", "HMM", ["LogisticStack_WF"]),
            ("HMM ~ expected-return fusion", "HMM", ["ExpectedReturnFusion_5d"]),
            ("HMM ~ RF + logistic", "HMM", ["RF", "LogisticStack_WF"]),
        ],
        maxlags=HAC_LAGS,
        periods_per_year=52,
    )
    table.insert(0, "AUM_dollars", float(aum))
    table.insert(1, "AUM_millions", float(aum) / 1e6)
    net_spanning_parts.append(table)

net_spanning = pd.concat(net_spanning_parts, ignore_index=True)
net_spanning.to_csv(OUTPUT_DIR / "corrected_net_spanning_by_aum.csv", index=False)

display(
    net_spanning[
        [
            "AUM_millions", "regression", "ann_alpha",
            "ann_alpha_ci_low", "ann_alpha_ci_high",
            "alpha_t_hac", "alpha_p_two_sided", "r_squared",
        ]
    ].style.format(
        {
            "AUM_millions": "${:,.0f}m",
            "ann_alpha": "{:+.2%}",
            "ann_alpha_ci_low": "{:+.2%}",
            "ann_alpha_ci_high": "{:+.2%}",
            "alpha_t_hac": "{:.3f}",
            "alpha_p_two_sided": "{:.4g}",
            "r_squared": "{:.4f}",
        }
    )
)

# Capacity inference for every model.
capacity_inference_parts = []
capacity_threshold_rows = []

for i, model in enumerate(optimizer_signals):
    returns_by_aum = {
        float(aum): net_panels_by_aum[float(aum)][model]
        for aum in AUM_GRID
    }

    table = sv.capacity_sharpe_inference(
        returns_by_aum,
        model=model,
        reps=BOOTSTRAP_REPS,
        block_len=BOOTSTRAP_BLOCK_WEEKS,
        seed=RANDOM_SEED + 10_000 + 100 * i,
        periods_per_year=52,
    )
    capacity_inference_parts.append(table)

    thresholds = sv.capacity_thresholds(table)
    thresholds["model"] = model
    capacity_threshold_rows.append(thresholds)

statistical_capacity_table = pd.concat(capacity_inference_parts, ignore_index=True)
capacity_threshold_table = pd.DataFrame(capacity_threshold_rows)

statistical_capacity_table.to_csv(
    OUTPUT_DIR / "statistical_capacity_sharpe_grid.csv",
    index=False,
)
capacity_threshold_table.to_csv(
    OUTPUT_DIR / "economic_and_statistical_capacity_thresholds.csv",
    index=False,
)

print("GRID-BASED CAPACITY THRESHOLDS")
display(capacity_threshold_table)


# Sharpe point estimates and uncertainty across the complete AUM grid.
fig, ax = plt.subplots(figsize=(12, 7))

for model, g in statistical_capacity_table.groupby("model", sort=False):
    d = g.sort_values("AUM_millions")
    observed = d["observed_sharpe"].to_numpy(dtype=float)
    lower = d["ci_low"].to_numpy(dtype=float)
    upper = d["ci_high"].to_numpy(dtype=float)

    yerr = np.vstack(
        [
            observed - lower,
            upper - observed,
        ]
    )

    ax.errorbar(
        d["AUM_millions"],
        observed,
        yerr=yerr,
        marker="o",
        capsize=2,
        label=model,
    )

ax.axhline(0.0, linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("AUM ($ millions, log scale)")
ax.set_ylabel("Annualized net Sharpe")
ax.set_title("Statistical Capacity: Net Sharpe with 95% Bootstrap Intervals")
ax.legend(ncol=2)

_save_show(fig, "statistical_capacity_sharpe_ci.png")


## 27. Multiple-testing robustness: Hansen SPA and Deflated Sharpe

SPA is used as a limited robustness check at selected AUM levels. It tests mean-return performance differentials relative to a declared benchmark; it is not described as a correction for Sharpe selection.

Deflated Sharpe is reported only as supplementary evidence because its interpretation depends on the effective number of strategies tried.


In [ ]:
spa_rows = []
dsr_rows = []

for i, aum in enumerate((100_000_000.0, 1_000_000_000.0)):
    panel = net_panels_by_aum[float(aum)]

    spa, _ = sv.hansen_spa_test(
        panel,
        benchmark="RF",
        candidates=[
            "HMM",
            "RankBlend_50_50",
            "LogisticStack_WF",
            "ExpectedReturnFusion_5d",
        ],
        reps=BOOTSTRAP_REPS,
        block_len=BOOTSTRAP_BLOCK_WEEKS,
        maxlags=HAC_LAGS,
        seed=RANDOM_SEED + 20_000 + i,
    )
    spa["AUM_dollars"] = float(aum)
    spa["AUM_millions"] = float(aum) / 1e6
    spa_rows.append(spa)

    dsr = sv.deflated_sharpe_from_panel(
        panel,
        selected_model="HMM",
        trial_models=list(panel.columns),
        effective_num_trials=None,
        periods_per_year=52,
    )
    dsr["AUM_dollars"] = float(aum)
    dsr["AUM_millions"] = float(aum) / 1e6
    dsr_rows.append(dsr)

spa_results = pd.DataFrame(spa_rows)
dsr_results = pd.DataFrame(dsr_rows)

spa_results.to_csv(OUTPUT_DIR / "hansen_spa_selected_aum.csv", index=False)
dsr_results.to_csv(OUTPUT_DIR / "deflated_sharpe_selected_aum.csv", index=False)

print("HANSEN SPA")
display(spa_results)

print("DEFLATED SHARPE — SUPPLEMENTARY")
display(dsr_results)


# Part V — Dynamic NAV

Fixed-AUM capacity holds fund size constant. Dynamic NAV instead feeds current fund size back into market impact, position caps, trade caps, and dollar trades:

$$
NAV_{t+1}=NAV_t(1+R^{net}_t).
$$

The resulting path is descriptive and path-dependent. A simple bootstrap of one realized dynamic-NAV return sequence does **not** reproduce the optimizer/NAV feedback law under alternative histories.


## 28. Dynamic-NAV simulation

Dynamic-NAV models are likewise executed one at a time and released after their saved accounting path is captured.


In [ ]:
import importlib
import Scripts.Portfolio.dynamic_nav_runner as dyn

importlib.reload(dyn)

DynamicNAVConfig = dyn.DynamicNAVConfig
run_one_dynamic_nav = dyn.run_one_dynamic_nav

print("dynamic_nav_runner reloaded")

In [ ]:

from Scripts.Portfolio.dynamic_nav_runner import (
    DynamicNAVConfig,
    run_one_dynamic_nav,
)

DYNAMIC_NAV_DIR = OUTPUT_DIR / "dynamic_nav"
DYNAMIC_NAV_DIR.mkdir(parents=True, exist_ok=True)

dynamic_config = DynamicNAVConfig(
    output_dir=str(DYNAMIC_NAV_DIR),
    initial_nav=PRIMARY_AUM,
    weight_type=COST_AWARE_MARKOWITZ_WEIGHT_TYPE,
    freq=FREQ,
    cut=CUT,
    delay=DELAY,
    start_year=ANALYSIS_START_YEAR,
    end_year=ANALYSIS_END_YEAR,
    eval_start_year=ANALYSIS_START_YEAR,
    country=COUNTRY,
    execution_days=EXECUTION_DAYS,
    require_tradability_screens=True,
    require_forecast_inputs=True,
    save_run_details=True,
    resume=True,
    verbose=True,
)

dynamic_summary_path = (
    DYNAMIC_NAV_DIR / "dynamic_nav_summary.csv"
)
dynamic_paths_path = (
    DYNAMIC_NAV_DIR / "dynamic_nav_paths.csv"
)

if RUN_DYNAMIC_NAV:
    dynamic_rows = []
    nav_parts = []
    dynamic_missing_parts = []

    for model, signal_df in optimizer_signals.items():
        run = run_one_dynamic_nav(
            signal_df,
            model=model,
            config=dynamic_config,
            portfolio_kwargs=capacity_kwargs,
        )

        run_row = dict(run.summary)

        # Preserve missing-return diagnostics before releasing the manager.
        missing_df = None
        missing_summary = {}

        if run.manager is not None:
            raw_missing = getattr(
                run.manager,
                "_last_missing_return_df",
                None,
            )
            if isinstance(raw_missing, pd.DataFrame):
                missing_df = raw_missing.copy()

            missing_summary = dict(
                getattr(
                    run.manager,
                    "_last_missing_return_summary",
                    None,
                )
                or {}
            )

        per_run_missing_path = (
            run.output_dir
            / "missing_return_events.csv"
        )

        if missing_df is not None and len(missing_df):
            missing_df.to_csv(
                per_run_missing_path,
                index=False,
            )
        elif per_run_missing_path.exists():
            missing_df = pd.read_csv(
                per_run_missing_path
            )

        if isinstance(missing_df, pd.DataFrame) and len(missing_df):
            detail = missing_df.copy()
            detail.insert(0, "model", model)
            dynamic_missing_parts.append(detail)

            # Reconstruct the summary if this run was loaded from a checkpoint.
            if not missing_summary:
                materiality = (
                    detail["materiality_warning"]
                    .astype(str)
                    .str.lower()
                    .isin(["true", "1"])
                    if "materiality_warning" in detail.columns
                    else pd.Series(False, index=detail.index)
                )
                missing_summary = {
                    "event_count": int(len(detail)),
                    "unique_stock_count": int(
                        detail["stock_id"].astype(str).nunique()
                    )
                    if "stock_id" in detail.columns
                    else 0,
                    "terminal_event_count": int(
                        detail["action"].astype(str)
                        .eq("terminal_mark_to_cash")
                        .sum()
                    )
                    if "action" in detail.columns
                    else 0,
                    "materiality_warning_count": int(
                        materiality.sum()
                    ),
                    "max_missing_abs_sleeve_weight": float(
                        pd.to_numeric(
                            detail.get(
                                "missing_abs_sleeve_weight_total",
                                pd.Series(0.0, index=detail.index),
                            ),
                            errors="coerce",
                        )
                        .fillna(0.0)
                        .max()
                    ),
                }

        run_row.update(
            {
                "missing_return_event_count": int(
                    missing_summary.get(
                        "event_count",
                        0,
                    )
                    or 0
                ),
                "missing_return_unique_stock_count": int(
                    missing_summary.get(
                        "unique_stock_count",
                        0,
                    )
                    or 0
                ),
                "missing_return_terminal_event_count": int(
                    missing_summary.get(
                        "terminal_event_count",
                        0,
                    )
                    or 0
                ),
                "missing_return_materiality_warning_count": int(
                    missing_summary.get(
                        "materiality_warning_count",
                        0,
                    )
                    or 0
                ),
                "missing_return_max_abs_sleeve_weight": float(
                    missing_summary.get(
                        "max_missing_abs_sleeve_weight",
                        0.0,
                    )
                    or 0.0
                ),
            }
        )

        dynamic_rows.append(run_row)

        path = (
            run.nav_accounting[
                ["date", "nav_end"]
            ]
            .copy()
            .rename(
                columns={"nav_end": model}
            )
        )
        nav_parts.append(path)

        # The per-model files are already checkpointed by the runner.
        # Release the heavy manager/debug state before starting the next model.
        run.manager = None
        run.portfolio_returns = None
        run.tc_debug_df = None
        run.portfolio_debug_df = None

        del run
        gc.collect()

    dynamic_nav_summary = (
        pd.DataFrame(dynamic_rows)
        .sort_values(
            "net_sharpe",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    dynamic_nav_summary.to_csv(
        dynamic_summary_path,
        index=False,
    )

    if nav_parts:
        nav_paths = nav_parts[0]

        for part in nav_parts[1:]:
            nav_paths = nav_paths.merge(
                part,
                on="date",
                how="outer",
                validate="one_to_one",
            )

        nav_paths = nav_paths.sort_values(
            "date"
        )

        nav_paths.to_csv(
            dynamic_paths_path,
            index=False,
        )

    dynamic_manifest = {
        "initial_nav": float(dynamic_config.initial_nav),
        "weight_type": dynamic_config.weight_type,
        "freq": dynamic_config.freq,
        "cut": int(dynamic_config.cut),
        "delay": int(dynamic_config.delay),
        "start_year": int(dynamic_config.start_year),
        "end_year": int(dynamic_config.end_year),
        "eval_start_year": int(dynamic_config.eval_start_year),
        "country": dynamic_config.country,
        "execution_days": float(dynamic_config.execution_days),
        "models": list(optimizer_signals.keys()),
        "missing_return_max_carry_periods": int(MISSING_RETURN_MAX_CARRY_PERIODS),
        "missing_return_warn_weight": float(MISSING_RETURN_WARN_WEIGHT),
        "missing_return_terminal_action": "last_mark_to_cash_without_artificial_trade",
        "accounting_identity": (
            "NAV_{t+1}=NAV_t*(1+R_net,t)"
        ),
        "memory_policy": (
            "Models are simulated sequentially and PortfolioManager/debug "
            "objects are released after each saved run."
        ),
    }

    with (
        DYNAMIC_NAV_DIR
        / "dynamic_nav_manifest.json"
    ).open("w", encoding="utf-8") as f:
        json.dump(
            dynamic_manifest,
            f,
            indent=2,
        )

    if dynamic_missing_parts:
        dynamic_missing_return_events = pd.concat(
            dynamic_missing_parts,
            ignore_index=True,
        )
    else:
        dynamic_missing_return_events = pd.DataFrame()

    dynamic_missing_return_events.to_csv(
        DYNAMIC_NAV_DIR / "dynamic_nav_missing_return_events.csv",
        index=False,
    )

    del dynamic_rows
    del nav_parts
    del dynamic_missing_parts
    gc.collect()

else:
    if not dynamic_summary_path.exists():
        raise FileNotFoundError(
            "RUN_DYNAMIC_NAV=False but saved dynamic-NAV summary is missing."
        )

    dynamic_nav_summary = pd.read_csv(
        dynamic_summary_path
    )

    dynamic_missing_path = (
        DYNAMIC_NAV_DIR
        / "dynamic_nav_missing_return_events.csv"
    )
    dynamic_missing_return_events = (
        pd.read_csv(dynamic_missing_path)
        if dynamic_missing_path.exists()
        else pd.DataFrame()
    )

display(dynamic_nav_summary)


### Dynamic-NAV paths


In [ ]:
nav_path_file = DYNAMIC_NAV_DIR / "dynamic_nav_paths.csv"

if nav_path_file.exists():
    nav_paths = pd.read_csv(nav_path_file)
    nav_paths["date"] = pd.to_datetime(
        nav_paths["date"],
        errors="coerce",
    ).dt.normalize()

    nav_paths = (
        nav_paths
        .set_index("date")
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    for model in nav_paths.columns:
        ax.plot(
            nav_paths.index,
            nav_paths[model],
            label=model,
        )
    ax.set_title(
        f"Dynamic NAV from ${PRIMARY_AUM/1e6:,.0f}m Initial Capital"
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("NAV ($)")
    ax.legend(ncol=2)
    _save_show(fig, "dynamic_nav_paths.png")

    # Drawdown paths reveal implementation risk that terminal NAV alone can hide.
    dynamic_drawdowns = (
        nav_paths
        / nav_paths.cummax()
        - 1.0
    )

    dynamic_drawdowns.to_csv(
        OUTPUT_DIR / "dynamic_nav_drawdown_paths.csv"
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    for model in dynamic_drawdowns.columns:
        ax.plot(
            dynamic_drawdowns.index,
            dynamic_drawdowns[model],
            label=model,
        )
    ax.axhline(0.0, linewidth=0.8)
    ax.set_title("Dynamic-NAV Drawdown Paths")
    ax.set_xlabel("Date")
    ax.set_ylabel("Drawdown")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(ncol=2)
    _save_show(fig, "dynamic_nav_drawdowns.png")

else:
    print(
        "Dynamic NAV path file not found yet:",
        nav_path_file,
    )


terminal_cols = [
    c for c in [
        "model",
        "initial_nav",
        "simulation_final_nav",
        "eval_final_nav",
        "net_sharpe",
        "max_drawdown",
        "avg_turnover_oneway",
        "mean_unfilled_target_share",
        "missing_return_event_count",
        "missing_return_terminal_event_count",
        "missing_return_materiality_warning_count",
        "missing_return_max_abs_sleeve_weight",
    ]
    if c in dynamic_nav_summary.columns
]

display(
    dynamic_nav_summary[terminal_cols].style.format(
        {
            "initial_nav": "${:,.0f}",
            "simulation_final_nav": "${:,.0f}",
            "eval_final_nav": "${:,.0f}",
            "net_sharpe": "{:.3f}",
            "max_drawdown": "{:.2%}",
            "avg_turnover_oneway": "{:.3f}",
            "mean_unfilled_target_share": "{:.2%}",
            "missing_return_max_abs_sleeve_weight": "{:.3%}",
        }
    )
)

if "missing_return_materiality_warning_count" in dynamic_nav_summary.columns:
    dynamic_warning_total = int(
        pd.to_numeric(
            dynamic_nav_summary[
                "missing_return_materiality_warning_count"
            ],
            errors="coerce",
        )
        .fillna(0)
        .sum()
    )

    if dynamic_warning_total:
        print(
            "Dynamic NAV recorded missing-return materiality warnings:",
            dynamic_warning_total,
        )
    else:
        print(
            "Dynamic NAV recorded no missing-return materiality warnings."
        )


# Part VI — Final research dashboard

The dashboard is deliberately compact. It should be the starting point for the written interpretation, while the preceding sections provide the evidence behind each conclusion.


In [ ]:
# =========================================================================
# FINAL-DASHBOARD RECOVERY AFTER KERNEL RESTART
#
# Reload all expensive results from saved artifacts instead of recomputing
# the earlier portfolio, bootstrap, capacity, and dynamic-NAV sections.
# =========================================================================

from pathlib import Path
import pandas as pd
import numpy as np


# -------------------------------------------------------------------------
# Gross equal-weight results
# -------------------------------------------------------------------------

gross_perf_path = OUTPUT_DIR / "gross_ew_performance.csv"
gross_boot_path = OUTPUT_DIR / "gross_sharpe_difference_bootstrap.csv"
spanning_path = OUTPUT_DIR / "gross_spanning_regressions.csv"

for path in (
    gross_perf_path,
    gross_boot_path,
    spanning_path,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Required saved result is missing: {path}"
        )

gross_perf = pd.read_csv(
    gross_perf_path,
    index_col=0,
)

gross_sharpe_boot = pd.read_csv(
    gross_boot_path
)

spanning_results = pd.read_csv(
    spanning_path
)

# Keep the final formatting cell robust if an older saved table did not
# persist turnover as a column.
if "avg_turnover" not in gross_perf.columns:
    gross_perf["avg_turnover"] = np.nan


# -------------------------------------------------------------------------
# Common-optimizer gross Markowitz results
# -------------------------------------------------------------------------

gross_mw_perf_path = (
    OUTPUT_DIR
    / "same_optimizer_gross_markowitz_performance.csv"
)

if not gross_mw_perf_path.exists():
    raise FileNotFoundError(
        f"Saved gross Markowitz performance is missing: "
        f"{gross_mw_perf_path}"
    )

gross_mw_perf = pd.read_csv(
    gross_mw_perf_path,
    index_col=0,
)

if "avg_turnover" not in gross_mw_perf.columns:
    gross_mw_perf["avg_turnover"] = np.nan


# -------------------------------------------------------------------------
# Fixed-AUM capacity results
# -------------------------------------------------------------------------

CAPACITY_DIR = OUTPUT_DIR / "fixed_aum_capacity"

capacity_summary_path = (
    CAPACITY_DIR
    / "fixed_aum_capacity_summary.csv"
)

if not capacity_summary_path.exists():
    raise FileNotFoundError(
        f"Saved capacity summary is missing: "
        f"{capacity_summary_path}"
    )

capacity_summary = pd.read_csv(
    capacity_summary_path
)


# -------------------------------------------------------------------------
# Missing-return materiality results
# -------------------------------------------------------------------------

missing_return_grid_path = (
    OUTPUT_DIR
    / "missing_return_materiality_grid.csv"
)

if not missing_return_grid_path.exists():
    raise FileNotFoundError(
        f"Saved missing-return grid is missing: "
        f"{missing_return_grid_path}"
    )

missing_return_grid = pd.read_csv(
    missing_return_grid_path
)


# -------------------------------------------------------------------------
# Net HMM bootstrap comparisons
# -------------------------------------------------------------------------

net_sharpe_boot_path = (
    OUTPUT_DIR
    / "corrected_net_sharpe_bootstrap_by_aum.csv"
)

if not net_sharpe_boot_path.exists():
    raise FileNotFoundError(
        f"Saved net Sharpe bootstrap is missing: "
        f"{net_sharpe_boot_path}"
    )

net_sharpe_boot = pd.read_csv(
    net_sharpe_boot_path
)


# -------------------------------------------------------------------------
# Economic / statistical capacity thresholds
# -------------------------------------------------------------------------

capacity_threshold_path = (
    OUTPUT_DIR
    / "economic_and_statistical_capacity_thresholds.csv"
)

if not capacity_threshold_path.exists():
    raise FileNotFoundError(
        f"Saved capacity thresholds are missing: "
        f"{capacity_threshold_path}"
    )

capacity_threshold_table = pd.read_csv(
    capacity_threshold_path
)


# -------------------------------------------------------------------------
# Dynamic NAV
# -------------------------------------------------------------------------

DYNAMIC_NAV_DIR = OUTPUT_DIR / "dynamic_nav"

dynamic_summary_path = (
    DYNAMIC_NAV_DIR
    / "dynamic_nav_summary.csv"
)

if not dynamic_summary_path.exists():
    raise FileNotFoundError(
        f"Saved Dynamic-NAV summary is missing: "
        f"{dynamic_summary_path}"
    )

dynamic_nav_summary = pd.read_csv(
    dynamic_summary_path
)


# -------------------------------------------------------------------------
# Sanity checks
# -------------------------------------------------------------------------

required_capacity_cols = {
    "model",
    "aum_millions",
    "net_sharpe",
}

missing = required_capacity_cols - set(
    capacity_summary.columns
)

if missing:
    raise RuntimeError(
        "Recovered capacity summary is incomplete: "
        f"{sorted(missing)}"
    )

required_threshold_cols = {
    "model",
    "economic_capacity_max_tested_aum",
    "statistical_capacity_max_tested_aum",
}

missing = required_threshold_cols - set(
    capacity_threshold_table.columns
)

if missing:
    raise RuntimeError(
        "Recovered capacity-threshold table is incomplete: "
        f"{sorted(missing)}"
    )


print("Final-dashboard recovery: PASS")
print()
print(f"Gross performance:       {gross_perf.shape}")
print(f"Gross bootstrap:         {gross_sharpe_boot.shape}")
print(f"Spanning results:        {spanning_results.shape}")
print(f"Gross Markowitz:         {gross_mw_perf.shape}")
print(f"Capacity summary:        {capacity_summary.shape}")
print(f"Missing-return grid:     {missing_return_grid.shape}")
print(f"Net bootstrap:           {net_sharpe_boot.shape}")
print(f"Capacity thresholds:     {capacity_threshold_table.shape}")
print(f"Dynamic NAV summary:     {dynamic_nav_summary.shape}")

In [ ]:
print("=== GROSS EQUAL-WEIGHT PERFORMANCE ===")
display(
    gross_perf.style.format(
        {
            "ann_mean": "{:.2%}",
            "ann_vol": "{:.2%}",
            "sharpe": "{:.3f}",
            "avg_turnover": "{:.3f}",
        }
    )
)

print("\n=== GROSS PAIRED HMM INFERENCE ===")
display(
    gross_sharpe_boot[
        ["model_a", "model_b", "observed_delta_sharpe", "ci_low", "ci_high", "bootstrap_prob_delta_gt_0"]
    ].style.format(
        {
            "observed_delta_sharpe": "{:+.3f}",
            "ci_low": "{:+.3f}",
            "ci_high": "{:+.3f}",
            "bootstrap_prob_delta_gt_0": "{:.3f}",
        }
    )
)

print("\n=== GROSS SPANNING ===")
display(
    spanning_results[
        ["regression", "ann_alpha", "ann_alpha_ci_low", "ann_alpha_ci_high", "alpha_p_two_sided", "r_squared"]
    ].style.format(
        {
            "ann_alpha": "{:+.2%}",
            "ann_alpha_ci_low": "{:+.2%}",
            "ann_alpha_ci_high": "{:+.2%}",
            "alpha_p_two_sided": "{:.4g}",
            "r_squared": "{:.4f}",
        }
    )
)

print("\n=== COMMON-OPTIMIZER GROSS MARKOWITZ ===")
display(
    gross_mw_perf.style.format(
        {
            "ann_mean": "{:.2%}",
            "ann_vol": "{:.2%}",
            "sharpe": "{:.3f}",
            "avg_turnover": "{:.3f}",
        }
    )
)

print("\n=== CORRECTED FIXED-AUM NET SHARPE ===")
display(
    capacity_summary.pivot(
        index="model",
        columns="aum_millions",
        values="net_sharpe",
    ).round(3)
)

print("\n=== ECONOMIC / STATISTICAL CAPACITY ===")
display(capacity_threshold_table)

print("\n=== CORRECTED NET HMM SHARPE DIFFERENCES ===")
display(
    net_sharpe_boot[
        [
            "AUM_millions", "model_a", "model_b",
            "observed_delta_sharpe", "ci_low", "ci_high",
            "bootstrap_prob_delta_gt_0",
        ]
    ].style.format(
        {
            "AUM_millions": "${:,.0f}m",
            "observed_delta_sharpe": "{:+.3f}",
            "ci_low": "{:+.3f}",
            "ci_high": "{:+.3f}",
            "bootstrap_prob_delta_gt_0": "{:.3f}",
        }
    )
)

print("\n=== DYNAMIC NAV ===")
display(dynamic_nav_summary)


print("\n=== FINAL MODEL DASHBOARD ===")

capacity_pivot = capacity_summary.pivot(
    index="model",
    columns="aum_millions",
    values="net_sharpe",
)

_capacity_cols = [
    float(PRIMARY_AUM / 1e6),
    1000.0,
]

missing_dashboard_aums = [
    x for x in _capacity_cols
    if x not in capacity_pivot.columns
]

if missing_dashboard_aums:
    raise RuntimeError(
        "Final dashboard is missing required AUM columns: "
        f"{missing_dashboard_aums}"
    )

capacity_key = (
    capacity_pivot[_capacity_cols]
    .rename(
        columns={
            float(PRIMARY_AUM / 1e6): "net_sharpe_primary_aum",
            1000.0: "net_sharpe_1bn",
        }
    )
)

dashboard = capacity_key.copy()

thresholds = (
    capacity_threshold_table
    .set_index("model")
    [
        [
            "economic_capacity_max_tested_aum",
            "statistical_capacity_max_tested_aum",
        ]
    ]
)

dashboard = dashboard.join(
    thresholds,
    how="left",
)

missing_by_model = (
    missing_return_grid
    .groupby("model", as_index=True)
    .agg(
        missing_return_events_all_runs=(
            "missing_return_event_count",
            "sum",
        ),
        missing_return_terminal_events_all_runs=(
            "missing_return_terminal_event_count",
            "sum",
        ),
        missing_return_warnings_all_runs=(
            "missing_return_materiality_warning_count",
            "sum",
        ),
        max_missing_return_sleeve_weight=(
            "missing_return_max_abs_sleeve_weight",
            "max",
        ),
    )
)

dashboard = dashboard.join(
    missing_by_model,
    how="left",
)

if "model" in dynamic_nav_summary.columns:
    dynamic_key_cols = [
        c for c in [
            "model",
            "simulation_final_nav",
            "net_sharpe",
            "max_drawdown",
        ]
        if c in dynamic_nav_summary.columns
    ]

    dynamic_key = (
        dynamic_nav_summary[dynamic_key_cols]
        .drop_duplicates("model")
        .set_index("model")
        .rename(
            columns={
                "simulation_final_nav": "dynamic_final_nav",
                "net_sharpe": "dynamic_net_sharpe",
                "max_drawdown": "dynamic_max_drawdown",
            }
        )
    )

    dashboard = dashboard.join(
        dynamic_key,
        how="left",
    )

dashboard.index.name = "model"
final_model_dashboard = dashboard.reset_index()

final_model_dashboard.to_csv(
    OUTPUT_DIR / "final_model_dashboard.csv",
    index=False,
)

format_map = {
    "net_sharpe_primary_aum": "{:.3f}",
    "net_sharpe_1bn": "{:.3f}",
    "economic_capacity_max_tested_aum": "${:,.0f}",
    "statistical_capacity_max_tested_aum": "${:,.0f}",
    "max_missing_return_sleeve_weight": "{:.3%}",
    "dynamic_final_nav": "${:,.0f}",
    "dynamic_net_sharpe": "{:.3f}",
    "dynamic_max_drawdown": "{:.2%}",
}

display(
    final_model_dashboard.style.format(
        {
            k: v
            for k, v in format_map.items()
            if k in final_model_dashboard.columns
        }
    )
)

print("\nAll research outputs:")
print(OUTPUT_DIR)


In [ ]:
# =============================================================================
# ROBUSTNESS AUDIT A — MISSING-RETURN SENSITIVITY
# =============================================================================
#
# Purpose
# -------
# Test whether the fixed-AUM RF/HMM conclusions depend materially on the
# convention used when a NONZERO legacy holding has an unobservable realized
# return.
#
# Three cases are rerun:
#
#   1. baseline_0pct
#        Current research convention. Missing held return = 0%.
#
#   2. first_gap_minus25pct
#        A 25% loss is imposed on the FIRST missing period in a consecutive
#        missing-return spell. Subsequent missing periods in the same spell
#        remain flat.
#
#   3. first_gap_total_loss
#        A 100% loss is imposed on the FIRST missing period. This is an
#        intentionally severe downside bound.
#
# The existing bounded carry/terminal-mark-to-cash policy remains intact.
#
# The audit compares:
#   - net Sharpe;
#   - HMM - RF Sharpe spread;
#   - weekly return-path changes relative to baseline;
#   - whether the HMM/RF ordering at each AUM is preserved.
#
# Run after optimizer_signals, capacity_kwargs, and the main capacity setup
# have been created.
# =============================================================================

from contextlib import contextmanager
from pathlib import Path
import gc
import warnings

import numpy as np
import pandas as pd

from Scripts.Portfolio.capacity_runner import (
    FixedAUMCapacityConfig,
    run_one_fixed_aum,
)
from Scripts.Portfolio.rebalance_state import build_rebalance_result
from Scripts.Portfolio.strategies import TCOptimizerMixin


# -----------------------------------------------------------------------------
# Audit configuration
# -----------------------------------------------------------------------------

MR_AUDIT_MODELS = ("RF", "HMM")

MR_AUDIT_AUMS = (
    25_000_000.0,
    100_000_000.0,
    1_000_000_000.0,
)

MR_SCENARIOS = {
    "baseline_0pct": 0.00,
    "first_gap_minus25pct": -0.25,
    "first_gap_total_loss": -1.00,
}

MR_AUDIT_DIR = OUTPUT_DIR / "robustness_missing_return"
MR_AUDIT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def _annualized_sharpe(x, periods_per_year=52):
    x = pd.Series(x, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()

    if len(x) < 2:
        return np.nan

    vol = float(x.std(ddof=1))
    if not np.isfinite(vol) or vol <= 0.0:
        return np.nan

    return float(
        np.sqrt(periods_per_year)
        * float(x.mean())
        / vol
    )


@contextmanager
def _missing_return_stress(first_missing_return):
    """
    Temporarily stress the realized return assigned to a missing NONZERO
    legacy holding while preserving the project's canonical state/accounting
    implementation.

    The canonical `_tc_store_leg_state` method remains responsible for:
      - transaction-cost deductions from cash;
      - missing-return streak accounting;
      - the maximum carry-period rule;
      - terminal mark-to-cash;
      - materiality diagnostics;
      - any other hardening implemented in the current strategy code.

    This wrapper changes only the economic return assumption used for the
    first period of a consecutive missing-return spell.
    """

    original_method = TCOptimizerMixin._tc_store_leg_state

    def stressed_store_leg_state(
        self,
        *,
        prev_state,
        state_key,
        pretrade_state,
        posttrade_weights,
        frame,
        ret_name,
        transaction_cost=0.0,
    ):
        frame.index = pd.Index(frame.index).astype(str)

        if ret_name not in frame.columns:
            return original_method(
                self,
                prev_state=prev_state,
                state_key=state_key,
                pretrade_state=pretrade_state,
                posttrade_weights=posttrade_weights,
                frame=frame,
                ret_name=ret_name,
                transaction_cost=transaction_cost,
            )

        post = pd.to_numeric(
            pd.Series(posttrade_weights, dtype=float),
            errors="coerce",
        ).fillna(0.0)
        post.index = pd.Index(post.index).astype(str)

        held_ids = pd.Index(
            post.index[post.abs() > 1e-12]
        ).astype(str)

        raw_realized = pd.to_numeric(
            frame[ret_name].reindex(held_ids),
            errors="coerce",
        )

        missing_ids = pd.Index(
            raw_realized.index[raw_realized.isna()]
        ).astype(str)

        streak_store = prev_state.get(
            "tc_missing_return_streaks",
            {},
        )
        prior_leg_streaks = dict(
            streak_store.get(
                str(state_key),
                {},
            )
        )

        first_missing_ids = {
            str(sid)
            for sid in missing_ids
            if int(prior_leg_streaks.get(str(sid), 0)) == 0
        }

        events = prev_state.setdefault(
            "tc_missing_return_events",
            [],
        )
        n_events_before = len(events)

        # Preserve the CURRENT production implementation exactly, including
        # transaction-cost cash accounting and terminal mark-to-cash logic.
        result = original_method(
            self,
            prev_state=prev_state,
            state_key=state_key,
            pretrade_state=pretrade_state,
            posttrade_weights=posttrade_weights,
            frame=frame,
            ret_name=ret_name,
            transaction_cost=transaction_cost,
        )

        assumed_by_sid = {
            str(sid): (
                float(first_missing_return)
                if str(sid) in first_missing_ids
                else 0.0
            )
            for sid in missing_ids
        }

        # Current-period P&L is calculated later from this same execution frame.
        for sid, assumed_ret in assumed_by_sid.items():
            if sid in frame.index:
                frame.loc[sid, ret_name] = float(assumed_ret)

        # Next-period drift uses the canonical saved realized-return vector.
        leg_store = self._tc_leg_state_store(prev_state)
        saved_leg = leg_store.get(str(state_key))
        if saved_leg is None:
            raise RuntimeError(
                "Missing-return sensitivity audit could not locate the "
                "canonical saved leg state after _tc_store_leg_state."
            )

        saved_realized = pd.Series(
            saved_leg["realized_returns"],
            dtype=float,
        ).copy()
        saved_realized.index = pd.Index(saved_realized.index).astype(str)

        for sid, assumed_ret in assumed_by_sid.items():
            if sid in saved_realized.index:
                saved_realized.loc[sid] = float(assumed_ret)

        saved_leg["realized_returns"] = saved_realized

        # Record the exact stress assumption alongside canonical diagnostics.
        new_events = events[n_events_before:]
        for event in new_events:
            sid = str(event.get("stock_id"))
            if sid in assumed_by_sid:
                event["audit_assumed_return"] = float(
                    assumed_by_sid[sid]
                )

        return result

    TCOptimizerMixin._tc_store_leg_state = stressed_store_leg_state

    try:
        yield
    finally:
        TCOptimizerMixin._tc_store_leg_state = original_method


# -----------------------------------------------------------------------------
# Run sensitivity grid
# -----------------------------------------------------------------------------

mr_rows = []
mr_returns = {}

for scenario, assumed_return in MR_SCENARIOS.items():

    print("\n" + "=" * 90)
    print(
        f"MISSING-RETURN SCENARIO: {scenario} "
        f"(first-gap return = {assumed_return:+.0%})"
    )

    scenario_dir = (
        MR_AUDIT_DIR / scenario
    )
    scenario_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    scenario_config = (
        FixedAUMCapacityConfig(
            output_dir=str(
                scenario_dir
            ),
            aum_grid=MR_AUDIT_AUMS,
            weight_type=(
                COST_AWARE_MARKOWITZ_WEIGHT_TYPE
            ),
            freq=FREQ,
            cut=CUT,
            delay=DELAY,
            start_year=ANALYSIS_START_YEAR,
            end_year=ANALYSIS_END_YEAR,
            eval_start_year=(
                ANALYSIS_START_YEAR
            ),
            country=COUNTRY,
            execution_days=(
                EXECUTION_DAYS
            ),
            require_tradability_screens=True,
            require_forecast_inputs=True,
            save_run_details=True,

            # Do not reuse canonical or prior stress runs.
            resume=False,

            verbose=False,
        )
    )

    with _missing_return_stress(
        assumed_return
    ):

        for aum in MR_AUDIT_AUMS:

            for model in MR_AUDIT_MODELS:

                print(
                    f"  {model:4s} "
                    f"@ ${aum / 1e6:,.0f}m"
                )

                run = run_one_fixed_aum(
                    optimizer_signals[
                        model
                    ],
                    model=model,
                    aum=float(aum),
                    config=scenario_config,
                    portfolio_kwargs=(
                        dict(
                            capacity_kwargs
                        )
                    ),
                )

                net = (
                    run.net_returns
                    .copy()
                    .sort_index()
                )

                mr_returns[
                    (
                        scenario,
                        model,
                        float(aum),
                    )
                ] = net

                row = dict(run.summary)

                row.update(
                    {
                        "scenario": scenario,
                        "assumed_first_missing_return": (
                            float(
                                assumed_return
                            )
                        ),
                        "model": model,
                        "aum_dollars": float(
                            aum
                        ),
                        "aum_millions": float(
                            aum
                        ) / 1e6,
                        "net_sharpe_recalculated": (
                            _annualized_sharpe(
                                net
                            )
                        ),
                    }
                )

                mr_rows.append(row)

                # Release heavy objects immediately.
                run.manager = None
                run.portfolio_returns = None
                run.tc_debug_df = None
                run.portfolio_debug_df = None
                run.missing_return_df = None

                del run
                gc.collect()


mr_summary = (
    pd.DataFrame(mr_rows)
    .sort_values(
        [
            "scenario",
            "aum_dollars",
            "model",
        ]
    )
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# HMM vs RF comparison
# -----------------------------------------------------------------------------

mr_sharpe_pivot = (
    mr_summary
    .pivot_table(
        index=[
            "scenario",
            "aum_millions",
        ],
        columns="model",
        values="net_sharpe",
        aggfunc="first",
    )
    .sort_index()
)

mr_sharpe_pivot[
    "HMM_minus_RF"
] = (
    mr_sharpe_pivot["HMM"]
    - mr_sharpe_pivot["RF"]
)


# -----------------------------------------------------------------------------
# Compare each stressed return path with the 0% baseline
# -----------------------------------------------------------------------------

mr_path_rows = []

for aum in MR_AUDIT_AUMS:

    for model in MR_AUDIT_MODELS:

        baseline = mr_returns[
            (
                "baseline_0pct",
                model,
                float(aum),
            )
        ]

        for scenario in MR_SCENARIOS:

            if scenario == "baseline_0pct":
                continue

            stressed = mr_returns[
                (
                    scenario,
                    model,
                    float(aum),
                )
            ]

            if not baseline.index.equals(
                stressed.index
            ):
                raise RuntimeError(
                    "Missing-return sensitivity "
                    "produced different evaluation "
                    "date indexes."
                )

            diff = (
                stressed
                - baseline
            )

            baseline_sr = (
                _annualized_sharpe(
                    baseline
                )
            )

            stressed_sr = (
                _annualized_sharpe(
                    stressed
                )
            )

            mr_path_rows.append(
                {
                    "scenario": scenario,
                    "model": model,
                    "aum_millions": (
                        float(aum) / 1e6
                    ),
                    "baseline_sharpe": (
                        baseline_sr
                    ),
                    "stressed_sharpe": (
                        stressed_sr
                    ),
                    "delta_sharpe": (
                        stressed_sr
                        - baseline_sr
                    ),
                    "mean_weekly_return_diff_bps": (
                        float(
                            diff.mean()
                        )
                        * 1e4
                    ),
                    "mean_abs_weekly_return_diff_bps": (
                        float(
                            diff.abs().mean()
                        )
                        * 1e4
                    ),
                    "max_abs_weekly_return_diff_bps": (
                        float(
                            diff.abs().max()
                        )
                        * 1e4
                    ),
                    "weekly_return_correlation": (
                        float(
                            baseline.corr(
                                stressed
                            )
                        )
                    ),
                }
            )

mr_path_comparison = pd.DataFrame(
    mr_path_rows
)


# -----------------------------------------------------------------------------
# Does each scenario preserve the baseline HMM/RF ordering?
# -----------------------------------------------------------------------------

mr_order_rows = []

baseline_delta = (
    mr_sharpe_pivot
    .xs(
        "baseline_0pct",
        level="scenario",
    )["HMM_minus_RF"]
)

for scenario in MR_SCENARIOS:

    scenario_delta = (
        mr_sharpe_pivot
        .xs(
            scenario,
            level="scenario",
        )["HMM_minus_RF"]
    )

    for aum_m in scenario_delta.index:

        base_value = float(
            baseline_delta.loc[
                aum_m
            ]
        )

        scenario_value = float(
            scenario_delta.loc[
                aum_m
            ]
        )

        mr_order_rows.append(
            {
                "scenario": scenario,
                "aum_millions": (
                    float(aum_m)
                ),
                "baseline_HMM_minus_RF": (
                    base_value
                ),
                "scenario_HMM_minus_RF": (
                    scenario_value
                ),
                "ordering_preserved": (
                    np.sign(base_value)
                    == np.sign(
                        scenario_value
                    )
                ),
            }
        )

mr_ordering = pd.DataFrame(
    mr_order_rows
)


# -----------------------------------------------------------------------------
# Save results
# -----------------------------------------------------------------------------

mr_summary.to_csv(
    MR_AUDIT_DIR
    / "missing_return_sensitivity_summary.csv",
    index=False,
)

mr_sharpe_pivot.to_csv(
    MR_AUDIT_DIR
    / "missing_return_sensitivity_sharpe_pivot.csv"
)

mr_path_comparison.to_csv(
    MR_AUDIT_DIR
    / "missing_return_sensitivity_path_comparison.csv",
    index=False,
)

mr_ordering.to_csv(
    MR_AUDIT_DIR
    / "missing_return_sensitivity_ordering.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Report
# -----------------------------------------------------------------------------

print("\nNET SHARPE BY SCENARIO")
display(
    mr_sharpe_pivot.style.format(
        {
            "RF": "{:.3f}",
            "HMM": "{:.3f}",
            "HMM_minus_RF": "{:+.3f}",
        }
    )
)

print(
    "\nRETURN-PATH CHANGE RELATIVE "
    "TO BASELINE"
)
display(
    mr_path_comparison.style.format(
        {
            "aum_millions": "${:,.0f}m",
            "baseline_sharpe": "{:.3f}",
            "stressed_sharpe": "{:.3f}",
            "delta_sharpe": "{:+.3f}",
            "mean_weekly_return_diff_bps": "{:+.2f}",
            "mean_abs_weekly_return_diff_bps": "{:.2f}",
            "max_abs_weekly_return_diff_bps": "{:.2f}",
            "weekly_return_correlation": "{:.6f}",
        }
    )
)

print("\nHMM/RF ORDERING CHECK")
display(
    mr_ordering.style.format(
        {
            "aum_millions": "${:,.0f}m",
            "baseline_HMM_minus_RF": "{:+.3f}",
            "scenario_HMM_minus_RF": "{:+.3f}",
        }
    )
)

all_orderings_preserved = bool(
    mr_ordering[
        "ordering_preserved"
    ].all()
)

print()
print(
    "Missing-return ranking robustness:",
    "PASS"
    if all_orderings_preserved
    else "REVIEW",
)

print(
    "Saved audit outputs to:",
    MR_AUDIT_DIR,
)

In [ ]:
# =============================================================================
# MINIMAL KERNEL-RESTART PREP FOR REDUCED CONVERGENCE AUDIT
# =============================================================================

import pandas as pd
import numpy as np


# -------------------------------------------------------------------------
# Reconstruct only the two signals needed for the convergence audit
# -------------------------------------------------------------------------

def _audit_signal_from_common(col):
    out = (
        common[
            [
                "Date",
                "StockID",
                col,
            ]
        ]
        .rename(
            columns={
                col: "up_prob"
            }
        )
        .copy()
    )

    out["Date"] = pd.to_datetime(
        out["Date"],
        errors="raise",
    ).dt.normalize()

    out["StockID"] = pd.to_numeric(
        out["StockID"],
        errors="raise",
    ).astype(int).astype(str)

    return out


optimizer_signals = {
    "RF": _audit_signal_from_common(
        "p_rf"
    ),
    "HMM": _audit_signal_from_common(
        "p_hmm"
    ),
}


# -------------------------------------------------------------------------
# Reconstruct the same transaction-cost portfolio kwargs used in research
# -------------------------------------------------------------------------

capacity_kwargs = dict(
    SCREENED_KWARGS
)

for key in (
    "signal_df",
    "freq",
    "portfolio_dir",
    "start_year",
    "end_year",
    "eval_start_year",
    "country",
    "delay_list",
    "load_signal",
    "tc_aum_dollars",
    "tc_execution_days",
):
    capacity_kwargs.pop(
        key,
        None,
    )

capacity_kwargs.update(
    {
        "tc_enable": True,
        "tc_use_nonlinear": True,
        "tc_use_forecast_inputs": True,
        "tc_forecast_path": str(
            TC_FORECAST_PATH
        ),
        "tradability_screens": True,
        "include_price_adv": True,
        "tc_missing_return_max_carry_weeks": (
            MISSING_RETURN_MAX_CARRY_PERIODS
        ),
        "tc_missing_return_warn_weight": (
            MISSING_RETURN_WARN_WEIGHT
        ),
    }
)


# -------------------------------------------------------------------------
# Recover the canonical fixed-AUM summary for the audit's validation check
# -------------------------------------------------------------------------

CAPACITY_DIR = (
    OUTPUT_DIR
    / "fixed_aum_capacity"
)

capacity_summary_path = (
    CAPACITY_DIR
    / "fixed_aum_capacity_summary.csv"
)

if not capacity_summary_path.exists():
    raise FileNotFoundError(
        f"Saved capacity summary missing: "
        f"{capacity_summary_path}"
    )

capacity_summary = pd.read_csv(
    capacity_summary_path
)


# -------------------------------------------------------------------------
# Sanity checks
# -------------------------------------------------------------------------

if not TC_FORECAST_PATH.exists():
    raise FileNotFoundError(
        f"TC forecast file missing: "
        f"{TC_FORECAST_PATH}"
    )

for name, sig in optimizer_signals.items():
    print(
        f"{name}: {len(sig):,} rows"
    )

print()
print(
    "Capacity summary:",
    capacity_summary.shape,
)

print(
    "Minimal convergence-audit recovery: PASS"
)

In [ ]:
# =============================================================================
# ROBUSTNESS AUDIT B — REDUCED NONLINEAR OPTIMIZER CONVERGENCE AUDIT
# =============================================================================
#
# Purpose
# -------
# Diagnose whether the repeated max_iters warnings are economically material
# without rerunning the entire 3-AUM x 3-iteration convergence grid.
#
# This reduced audit performs only TWO full portfolio reruns:
#
#     RF  @ $100m using the current 2,000-iteration production setting
#     HMM @ $100m using the current 2,000-iteration production setting
#
# During those runs it captures a small random sample of exact optimization
# problems that either:
#
#     - return solver_converged=False, or
#     - use the fallback solver.
#
# Each sampled problem is then re-solved directly at:
#
#     1x =  2,000 iterations  (already observed in the full baseline run)
#     2x =  4,000 iterations
#     5x = 10,000 iterations
#
# The exact-problem comparison holds alpha, covariance, previous weights, ADV,
# volatility, spreads, AUM, constraints, and target sum fixed. Only max_iters
# changes. This is the clean numerical test.
#
# If the exact 1x and 5x solutions are effectively identical, there is no need
# to pay for full 4,000/10,000-iteration portfolio-path reruns.
# =============================================================================

from contextlib import contextmanager
from dataclasses import replace
from pathlib import Path
import copy
import gc
import warnings

import numpy as np
import pandas as pd

import Scripts.Portfolio.cost_aware_optimizer as cao

from Scripts.Portfolio.capacity_runner import (
    FixedAUMCapacityConfig,
    run_one_fixed_aum,
)
from Scripts.Portfolio.strategies import MWHPlusLowNegTCStrategy


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

CONV_AUDIT_MODELS = ("RF", "HMM")
CONV_AUDIT_AUM = 100_000_000.0

# Ten problematic solver calls per model is enough for a targeted first-pass
# audit while keeping the expensive 10,000-iteration direct re-solves bounded.
CONV_SAMPLE_PER_MODEL = 10
CONV_RANDOM_SEED = 20260830

BASE_MAX_ITERS = int(cao.CostAwareParams().max_iters)
CONV_MULTIPLIERS = (1, 2, 5)

CONV_AUDIT_DIR = (
    OUTPUT_DIR
    / "robustness_optimizer_convergence_reduced"
)
CONV_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Reduced convergence audit:",
    f"2 full $100m portfolio runs + "
    f"up to {2 * CONV_SAMPLE_PER_MODEL} exact problem samples.",
)
print(
    "Iteration levels:",
    {
        m: BASE_MAX_ITERS * m
        for m in CONV_MULTIPLIERS
    },
)


# -----------------------------------------------------------------------------
# Objective reconstruction
# -----------------------------------------------------------------------------

def _conv_objective(
    *,
    weights,
    alpha,
    Sigma,
    w_prev,
    adv,
    sigma,
    aum,
    spreads_oneway,
    spreads_bps,
    params,
):
    """
    Reconstruct the exact smooth minimization objective used by the
    alpha-driven cost-aware optimizer. Lower is better.
    """

    idx = pd.Index(
        alpha.index.astype(str)
    )

    w = (
        pd.to_numeric(
            weights.reindex(idx),
            errors="coerce",
        )
        .to_numpy(dtype=float)
    )

    alpha_vec = (
        pd.to_numeric(
            alpha.reindex(idx),
            errors="coerce",
        )
        .to_numpy(dtype=float)
    )

    w_prev_vec = (
        pd.to_numeric(
            w_prev.reindex(idx),
            errors="coerce",
        )
        .to_numpy(dtype=float)
    )

    adv_vec = cao._resolve_adv_vector(
        idx,
        adv,
    )

    sigma_vec = cao._resolve_sigma_vector(
        idx,
        sigma,
    )

    spreads_vec = cao._resolve_spreads_fraction(
        idx,
        spreads_oneway=spreads_oneway,
        spreads_bps=spreads_bps,
        default_bps=params.spread_bps_default,
    )

    impact_coeff_vec = cao._impact_coeff_vector(
        sigma_vec=sigma_vec,
        adv_vec=adv_vec,
        aum=float(aum),
        impact_kappa=float(params.impact_kappa),
        execution_days=params.resolved_execution_days(),
    )

    Sigma_arr = np.asarray(
        Sigma,
        dtype=float,
    )

    Sigma_arr = (
        0.5
        * (
            Sigma_arr
            + Sigma_arr.T
        )
    )

    G = (
        2.0
        * float(params.lambda_risk)
        * Sigma_arr
    )

    return float(
        cao._smooth_objective(
            w=w,
            w_prev=w_prev_vec,
            alpha_vec=alpha_vec,
            G=G,
            spreads_vec=spreads_vec,
            impact_coeff_vec=impact_coeff_vec,
            params=params,
        )
    )


def _copy_optional_series(x):
    if x is None:
        return None

    if isinstance(x, pd.Series):
        return x.copy()

    return copy.deepcopy(x)


# -----------------------------------------------------------------------------
# Instrumentation
# -----------------------------------------------------------------------------

_rng = np.random.default_rng(
    CONV_RANDOM_SEED
)

_active = {
    "model": None,
    "date": None,
    "solve_in_date": 0,
}

solve_log = []

# One bounded reservoir per model.
problem_reservoir = {
    model: {
        "seen": 0,
        "items": [],
    }
    for model in CONV_AUDIT_MODELS
}


def _capture_problem_if_selected(
    *,
    model,
    problem_factory,
):
    """
    Reservoir-sample problematic calls without copying every covariance matrix.
    """

    bucket = problem_reservoir[
        model
    ]

    bucket["seen"] += 1
    seen = int(
        bucket["seen"]
    )

    items = bucket[
        "items"
    ]

    if len(items) < CONV_SAMPLE_PER_MODEL:
        items.append(
            problem_factory()
        )
        return

    j = int(
        _rng.integers(
            0,
            seen,
        )
    )

    if j < CONV_SAMPLE_PER_MODEL:
        items[j] = problem_factory()


_original_compute_for_date = (
    MWHPlusLowNegTCStrategy.compute_for_date
)

_original_alpha_solver = (
    cao.solve_cost_aware_nonlinear_pg
)


def _audit_compute_for_date(
    self,
    df,
    date,
    cut,
    ret_name,
    prev_state,
):
    previous_date = _active[
        "date"
    ]
    previous_counter = _active[
        "solve_in_date"
    ]

    _active["date"] = pd.Timestamp(
        date
    )
    _active["solve_in_date"] = 0

    try:
        return _original_compute_for_date(
            self,
            df,
            date,
            cut,
            ret_name,
            prev_state,
        )
    finally:
        _active["date"] = (
            previous_date
        )
        _active["solve_in_date"] = (
            previous_counter
        )


def _audit_alpha_solver(
    *,
    alpha,
    Sigma,
    w_prev,
    adv,
    sigma,
    aum,
    spreads_oneway=None,
    spreads_bps=None,
    params=None,
    desired_target_sum=None,
):
    if params is None:
        params = cao.CostAwareParams()

    out = _original_alpha_solver(
        alpha=alpha,
        Sigma=Sigma,
        w_prev=w_prev,
        adv=adv,
        sigma=sigma,
        aum=aum,
        spreads_oneway=spreads_oneway,
        spreads_bps=spreads_bps,
        params=params,
        desired_target_sum=desired_target_sum,
    )

    _active[
        "solve_in_date"
    ] += 1

    solve_in_date = int(
        _active[
            "solve_in_date"
        ]
    )

    objective = _conv_objective(
        weights=out,
        alpha=alpha,
        Sigma=Sigma,
        w_prev=w_prev,
        adv=adv,
        sigma=sigma,
        aum=aum,
        spreads_oneway=spreads_oneway,
        spreads_bps=spreads_bps,
        params=params,
    )

    converged = bool(
        out.attrs.get(
            "solver_converged",
            False,
        )
    )

    fallback = bool(
        out.attrs.get(
            "solver_fallback_used",
            False,
        )
    )

    method = str(
        out.attrs.get(
            "solver_method",
            "unknown",
        )
    )

    model = str(
        _active["model"]
    )

    solve_log.append(
        {
            "model": model,
            "aum_millions": (
                float(aum) / 1e6
            ),
            "date": (
                pd.Timestamp(
                    _active["date"]
                )
                if _active["date"]
                is not None
                else pd.NaT
            ),
            "solve_in_date": solve_in_date,
            "n_assets": int(
                len(out)
            ),
            "max_iters": int(
                params.max_iters
            ),
            "objective": objective,
            "solver_method": method,
            "solver_converged": converged,
            "solver_fallback_used": fallback,
        }
    )

    if (
        model in problem_reservoir
        and (
            (not converged)
            or fallback
        )
    ):
        def _problem_factory():
            return {
                "model": model,
                "date": (
                    pd.Timestamp(
                        _active["date"]
                    )
                    if _active["date"]
                    is not None
                    else pd.NaT
                ),
                "solve_in_date": solve_in_date,
                "alpha": alpha.copy(),
                "Sigma": np.asarray(
                    Sigma,
                    dtype=float,
                ).copy(),
                "w_prev": w_prev.copy(),
                "adv": adv.copy(),
                "sigma": sigma.copy(),
                "spreads_oneway": (
                    _copy_optional_series(
                        spreads_oneway
                    )
                ),
                "spreads_bps": (
                    _copy_optional_series(
                        spreads_bps
                    )
                ),
                "aum": float(aum),
                "params": copy.deepcopy(
                    params
                ),
                "desired_target_sum": (
                    desired_target_sum
                ),
                "baseline_weights": (
                    out.copy()
                ),
                "baseline_attrs": dict(
                    out.attrs
                ),
                "baseline_objective": (
                    objective
                ),
            }

        _capture_problem_if_selected(
            model=model,
            problem_factory=_problem_factory,
        )

    return out


@contextmanager
def _install_convergence_instrumentation():
    current_compute = (
        MWHPlusLowNegTCStrategy.compute_for_date
    )
    current_solver = (
        cao.solve_cost_aware_nonlinear_pg
    )

    MWHPlusLowNegTCStrategy.compute_for_date = (
        _audit_compute_for_date
    )
    cao.solve_cost_aware_nonlinear_pg = (
        _audit_alpha_solver
    )

    try:
        yield
    finally:
        MWHPlusLowNegTCStrategy.compute_for_date = (
            current_compute
        )
        cao.solve_cost_aware_nonlinear_pg = (
            current_solver
        )


# -----------------------------------------------------------------------------
# Part A — TWO baseline full-path reruns at the production iteration setting
# -----------------------------------------------------------------------------

baseline_dir = (
    CONV_AUDIT_DIR
    / "baseline_1x"
)
baseline_dir.mkdir(
    parents=True,
    exist_ok=True,
)

baseline_config = FixedAUMCapacityConfig(
    output_dir=str(
        baseline_dir
    ),
    aum_grid=(
        CONV_AUDIT_AUM,
    ),
    weight_type=(
        COST_AWARE_MARKOWITZ_WEIGHT_TYPE
    ),
    freq=FREQ,
    cut=CUT,
    delay=DELAY,
    start_year=ANALYSIS_START_YEAR,
    end_year=ANALYSIS_END_YEAR,
    eval_start_year=ANALYSIS_START_YEAR,
    country=COUNTRY,
    execution_days=EXECUTION_DAYS,
    require_tradability_screens=True,
    require_forecast_inputs=True,

    # No need to write every large run-detail artifact for this diagnostic.
    save_run_details=False,

    # The run must execute so the actual optimization problems can be captured.
    resume=False,

    verbose=False,
)

baseline_rows = []

with _install_convergence_instrumentation():

    for model in CONV_AUDIT_MODELS:

        print(
            "\n" + "=" * 80
        )
        print(
            f"BASELINE CONVERGENCE CAPTURE: "
            f"{model} @ $100m"
        )

        _active[
            "model"
        ] = model
        _active[
            "date"
        ] = None
        _active[
            "solve_in_date"
        ] = 0

        log_start = len(
            solve_log
        )

        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=(
                    r".*reached max_iters; "
                    r"returning best feasible iterate found\."
                ),
                category=RuntimeWarning,
            )

            run = run_one_fixed_aum(
                optimizer_signals[
                    model
                ],
                model=model,
                aum=(
                    CONV_AUDIT_AUM
                ),
                config=baseline_config,
                portfolio_kwargs=dict(
                    capacity_kwargs
                ),
            )

        model_log = solve_log[
            log_start:
        ]

        n_solves = len(
            model_log
        )

        n_nonconverged = sum(
            not bool(
                row[
                    "solver_converged"
                ]
            )
            for row in model_log
        )

        n_fallback = sum(
            bool(
                row[
                    "solver_fallback_used"
                ]
            )
            for row in model_log
        )

        baseline_rows.append(
            {
                "model": model,
                "aum_millions": 100.0,
                "net_sharpe": float(
                    run.summary[
                        "net_sharpe"
                    ]
                ),
                "optimizer_solve_calls": (
                    n_solves
                ),
                "optimizer_nonconverged": (
                    n_nonconverged
                ),
                "optimizer_nonconverged_pct": (
                    n_nonconverged
                    / n_solves
                    if n_solves
                    else np.nan
                ),
                "optimizer_fallback_used": (
                    n_fallback
                ),
                "optimizer_fallback_pct": (
                    n_fallback
                    / n_solves
                    if n_solves
                    else np.nan
                ),
                "problematic_calls_seen": int(
                    problem_reservoir[
                        model
                    ][
                        "seen"
                    ]
                ),
                "problematic_calls_sampled": int(
                    len(
                        problem_reservoir[
                            model
                        ][
                            "items"
                        ]
                    )
                ),
            }
        )

        run.manager = None
        run.portfolio_returns = None
        run.tc_debug_df = None
        run.portfolio_debug_df = None
        run.missing_return_df = None

        del run
        gc.collect()


baseline_summary = pd.DataFrame(
    baseline_rows
)


# -----------------------------------------------------------------------------
# Confirm that instrumentation did not alter the canonical $100m result
# -----------------------------------------------------------------------------

if "capacity_summary" in globals():
    canonical_capacity = (
        capacity_summary.copy()
    )
else:
    canonical_capacity = pd.read_csv(
        CAPACITY_DIR
        / "fixed_aum_capacity_summary.csv"
    )

canonical_100m = (
    canonical_capacity[
        np.isclose(
            pd.to_numeric(
                canonical_capacity[
                    "aum_millions"
                ],
                errors="coerce",
            ),
            100.0,
        )
        & canonical_capacity[
            "model"
        ].astype(str).isin(
            CONV_AUDIT_MODELS
        )
    ][
        [
            "model",
            "net_sharpe",
        ]
    ]
    .drop_duplicates(
        "model"
    )
    .rename(
        columns={
            "net_sharpe": (
                "canonical_net_sharpe"
            )
        }
    )
)

baseline_summary = (
    baseline_summary
    .merge(
        canonical_100m,
        on="model",
        how="left",
        validate="one_to_one",
    )
)

baseline_summary[
    "sharpe_diff_vs_canonical"
] = (
    baseline_summary[
        "net_sharpe"
    ]
    - baseline_summary[
        "canonical_net_sharpe"
    ]
)

if (
    baseline_summary[
        "canonical_net_sharpe"
    ].isna().any()
):
    raise RuntimeError(
        "Could not locate both canonical $100m RF/HMM "
        "Sharpe values for the convergence audit."
    )

if (
    baseline_summary[
        "sharpe_diff_vs_canonical"
    ].abs().max()
    > 1e-8
):
    raise RuntimeError(
        "Instrumented baseline run does not reproduce "
        "the canonical $100m portfolio Sharpe."
    )


# -----------------------------------------------------------------------------
# Part B — exact same-problem re-solves at 2x and 5x
# -----------------------------------------------------------------------------

problem_rows = []
problem_id = 0

for model in CONV_AUDIT_MODELS:

    bucket = problem_reservoir[
        model
    ]

    print(
        f"\nExact-problem sample: {model} | "
        f"sampled {len(bucket['items'])} "
        f"of {bucket['seen']} problematic calls"
    )

    for problem in bucket[
        "items"
    ]:

        problem_id += 1

        baseline_w = problem[
            "baseline_weights"
        ].copy()

        baseline_attrs = problem[
            "baseline_attrs"
        ]

        problem_rows.append(
            {
                "problem_id": problem_id,
                "model": model,
                "date": problem[
                    "date"
                ],
                "solve_in_date": problem[
                    "solve_in_date"
                ],
                "multiplier": 1,
                "max_iters": (
                    BASE_MAX_ITERS
                ),
                "n_assets": len(
                    baseline_w
                ),
                "objective": float(
                    problem[
                        "baseline_objective"
                    ]
                ),
                "solver_method": str(
                    baseline_attrs.get(
                        "solver_method",
                        "unknown",
                    )
                ),
                "solver_converged": bool(
                    baseline_attrs.get(
                        "solver_converged",
                        False,
                    )
                ),
                "solver_fallback_used": bool(
                    baseline_attrs.get(
                        "solver_fallback_used",
                        False,
                    )
                ),
                "_weights": baseline_w,
            }
        )

        for multiplier in (
            2,
            5,
        ):

            params = replace(
                problem[
                    "params"
                ],
                max_iters=(
                    BASE_MAX_ITERS
                    * multiplier
                ),
            )

            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message=(
                        r".*reached max_iters; "
                        r"returning best feasible iterate found\."
                    ),
                    category=RuntimeWarning,
                )

                w = (
                    _original_alpha_solver(
                        alpha=problem[
                            "alpha"
                        ],
                        Sigma=problem[
                            "Sigma"
                        ],
                        w_prev=problem[
                            "w_prev"
                        ],
                        adv=problem[
                            "adv"
                        ],
                        sigma=problem[
                            "sigma"
                        ],
                        aum=problem[
                            "aum"
                        ],
                        spreads_oneway=(
                            problem[
                                "spreads_oneway"
                            ]
                        ),
                        spreads_bps=(
                            problem[
                                "spreads_bps"
                            ]
                        ),
                        params=params,
                        desired_target_sum=(
                            problem[
                                "desired_target_sum"
                            ]
                        ),
                    )
                )

            objective = _conv_objective(
                weights=w,
                alpha=problem[
                    "alpha"
                ],
                Sigma=problem[
                    "Sigma"
                ],
                w_prev=problem[
                    "w_prev"
                ],
                adv=problem[
                    "adv"
                ],
                sigma=problem[
                    "sigma"
                ],
                aum=problem[
                    "aum"
                ],
                spreads_oneway=(
                    problem[
                        "spreads_oneway"
                    ]
                ),
                spreads_bps=(
                    problem[
                        "spreads_bps"
                    ]
                ),
                params=params,
            )

            problem_rows.append(
                {
                    "problem_id": (
                        problem_id
                    ),
                    "model": model,
                    "date": problem[
                        "date"
                    ],
                    "solve_in_date": (
                        problem[
                            "solve_in_date"
                        ]
                    ),
                    "multiplier": (
                        multiplier
                    ),
                    "max_iters": (
                        BASE_MAX_ITERS
                        * multiplier
                    ),
                    "n_assets": len(
                        w
                    ),
                    "objective": (
                        objective
                    ),
                    "solver_method": str(
                        w.attrs.get(
                            "solver_method",
                            "unknown",
                        )
                    ),
                    "solver_converged": bool(
                        w.attrs.get(
                            "solver_converged",
                            False,
                        )
                    ),
                    "solver_fallback_used": bool(
                        w.attrs.get(
                            "solver_fallback_used",
                            False,
                        )
                    ),
                    "_weights": (
                        w.copy()
                    ),
                }
            )


problem_results = pd.DataFrame(
    problem_rows
)


# -----------------------------------------------------------------------------
# Compare 1x and 2x directly with the 5x solution
# -----------------------------------------------------------------------------

comparison_rows = []

for problem_id, group in (
    problem_results
    .groupby(
        "problem_id",
        sort=True,
    )
):

    group = group.set_index(
        "multiplier"
    )

    if 5 not in group.index:
        continue

    ref = group.loc[
        5
    ]

    w_ref = ref[
        "_weights"
    ]

    obj_ref = float(
        ref[
            "objective"
        ]
    )

    for multiplier in (
        1,
        2,
        5,
    ):

        row = group.loc[
            multiplier
        ]

        w = row[
            "_weights"
        ]

        idx = w.index.union(
            w_ref.index
        )

        diff = (
            w.reindex(
                idx
            ).fillna(0.0)
            - w_ref.reindex(
                idx
            ).fillna(0.0)
        )

        obj = float(
            row[
                "objective"
            ]
        )

        obj_gap = (
            obj
            - obj_ref
        )

        comparison_rows.append(
            {
                "problem_id": (
                    problem_id
                ),
                "model": row[
                    "model"
                ],
                "date": row[
                    "date"
                ],
                "solve_in_date": (
                    row[
                        "solve_in_date"
                    ]
                ),
                "multiplier": (
                    multiplier
                ),
                "max_iters": row[
                    "max_iters"
                ],
                "objective_gap_vs_5x": (
                    obj_gap
                ),
                "scaled_objective_gap_vs_5x": (
                    obj_gap
                    / (
                        1.0
                        + abs(
                            obj_ref
                        )
                    )
                ),
                "l1_weight_diff_vs_5x": float(
                    diff.abs().sum()
                ),
                "max_abs_weight_diff_vs_5x": float(
                    diff.abs().max()
                ),
                "solver_converged": bool(
                    row[
                        "solver_converged"
                    ]
                ),
                "solver_fallback_used": bool(
                    row[
                        "solver_fallback_used"
                    ]
                ),
            }
        )


problem_comparison = pd.DataFrame(
    comparison_rows
)

baseline_problem_diff = (
    problem_comparison[
        problem_comparison[
            "multiplier"
        ] == 1
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Model-level summary
# -----------------------------------------------------------------------------

model_rows = []

for model in CONV_AUDIT_MODELS:

    sub = baseline_problem_diff[
        baseline_problem_diff[
            "model"
        ] == model
    ]

    five = problem_comparison[
        (
            problem_comparison[
                "model"
            ] == model
        )
        & (
            problem_comparison[
                "multiplier"
            ] == 5
        )
    ]

    model_rows.append(
        {
            "model": model,
            "sampled_problems": int(
                len(sub)
            ),
            "median_abs_objective_gap_1x_vs_5x": float(
                sub[
                    "scaled_objective_gap_vs_5x"
                ].abs().median()
            )
            if len(sub)
            else np.nan,
            "max_abs_objective_gap_1x_vs_5x": float(
                sub[
                    "scaled_objective_gap_vs_5x"
                ].abs().max()
            )
            if len(sub)
            else np.nan,
            "median_l1_weight_diff_1x_vs_5x": float(
                sub[
                    "l1_weight_diff_vs_5x"
                ].median()
            )
            if len(sub)
            else np.nan,
            "max_l1_weight_diff_1x_vs_5x": float(
                sub[
                    "l1_weight_diff_vs_5x"
                ].max()
            )
            if len(sub)
            else np.nan,
            "median_max_name_weight_diff_1x_vs_5x": float(
                sub[
                    "max_abs_weight_diff_vs_5x"
                ].median()
            )
            if len(sub)
            else np.nan,
            "max_name_weight_diff_1x_vs_5x": float(
                sub[
                    "max_abs_weight_diff_vs_5x"
                ].max()
            )
            if len(sub)
            else np.nan,
            "still_nonconverged_at_5x": int(
                (
                    ~five[
                        "solver_converged"
                    ]
                ).sum()
            ),
        }
    )


model_summary = pd.DataFrame(
    model_rows
)


# -----------------------------------------------------------------------------
# Materiality flags
# -----------------------------------------------------------------------------
#
# These are diagnostic tolerances, not hypothesis tests.
#
# 10 bp maximum per-name weight movement is deliberately tight. If the exact
# solver problems pass this test, the max_iters warning is unlikely to be
# economically important.
# -----------------------------------------------------------------------------

MAX_NAME_WEIGHT_DIFF_TOL = 0.001
SCALED_OBJECTIVE_GAP_TOL = 1e-5

if len(
    baseline_problem_diff
):
    exact_problem_stable = bool(
        (
            baseline_problem_diff[
                "max_abs_weight_diff_vs_5x"
            ]
            <= MAX_NAME_WEIGHT_DIFF_TOL
        ).all()
        and
        (
            baseline_problem_diff[
                "scaled_objective_gap_vs_5x"
            ].abs()
            <= SCALED_OBJECTIVE_GAP_TOL
        ).all()
    )
else:
    exact_problem_stable = True


# -----------------------------------------------------------------------------
# Save compact outputs
# -----------------------------------------------------------------------------

pd.DataFrame(
    solve_log
).to_csv(
    CONV_AUDIT_DIR
    / "baseline_optimizer_solve_log.csv",
    index=False,
)

baseline_summary.to_csv(
    CONV_AUDIT_DIR
    / "baseline_100m_summary.csv",
    index=False,
)

problem_results.drop(
    columns=[
        "_weights"
    ],
    errors="ignore",
).to_csv(
    CONV_AUDIT_DIR
    / "exact_problem_results.csv",
    index=False,
)

problem_comparison.to_csv(
    CONV_AUDIT_DIR
    / "exact_problem_comparison.csv",
    index=False,
)

model_summary.to_csv(
    CONV_AUDIT_DIR
    / "exact_problem_model_summary.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print(
    "\nBASELINE $100m SOLVER DIAGNOSTICS"
)

display(
    baseline_summary.style.format(
        {
            "aum_millions": "${:,.0f}m",
            "net_sharpe": "{:.3f}",
            "canonical_net_sharpe": "{:.3f}",
            "sharpe_diff_vs_canonical": "{:+.3e}",
            "optimizer_nonconverged_pct": "{:.1%}",
            "optimizer_fallback_pct": "{:.1%}",
        }
    )
)


print(
    "\nEXACT-PROBLEM MODEL SUMMARY "
    "(1x compared with 5x)"
)

display(
    model_summary.style.format(
        {
            "median_abs_objective_gap_1x_vs_5x": "{:.3e}",
            "max_abs_objective_gap_1x_vs_5x": "{:.3e}",
            "median_l1_weight_diff_1x_vs_5x": "{:.6f}",
            "max_l1_weight_diff_1x_vs_5x": "{:.6f}",
            "median_max_name_weight_diff_1x_vs_5x": "{:.6f}",
            "max_name_weight_diff_1x_vs_5x": "{:.6f}",
        }
    )
)


print(
    "\nWORST SAMPLED 1x VS 5x SOLUTIONS"
)

display(
    baseline_problem_diff
    .sort_values(
        "max_abs_weight_diff_vs_5x",
        ascending=False,
    )
    .head(
        20
    )
    .style.format(
        {
            "objective_gap_vs_5x": "{:+.3e}",
            "scaled_objective_gap_vs_5x": "{:+.3e}",
            "l1_weight_diff_vs_5x": "{:.6f}",
            "max_abs_weight_diff_vs_5x": "{:.6f}",
        }
    )
)


print()
print(
    "Reduced convergence audit:",
    (
        "PASS — exact 2,000-iteration solutions are "
        "numerically stable relative to 10,000 iterations."
        if exact_problem_stable
        else
        "REVIEW — at least one sampled problem changes "
        "materially at 10,000 iterations."
    ),
)

if exact_problem_stable:
    print(
        "No full 4,000/10,000-iteration portfolio-path "
        "reruns are indicated by this first-pass audit."
    )
else:
    print(
        "Next step: rerun only RF and HMM at $100m with "
        "10,000 iterations and compare full return paths."
    )

print(
    "\nSaved reduced convergence audit to:",
    CONV_AUDIT_DIR,
)


In [ ]:
import os
# =========================================================================
# EXPECTED-RETURN FUSION PERSISTENCE DIAGNOSTIC
# Standalone kernel-restart cell.
#
# DOES NOT RETRAIN ANY MODEL.
#
# Loads the already-saved ExpectedReturnFusion predictions, reconstructs
# the same DataAssembler tradability screen used in the research persistence
# diagnostics, then calculates:
#   - week-to-week rank persistence
#   - long-tail retention
#   - short-tail retention
#   - long/short target turnover
#   - long/short holding-spell length
# =========================================================================

from pathlib import Path
import sys
import json
import inspect
import re

import numpy as np
import pandas as pd

from IPython.display import display


# -------------------------------------------------------------------------
# 1. Locate project and load research configuration
# -------------------------------------------------------------------------



def find_project_root():
    configured = os.environ.get("FUSION_RESEARCH_ROOT")
    if not configured:
        raise RuntimeError(
            "Set FUSION_RESEARCH_ROOT to the original licensed research environment. "
            "The public package does not include Scripts.Data or Scripts.Portfolio."
        )
    root = Path(configured).expanduser().resolve()
    if not (root / "Scripts").is_dir():
        raise FileNotFoundError("FUSION_RESEARCH_ROOT must contain the private Scripts directory.")
    return root


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


ANNUAL_ROOT = (
    PROJECT_ROOT
    / "WORK_SPACE"
    / "expanding_window_hmm_from_saved_cnn"
)

ROOT_MANIFEST = (
    ANNUAL_ROOT
    / "annual_expanding_window_run_manifest.json"
)

if not os.environ.get("FUSION_ANALYSIS_DIR"):
    raise RuntimeError("Set FUSION_ANALYSIS_DIR to the analysis output directory.")
OUTPUT_DIR = Path(os.environ["FUSION_ANALYSIS_DIR"]).expanduser().resolve()

if not ROOT_MANIFEST.exists():
    raise FileNotFoundError(
        f"Run manifest not found: {ROOT_MANIFEST}"
    )

with ROOT_MANIFEST.open("r", encoding="utf-8") as f:
    MANIFEST = json.load(f)


ANALYSIS_START_YEAR = 2001
ANALYSIS_END_YEAR = int(
    MANIFEST.get("last_pred_year", 2024)
)

FREQ = str(
    MANIFEST.get("freq", "week")
)

COUNTRY = str(
    MANIFEST.get("country", "USA")
)

CUT = int(
    MANIFEST.get("cut", 10)
)

DELAY = int(
    (MANIFEST.get("delay_list") or [0])[0]
)


SCREENED_KWARGS = dict(
    MANIFEST.get("screened_portfolio_kwargs")
    or {
        "min_price": 1.0,
        "min_adv_dollar": 10_000_000,
        "tradability_screens": True,
        "include_price_adv": True,
    }
)


# -------------------------------------------------------------------------
# 2. Import portfolio screening/tail utilities
# -------------------------------------------------------------------------

from Scripts.Portfolio.data_assembler import DataAssembler
from Scripts.Portfolio.portfolio import PortfolioManager
from Scripts.Portfolio.portfolio_calculations import PortfolioMath


_pm_signature = inspect.signature(
    PortfolioManager.__init__
)

PM_MW_WINDOW = int(
    _pm_signature.parameters["mw_window"].default
)

PM_MW_MIN_PERIODS = int(
    _pm_signature.parameters["mw_min_periods"].default
)


# -------------------------------------------------------------------------
# 3. Load SAVED ExpectedReturnFusion forecasts
# -------------------------------------------------------------------------

expected_return_path = (
    OUTPUT_DIR
    / "expected_return_fusion_5d_predictions.parquet"
)

if not expected_return_path.exists():
    raise FileNotFoundError(
        "Saved ExpectedReturnFusion predictions were not found:\n"
        f"{expected_return_path}\n\n"
        "This diagnostic requires the completed research saved artifact, "
        "but does not require model retraining."
    )


expected = pd.read_parquet(
    expected_return_path
)

required_cols = {
    "Date",
    "StockID",
    "mu_hat_5d",
}

missing = required_cols.difference(
    expected.columns
)

if missing:
    raise KeyError(
        "Expected-return prediction file is missing columns: "
        f"{sorted(missing)}"
    )


expected["Date"] = pd.to_datetime(
    expected["Date"],
    errors="raise",
).dt.normalize()

expected["StockID"] = (
    pd.to_numeric(
        expected["StockID"],
        errors="raise",
    )
    .astype(int)
    .astype(str)
)

expected["mu_hat_5d"] = pd.to_numeric(
    expected["mu_hat_5d"],
    errors="raise",
)


# Restrict to the paper's evaluation period.
expected = (
    expected.loc[
        expected["Date"].dt.year.between(
            ANALYSIS_START_YEAR,
            ANALYSIS_END_YEAR,
        ),
        [
            "Date",
            "StockID",
            "mu_hat_5d",
        ],
    ]
    .dropna()
    .drop_duplicates(
        ["Date", "StockID"],
        keep="last",
    )
    .sort_values(
        ["Date", "StockID"]
    )
    .reset_index(drop=True)
)


if expected.duplicated(
    ["Date", "StockID"]
).any():
    raise AssertionError(
        "ExpectedReturnFusion contains duplicate stock-date keys."
    )


print(
    "Loaded saved ExpectedReturnFusion:",
    f"{len(expected):,}",
    "rows |",
    f"{expected['Date'].nunique():,}",
    "dates",
)


# -------------------------------------------------------------------------
# 4. Reconstruct EXACT research tradability screen
#
# The research screen is independent of the score once all models use the same
# strict common stock-date keys. ExpectedReturnFusion was already verified
# against that strict common key set in research, so it can itself be used as
# the neutral seed signal here.
# -------------------------------------------------------------------------

screen_seed_signal = (
    expected[
        [
            "Date",
            "StockID",
            "mu_hat_5d",
        ]
    ]
    .rename(
        columns={
            "mu_hat_5d": "up_prob"
        }
    )
    .copy()
)


min_adv = SCREENED_KWARGS.get(
    "min_adv_dollar",
    None,
)

screen_assembler = DataAssembler(
    freq=FREQ,
    start_year=ANALYSIS_START_YEAR,
    end_year=ANALYSIS_END_YEAR,
    country=COUNTRY,
    delay_list=[DELAY],
    tradability_screens=True,
    min_price=float(
        SCREENED_KWARGS.get(
            "min_price",
            1.0,
        )
    ),
    min_adv_dollar=(
        None
        if min_adv is None
        else float(min_adv)
    ),
    mw_window=PM_MW_WINDOW,
    mw_min_periods=PM_MW_MIN_PERIODS,
    include_price_adv=True,
    verbose=False,
)


prepared_screen = (
    screen_assembler
    .prepare(screen_seed_signal)
    .reset_index()
)

screen_ret_col = (
    screen_assembler.no_delay_ret_name
)

required_screen_cols = [
    "Date",
    "StockID",
    screen_ret_col,
]

missing_screen = [
    c
    for c in required_screen_cols
    if c not in prepared_screen.columns
]

if missing_screen:
    raise KeyError(
        "DataAssembler output missing required columns: "
        f"{missing_screen}"
    )


prepared_screen["Date"] = pd.to_datetime(
    prepared_screen["Date"],
    errors="coerce",
).dt.normalize()

prepared_screen["StockID"] = (
    pd.to_numeric(
        prepared_screen["StockID"],
        errors="coerce",
    )
    .astype("Int64")
    .astype(str)
)

prepared_screen[screen_ret_col] = (
    pd.to_numeric(
        prepared_screen[screen_ret_col],
        errors="coerce",
    )
)


# Keep exactly the stock-date keys accepted by the screen
# with a valid forward return, matching the original diagnostic.
screen_keys = (
    prepared_screen[
        [
            "Date",
            "StockID",
            screen_ret_col,
        ]
    ]
    .dropna(
        subset=[
            "Date",
            "StockID",
            screen_ret_col,
        ]
    )
    .drop_duplicates(
        ["Date", "StockID"],
        keep="last",
    )
    [
        [
            "Date",
            "StockID",
        ]
    ]
)


panel = (
    expected
    .merge(
        screen_keys,
        on=[
            "Date",
            "StockID",
        ],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        ["Date", "StockID"]
    )
    .reset_index(drop=True)
)


print(
    "Exact screened ExpectedReturnFusion panel:",
    f"{len(panel):,}",
    "rows |",
    f"{panel['Date'].nunique():,}",
    "dates |",
    f"{panel['StockID'].nunique():,}",
    "stocks",
)


# -------------------------------------------------------------------------
# 5. Construct EXACT top/bottom decile masks
# -------------------------------------------------------------------------

score_col = "mu_hat_5d"


low_mask = pd.Series(
    False,
    index=panel.index,
    dtype=bool,
)

high_mask = pd.Series(
    False,
    index=panel.index,
    dtype=bool,
)

rank_pct = pd.Series(
    np.nan,
    index=panel.index,
    dtype=float,
)


for _, idx in panel.groupby(
    "Date",
    sort=True,
).groups.items():

    idx = pd.Index(idx)

    score = pd.to_numeric(
        panel.loc[idx, score_col],
        errors="coerce",
    )

    finite = (
        score.notna()
        & np.isfinite(
            score.to_numpy(float)
        )
    )

    valid_idx = idx[
        finite.to_numpy()
    ]

    if len(valid_idx) == 0:
        continue

    score_valid = (
        score.loc[valid_idx]
        .astype(float)
    )

    # Diagnostic within-week percentile rank.
    rank_pct.loc[valid_idx] = (
        score_valid.rank(
            method="average",
            pct=True,
        )
    )

    low_lo, low_hi = (
        PortfolioMath._decile_bounds(
            score_valid,
            CUT,
            0,
        )
    )

    high_lo, high_hi = (
        PortfolioMath._decile_bounds(
            score_valid,
            CUT,
            CUT - 1,
        )
    )

    low_mask.loc[valid_idx] = (
        PortfolioMath._decile_mask(
            score_valid,
            low_lo,
            low_hi,
            is_lowest=True,
        )
        .astype(bool)
    )

    high_mask.loc[valid_idx] = (
        PortfolioMath._decile_mask(
            score_valid,
            high_lo,
            high_hi,
            is_lowest=False,
        )
        .astype(bool)
    )


panel["analysis_rank"] = rank_pct
panel["expected_low"] = low_mask
panel["expected_high"] = high_mask


# -------------------------------------------------------------------------
# 6. Week-to-week rank persistence
# -------------------------------------------------------------------------

def rank_persistence_series(
    df,
    rank_col,
):
    x = df[
        [
            "Date",
            "StockID",
            rank_col,
        ]
    ].copy()

    x = x.sort_values(
        [
            "StockID",
            "Date",
        ]
    )

    x["prev_date_for_stock"] = (
        x.groupby("StockID")["Date"]
        .shift(1)
    )

    x["prev_rank"] = (
        x.groupby("StockID")[rank_col]
        .shift(1)
    )

    unique_dates = sorted(
        x["Date"].dropna().unique()
    )

    prev_date_map = {
        pd.Timestamp(unique_dates[i]):
        pd.Timestamp(unique_dates[i - 1])
        for i in range(
            1,
            len(unique_dates),
        )
    }

    expected_prev = (
        x["Date"].map(
            prev_date_map
        )
    )

    valid = (
        x["prev_date_for_stock"]
        == expected_prev
    )

    x = x[
        valid
        & x["prev_rank"].notna()
        & x[rank_col].notna()
    ].copy()

    def _corr(g):
        if len(g) < 10:
            return np.nan

        return float(
            g[
                [
                    rank_col,
                    "prev_rank",
                ]
            ]
            .corr(
                method="spearman"
            )
            .iloc[0, 1]
        )

    return (
        x.groupby("Date")
        .apply(_corr)
        .rename("rank_persistence")
    )


rank_persistence = (
    rank_persistence_series(
        panel,
        "analysis_rank",
    )
)


# -------------------------------------------------------------------------
# 7. Tail retention and target turnover
# -------------------------------------------------------------------------

def set_turnover_and_retention(
    df,
    mask_col,
):
    selected = (
        df.loc[
            df[mask_col],
            [
                "Date",
                "StockID",
            ],
        ]
        .groupby("Date")["StockID"]
        .agg(set)
    )

    dates = sorted(
        df["Date"].dropna().unique()
    )

    rows = []

    for i in range(
        1,
        len(dates),
    ):
        prev_dt = pd.Timestamp(
            dates[i - 1]
        )

        cur_dt = pd.Timestamp(
            dates[i]
        )

        prev_set = selected.get(
            prev_dt,
            set(),
        )

        cur_set = selected.get(
            cur_dt,
            set(),
        )

        n_prev = len(prev_set)
        n_cur = len(cur_set)

        inter = len(
            prev_set.intersection(
                cur_set
            )
        )

        retention = (
            inter / n_prev
            if n_prev > 0
            else np.nan
        )

        if (
            n_prev > 0
            and n_cur > 0
        ):
            turnover = 0.5 * (
                inter
                * abs(
                    1.0 / n_cur
                    - 1.0 / n_prev
                )
                + (n_cur - inter)
                * (1.0 / n_cur)
                + (n_prev - inter)
                * (1.0 / n_prev)
            )
        else:
            turnover = np.nan

        rows.append(
            {
                "Date": cur_dt,
                "retention": retention,
                "turnover": turnover,
            }
        )

    return (
        pd.DataFrame(rows)
        .set_index("Date")
    )


high_stats = (
    set_turnover_and_retention(
        panel,
        "expected_high",
    )
)

low_stats = (
    set_turnover_and_retention(
        panel,
        "expected_low",
    )
)


# -------------------------------------------------------------------------
# 8. Holding-spell lengths
# -------------------------------------------------------------------------

def holding_spell_lengths(
    df,
    mask_col,
):
    selected = df.loc[
        df[mask_col],
        [
            "Date",
            "StockID",
        ],
    ].copy()

    if selected.empty:
        return np.array(
            [],
            dtype=float,
        )

    unique_dates = sorted(
        df["Date"].dropna().unique()
    )

    date_pos = {
        pd.Timestamp(dt): i
        for i, dt in enumerate(
            unique_dates
        )
    }

    selected["date_pos"] = (
        selected["Date"]
        .map(date_pos)
        .astype(int)
    )

    selected = selected.sort_values(
        [
            "StockID",
            "date_pos",
        ]
    )

    selected["new_spell"] = (
        selected
        .groupby("StockID")["date_pos"]
        .diff()
        .ne(1)
        .fillna(True)
    )

    selected["spell_id"] = (
        selected
        .groupby("StockID")["new_spell"]
        .cumsum()
    )

    return (
        selected
        .groupby(
            [
                "StockID",
                "spell_id",
            ]
        )
        .size()
        .to_numpy(float)
    )


high_spells = holding_spell_lengths(
    panel,
    "expected_high",
)

low_spells = holding_spell_lengths(
    panel,
    "expected_low",
)


# -------------------------------------------------------------------------
# 9. Final ExpectedReturnFusion persistence row
# -------------------------------------------------------------------------

weekly = pd.concat(
    [
        rank_persistence,
        high_stats[
            [
                "retention",
                "turnover",
            ]
        ].rename(
            columns={
                "retention":
                    "high_retention",
                "turnover":
                    "high_turnover",
            }
        ),
        low_stats[
            [
                "retention",
                "turnover",
            ]
        ].rename(
            columns={
                "retention":
                    "low_retention",
                "turnover":
                    "low_turnover",
            }
        ),
    ],
    axis=1,
)

weekly[
    "total_target_turnover"
] = (
    weekly["high_turnover"]
    + weekly["low_turnover"]
)


result = pd.DataFrame(
    [
        {
            "model":
                "ExpectedReturnFusion_5d",

            "mean_rank_persistence":
                float(
                    weekly[
                        "rank_persistence"
                    ].mean()
                ),

            "mean_high_retention":
                float(
                    weekly[
                        "high_retention"
                    ].mean()
                ),

            "mean_low_retention":
                float(
                    weekly[
                        "low_retention"
                    ].mean()
                ),

            "mean_high_turnover":
                float(
                    weekly[
                        "high_turnover"
                    ].mean()
                ),

            "mean_low_turnover":
                float(
                    weekly[
                        "low_turnover"
                    ].mean()
                ),

            "mean_total_target_turnover":
                float(
                    weekly[
                        "total_target_turnover"
                    ].mean()
                ),

            "median_high_holding_weeks":
                float(
                    np.median(high_spells)
                ),

            "mean_high_holding_weeks":
                float(
                    np.mean(high_spells)
                ),

            "median_low_holding_weeks":
                float(
                    np.median(low_spells)
                ),

            "mean_low_holding_weeks":
                float(
                    np.mean(low_spells)
                ),
        }
    ]
)


# Save separately so we do not overwrite any existing research result.
save_path = (
    OUTPUT_DIR
    / "expected_return_signal_persistence_diagnostic.csv"
)

result.to_csv(
    save_path,
    index=False,
)


print("\nEXPECTED-RETURN FUSION PERSISTENCE DIAGNOSTIC")
print("=" * 60)

display(
    result.style.format(
        {
            "mean_rank_persistence": "{:.3f}",
            "mean_high_retention": "{:.1%}",
            "mean_low_retention": "{:.1%}",
            "mean_high_turnover": "{:.3f}",
            "mean_low_turnover": "{:.3f}",
            "mean_total_target_turnover": "{:.3f}",
            "median_high_holding_weeks": "{:.1f}",
            "mean_high_holding_weeks": "{:.2f}",
            "median_low_holding_weeks": "{:.1f}",
            "mean_low_holding_weeks": "{:.2f}",
        }
    )
)

print("\nSaved to:")
print(save_path)

# Interpretation checklist

The final write-up should answer the research questions in this order:

1. **Complementarity:** Do CNN and RF rank stocks differently enough to make fusion economically meaningful?
2. **Fusion:** How much gross improvement appears when the signals are combined?
3. **Complexity:** Does HMM outperform or add return variation beyond untuned rank fusion, logistic fusion, and expected-return fusion?
4. **Mechanism:** Are HMM gains associated with state confidence, state transitions, or material changes in expert mapping?
5. **Portfolio behavior:** Does HMM alter tail membership, unique trades, persistence, or turnover in a systematic way?
6. **Implementation:** Which gross findings survive the corrected spread/impact model under identical portfolio construction?
7. **Capacity:** At what tested AUM does net performance weaken economically, and where does the lower Sharpe confidence bound cross zero?
8. **Missing-return materiality:** How many missing-return events occur, how large are the affected sleeve weights, and do any runs breach the 0.5% warning threshold?
9. **Path dependence:** How does a dynamically evolving fund differ from the fixed-AUM capacity experiment?

### Claim discipline

- Do not claim that state sensitivity *causes* the gross improvement unless the mechanism tests support that statement.
- Distinguish **spanning alpha** from factor alpha or state alpha.
- Distinguish the gross return of an unconstrained zero-cost optimizer from the **executed pre-cost return of cost-aware weights**.
- Treat the old pre-refactor transaction-cost results as superseded.
- Treat a 0% missing return as an **accounting convention for an unobservable legacy-holding period**, not as the true economic return.
- If missing-return materiality warnings occur, report them explicitly and do not describe the approximation as immaterial without qualification.
- Treat dynamic NAV primarily as a path-dependent implementation simulation, not as an ordinary return stream for naive bootstrap inference.
- Report both economic magnitude and statistical uncertainty.
